# 04 — Seattle GCCP: Author-Aligned Check

## 目的

本 notebook 用于检查 GCCP 与原始 GeoSIMCP 是否真正做到同条件对比。

本阶段不修改 GeoSIMCP 的整体 conformal prediction（共形预测）框架，只检查将：

ordinary feature similarity（普通特征相似性）

替换为：

causal-weighted feature similarity（因果加权特征相似性）

之后会发生什么变化。

## 对齐原则

GeoSIMCP-EUC 与 GCCP 保持完全一致的：

- data split（数据划分）
- XGBoost predictor（XGBoost预测模型）
- Calibration residual（校正集残差）
- geographic distance（地理距离）
- Gaussian kernel（高斯核函数）
- bandwidth（带宽）
- lambda（融合参数）
- weighted quantile（加权分位数）
- prediction interval（预测区间）计算方式
- parameter selection（参数选择方式）

唯一计划修改：

GeoSIMCP-EUC:
d_feat = Euclidean feature distance（欧氏特征距离）

GCCP:
d_feat → d_causal = causal-weighted feature distance（因果加权特征距离）

## 首先进行两个退化测试

1. 当 lambda = 1 时：

GCCP 应严格退化为 GeoCP（地理共形预测）。

2. 当 8 个 causal weights（因果权重）完全相等时：

GCCP 的 causal distance（因果距离）应严格等于 GeoSIMCP-EUC 的 Euclidean feature distance（欧氏特征距离）。

只有两个退化测试均通过，才能继续比较 GCCM causal weights（GCCM因果权重）对结果的影响。

In [2]:
# ============================================================
# Step 0 — Environment & repository audit
# 环境和本地 GeoSIMCP 仓库版本审计
# 不运行任何模型，不修改任何文件
# ============================================================

from pathlib import Path
import sys
import subprocess
import importlib.metadata as metadata


# ------------------------------------------------------------
# 1. Local project path
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

print("===== LOCAL ENVIRONMENT AUDIT =====")

print("\n[1] Python")
print("Python executable :", sys.executable)
print("Python version    :", sys.version)


# ------------------------------------------------------------
# 2. Installed geoconformal package
# ------------------------------------------------------------

print("\n[2] Installed geoconformal")

try:
    installed_version = metadata.version("geoconformal")
    print("Package version   :", installed_version)
except Exception as e:
    print("Package version   : NOT FOUND")
    print("Reason            :", repr(e))

try:
    import geoconformal

    print(
        "Imported from     :",
        Path(geoconformal.__file__).resolve()
    )
except Exception as e:
    print("Import status     : FAILED")
    print("Reason            :", repr(e))


# ------------------------------------------------------------
# 3. Check local repository files
# ------------------------------------------------------------

print("\n[3] Local repository")

print(
    "Project root      :",
    PROJECT_ROOT
)

print(
    "Project exists    :",
    PROJECT_ROOT.exists()
)

print(
    "README exists     :",
    (PROJECT_ROOT / "README.md").exists()
)

print(
    "CHANGELOG exists  :",
    (PROJECT_ROOT / "CHANGELOG.md").exists()
)


# ------------------------------------------------------------
# 4. Git information
# ------------------------------------------------------------

def git_command(*args):
    try:
        result = subprocess.run(
            [
                "git",
                "-C",
                str(PROJECT_ROOT),
                *args
            ],
            capture_output=True,
            text=True,
            check=True
        )

        return result.stdout.strip()

    except Exception as e:
        return f"ERROR: {repr(e)}"


print("\n[4] Git information")

print(
    "Branch            :",
    git_command(
        "rev-parse",
        "--abbrev-ref",
        "HEAD"
    )
)

print(
    "Commit            :",
    git_command(
        "rev-parse",
        "HEAD"
    )
)

print(
    "Latest local log  :",
    git_command(
        "log",
        "-1",
        "--oneline"
    )
)

print(
    "Remote            :",
    git_command(
        "remote",
        "-v"
    )
)


# ------------------------------------------------------------
# 5. Read local CHANGELOG header
# ------------------------------------------------------------

print("\n[5] Local CHANGELOG preview")

CHANGELOG_PATH = PROJECT_ROOT / "CHANGELOG.md"

if CHANGELOG_PATH.exists():

    text = CHANGELOG_PATH.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    lines = text.splitlines()

    for line in lines[:30]:
        print(line)

else:
    print("No local CHANGELOG.md")


print(
    "\n✓ Audit finished. "
    "No model was run and no file was modified."
)

===== LOCAL ENVIRONMENT AUDIT =====

[1] Python
Python executable : D:\BaiduNetdiskDownload\Anaconda3\envs\geosimcp_repro\python.exe
Python version    : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]

[2] Installed geoconformal
Package version   : 0.3.0
Imported from     : E:\GeoSIMCP_Reproduce\geoconformal-main\geoconformal\__init__.py

[3] Local repository
Project root      : E:\GeoSIMCP_Reproduce\geoconformal-main
Project exists    : True
README exists     : True
CHANGELOG exists  : True

[4] Git information
Branch            : ERROR: FileNotFoundError(2, '系统找不到指定的文件。', None, 2, None)
Commit            : ERROR: FileNotFoundError(2, '系统找不到指定的文件。', None, 2, None)
Latest local log  : ERROR: FileNotFoundError(2, '系统找不到指定的文件。', None, 2, None)
Remote            : ERROR: FileNotFoundError(2, '系统找不到指定的文件。', None, 2, None)

[5] Local CHANGELOG preview
# Changelog

## 0.3.0

Integrated the finite-sample-valid weighted / Bayesian conformal fram

In [3]:
# ============================================================
# Step 1 — Inspect geoconformal 0.3.0 interfaces
# 检查最新版 GeoCP / GeoSIMCP 接口
# 不运行模型，不修改任何文件
# ============================================================

from pathlib import Path
import inspect
import geoconformal

from geoconformal import (
    GeoConformalSpatialRegression,
    GeoSIMConformalSpatialRegression
)

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

print("===== GEOCONFORMAL 0.3.0 INTERFACE AUDIT =====")


# ------------------------------------------------------------
# 1. 确认实际使用的是哪份源码
# ------------------------------------------------------------

print("\n[1] Source location")

print(
    "geoconformal package :",
    Path(geoconformal.__file__).resolve()
)

print(
    "GeoCP class          :",
    Path(
        inspect.getfile(
            GeoConformalSpatialRegression
        )
    ).resolve()
)

print(
    "GeoSIMCP class       :",
    Path(
        inspect.getfile(
            GeoSIMConformalSpatialRegression
        )
    ).resolve()
)


# ------------------------------------------------------------
# 2. 查看 GeoCP 构造函数
# ------------------------------------------------------------

print("\n[2] GeoCP constructor")

geocp_signature = inspect.signature(
    GeoConformalSpatialRegression
)

print(geocp_signature)


# ------------------------------------------------------------
# 3. 查看 GeoSIMCP 构造函数
# ------------------------------------------------------------

print("\n[3] GeoSIMCP constructor")

geosimcp_signature = inspect.signature(
    GeoSIMConformalSpatialRegression
)

print(geosimcp_signature)


# ------------------------------------------------------------
# 4. 检查我们后续实验最重要的参数
# ------------------------------------------------------------

print("\n[4] Important GeoSIMCP parameters")

parameters = geosimcp_signature.parameters

important_parameters = [
    "bandwidth",
    "lambda_weight",
    "distance_metric",
    "standardize_weights",
    "include_test_atom",
]

for name in important_parameters:

    if name in parameters:

        default_value = parameters[name].default

        print(
            f"{name:22s}: FOUND",
            f"| default = {default_value}"
        )

    else:

        print(
            f"{name:22s}: NOT FOUND"
        )


# ------------------------------------------------------------
# 5. 查看类的说明文档前 80 行
# ------------------------------------------------------------

print("\n[5] GeoSIMCP documentation preview")

doc = inspect.getdoc(
    GeoSIMConformalSpatialRegression
)

if doc is None:
    print("No docstring found.")

else:
    doc_lines = doc.splitlines()

    for line in doc_lines[:80]:
        print(line)


print(
    "\n✓ Interface audit finished."
)
print(
    "No model was run and no source file was modified."
)

===== GEOCONFORMAL 0.3.0 INTERFACE AUDIT =====

[1] Source location
geoconformal package : E:\GeoSIMCP_Reproduce\geoconformal-main\geoconformal\__init__.py
GeoCP class          : E:\GeoSIMCP_Reproduce\geoconformal-main\geoconformal\GeoConformalSpatialRegression.py
GeoSIMCP class       : E:\GeoSIMCP_Reproduce\geoconformal-main\geoconformal\GeoSIMConformalSpatialRegression.py

[2] GeoCP constructor
(predict_f: Callable, nonconformity_score_f: Callable = None, miscoverage_level: float = 0.1, bandwidth: float = None, coord_calib: numpy.ndarray = None, coord_test: numpy.ndarray = None, X_calib: numpy.ndarray = None, y_calib: numpy.ndarray = None, X_test: numpy.ndarray = None, y_test: numpy.ndarray = None, include_test_atom: bool = False)

[3] GeoSIMCP constructor
(predict_f: Callable, nonconformity_score_f: Callable = None, miscoverage_level: float = 0.1, bandwidth: float = None, coord_calib=None, coord_test=None, X_calib=None, y_calib=None, X_test=None, y_test=None, lambda_weight: float = 

In [4]:
# ============================================================
# Step 2A — Inspect actual GeoSIMCP distance implementation
# 检查 0.3.0 实际使用的 GeoSIMCP 距离公式
# 不运行模型，不修改源码
# ============================================================

import inspect

from geoconformal import (
    GeoSIMConformalSpatialRegression
)

from geoconformal.geocp.weights import (
    joint_geo_feature_weights
)


print(
    "===== ACTUAL GEOSIMCP IMPLEMENTATION ====="
)


# ------------------------------------------------------------
# 1. GeoSIMCP wrapper source
# ------------------------------------------------------------

print(
    "\n[1] GeoSIMConformalSpatialRegression source"
)

source_wrapper = inspect.getsource(
    GeoSIMConformalSpatialRegression
)

# 只打印和距离计算有关的行，避免输出太长
keywords = [
    "distance_metric",
    "euclidean",
    "mnd",
    "standardize",
    "lambda",
    "joint_geo_feature",
]

wrapper_lines = source_wrapper.splitlines()

for line_no, line in enumerate(
    wrapper_lines,
    start=1
):
    if any(
        keyword.lower() in line.lower()
        for keyword in keywords
    ):
        start = max(
            0,
            line_no - 4
        )

        end = min(
            len(wrapper_lines),
            line_no + 5
        )

        print(
            f"\n--- around wrapper line {line_no} ---"
        )

        for j in range(
            start,
            end
        ):
            print(
                f"{j+1:4d}: {wrapper_lines[j]}"
            )


# ------------------------------------------------------------
# 2. Unified joint geographic-feature weight function
# ------------------------------------------------------------

print(
    "\n\n[2] joint_geo_feature_weights source"
)

source_joint = inspect.getsource(
    joint_geo_feature_weights
)

print(source_joint)


# ------------------------------------------------------------
# 3. Important interpretation reminder
# ------------------------------------------------------------

print(
    "\n===== WHAT WE NEED TO VERIFY ====="
)

print(
    "1. Geographic distance formula"
)

print(
    "2. Euclidean feature distance formula"
)

print(
    "3. MND feature distance formula"
)

print(
    "4. Whether EUC features are standardized"
)

print(
    "5. Whether MND features are standardized/range-normalized"
)

print(
    "6. How lambda combines geographic and feature distance"
)

print(
    "7. How bandwidth enters the Gaussian kernel"
)

print(
    "\n✓ Source inspection finished."
)
print(
    "No model was run and no source file was modified."
)

===== ACTUAL GEOSIMCP IMPLEMENTATION =====

[1] GeoSIMConformalSpatialRegression source

--- around wrapper line 9 ---
   6:     feature-space similarity when estimating local uncertainty, for spatially
   7:     nonstationary processes. A thin, stateful wrapper over the unified
   8:     ``geoconformal.geocp`` engine (the joint kernel lives in
   9:     ``geoconformal.geocp.weights.joint_geo_feature_weights``); the classic
  10:     ``.analyze()`` / ``GeoConformalResults.to_gpd()`` workflow is preserved.
  11: 
  12:     Parameters
  13:     ----------
  14:     predict_f : Callable

--- around wrapper line 26 ---
  23:         Calibration / test coordinates.
  24:     X_calib, y_calib, X_test, y_test : arrays
  25:         Calibration / test features and targets (used for prediction).
  26:     lambda_weight : float
  27:         Trade-off between geographic distance (lambda=1, reduces to GeoCP) and
  28:         feature distance (lambda=0). Default 1.0.
  29:     X_calib_weight, X_t

In [5]:
# ============================================================
# Step 2B — Inspect weighted quantile implementation
# 检查最新版加权分位数到底如何处理权重
# 不运行模型，不修改源码
# ============================================================

import inspect

from geoconformal.geocp import (
    weighted_quantile
)

print(
    "===== WEIGHTED QUANTILE IMPLEMENTATION ====="
)

print(
    inspect.getsource(
        weighted_quantile
    )
)

print(
    "\n===== QUESTIONS TO CHECK ====="
)

print(
    "1. Are weights normalized internally?"
)

print(
    "2. How is the cumulative weight calculated?"
)

print(
    "3. Where is q = 1 - alpha applied?"
)

print(
    "4. What changes when include_test_atom=True?"
)

print(
    "\n✓ Inspection finished."
)
print(
    "No model was run and no source file was modified."
)

===== WEIGHTED QUANTILE IMPLEMENTATION =====
def weighted_quantile(
    scores: NDArray,
    weights: NDArray,
    q: float,
    self_weights: NDArray = None,
    interpolate: bool = False,
) -> NDArray:
    """
    Vectorized weighted quantile of calibration scores per test point.

    By default this is the conformal-valid *step* quantile
    ``inf{ z : F(z) >= q }`` (the appropriate order statistic), which is the
    definition under which the coverage guarantee holds and which reduces
    exactly to standard split conformal in the unweighted case. Set
    ``interpolate=True`` for the older linearly-interpolated variant (sharper
    but not finite-sample valid -- it can undercover).

    If ``self_weights`` is provided, the test point's own atom at +infinity is
    included with weight w(x), normalizing over (sum_j w(X_j) + w(x)) per
    Tibshirani et al. (2019). A test point whose calibration mass cannot reach q
    -- because the +infinity atom holds the remaining mass -- receives

In [6]:
# ============================================================
# Step 3 — Inspect official Seattle tutorial experiment setup
# 提取最新版 GeoSIMCP 0.3.0 Seattle 教程的实验设置
# 不运行模型，不修改任何文件
# ============================================================

from pathlib import Path
import json


PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

TUTORIAL_PATH = (
    PROJECT_ROOT
    / "example"
    / "geoconformal_tutorial.ipynb"
)


print("===== OFFICIAL SEATTLE TUTORIAL AUDIT =====")

print(
    "\nTutorial:",
    TUTORIAL_PATH
)

print(
    "Exists:",
    TUTORIAL_PATH.exists()
)


# ------------------------------------------------------------
# 1. Load notebook as JSON
# ------------------------------------------------------------

with open(
    TUTORIAL_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)


# ------------------------------------------------------------
# 2. Keywords relevant to experimental protocol
# ------------------------------------------------------------

keywords = [

    # Dataset / preprocessing
    "seattle",
    "seattle_sample",
    "log_price",
    "UTM_X",
    "UTM_Y",
    "Transformer",

    # Data split
    "train_test_split",
    "train_size",
    "test_size",
    "calib",
    "validation",
    "val",

    # Predictor
    "XGBRegressor",
    "n_estimators",
    "max_depth",

    # GeoCP / GeoSIMCP
    "GeoConformalSpatialRegression",
    "GeoSIMConformalSpatialRegression",

    # Hyperparameters
    "bandwidth",
    "lambda_weight",
    "lambda_list",
    "bandwidth_list",

    # Feature distance
    "distance_metric",
    "standardize_weights",

    # New 0.3.0 option
    "include_test_atom",

    # Evaluation
    "coverage",
    "interval_score",
    "uncertainty",
]


# ------------------------------------------------------------
# 3. Print all relevant cells
# ------------------------------------------------------------

matched_cells = []

for idx, cell in enumerate(nb["cells"]):

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    if any(
        keyword.lower() in source.lower()
        for keyword in keywords
    ):

        matched_cells.append(idx)

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"CELL {idx} | "
            f"type = {cell.get('cell_type')}"
        )

        print(
            "=" * 80
        )

        print(source)


# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print(
    "\n===== SUMMARY ====="
)

print(
    "Total notebook cells:",
    len(nb["cells"])
)

print(
    "Relevant cells found:",
    len(matched_cells)
)

print(
    "Cell indices:",
    matched_cells
)

print(
    "\n✓ Tutorial inspection finished."
)

print(
    "No model was run and no source file was modified."
)

===== OFFICIAL SEATTLE TUTORIAL AUDIT =====

Tutorial: E:\GeoSIMCP_Reproduce\geoconformal-main\example\geoconformal_tutorial.ipynb
Exists: True

CELL 0 | type = markdown
# geoconformal Tutorial: GeoCP & GeoSIMCP

This notebook demonstrates how to use the `geoconformal` package for **model-agnostic uncertainty quantification** in spatial prediction, using the Seattle housing price dataset as an example.

We will cover:
1. **Data loading & preparation** — how to split data into train / calibration / test
2. **GeoCP** — uncertainty quantification using geographic distance only
3. **GeoSIMCP** — uncertainty quantification using geographic distance + feature similarity
4. **Hyperparameter tuning** — grid search for optimal bandwidth and lambda
5. **Spatial mapping** — visualizing prediction and uncertainty on a map

CELL 3 | type = code
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from sklearn.model_selection import t

In [7]:
# ============================================================
# Step 4 — Reproduce official Seattle 0.3.0 data preparation
# 完全按照最新版 tutorial 恢复 Seattle 实验数据
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error
)

from xgboost import XGBRegressor


# ------------------------------------------------------------
# 1. Load Seattle dataset exactly as official tutorial
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

SEATTLE_CSV = (
    PROJECT_ROOT
    / "example"
    / "seattle_sample_3k.csv"
)

data = pd.read_csv(
    SEATTLE_CSV,
    index_col=0
)


# ------------------------------------------------------------
# 2. Define variables exactly as official tutorial
# ------------------------------------------------------------

feature_cols = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]

coord_cols = [
    "UTM_X",
    "UTM_Y",
]

target_col = "log_price"


X = data[feature_cols].values
coords = data[coord_cols].values
y = data[target_col].values


# ------------------------------------------------------------
# 3. Official 70 / 10 / 10 / 10 split
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp, coords_train, coords_temp = (
    train_test_split(
        X,
        y,
        coords,
        test_size=0.3,
        random_state=42
    )
)

X_calib, X_temp2, y_calib, y_temp2, coords_calib, coords_temp2 = (
    train_test_split(
        X_temp,
        y_temp,
        coords_temp,
        test_size=2/3,
        random_state=42
    )
)

X_val, X_test, y_val, y_test, coords_val, coords_test = (
    train_test_split(
        X_temp2,
        y_temp2,
        coords_temp2,
        test_size=0.5,
        random_state=42
    )
)


# ------------------------------------------------------------
# 4. Combine prediction features + coordinates
# ------------------------------------------------------------

X_train_spatial = np.hstack([
    X_train,
    coords_train
])

X_calib_spatial = np.hstack([
    X_calib,
    coords_calib
])

X_val_spatial = np.hstack([
    X_val,
    coords_val
])

X_test_spatial = np.hstack([
    X_test,
    coords_test
])


# ------------------------------------------------------------
# 5. Train official XGBoost configuration
# ------------------------------------------------------------

model = XGBRegressor(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model.fit(
    X_train_spatial,
    y_train
)


# ------------------------------------------------------------
# 6. Basic checks
# ------------------------------------------------------------

y_test_pred = model.predict(
    X_test_spatial
)

print(
    "===== OFFICIAL SEATTLE DATA PREPARATION ====="
)

print(
    "\nSplit sizes:"
)

print(
    "Train       :",
    len(X_train)
)

print(
    "Calibration :",
    len(X_calib)
)

print(
    "Validation  :",
    len(X_val)
)

print(
    "Test        :",
    len(X_test)
)

print(
    "\nTarget:"
)

print(
    "Target variable :",
    target_col
)

print(
    "Target min      :",
    y.min()
)

print(
    "Target max      :",
    y.max()
)

print(
    "\nCoordinates:"
)

print(
    "Coordinate system : UTM EPSG:32610"
)

print(
    "X range:",
    coords[:, 0].min(),
    "to",
    coords[:, 0].max()
)

print(
    "Y range:",
    coords[:, 1].min(),
    "to",
    coords[:, 1].max()
)

print(
    "\nXGBoost Test performance:"
)

print(
    f"R²  = {r2_score(y_test, y_test_pred):.6f}"
)

print(
    f"MAE = {mean_absolute_error(y_test, y_test_pred):.6f}"
)

print(
    "\n✓ Official data preparation reproduced."
)

===== OFFICIAL SEATTLE DATA PREPARATION =====

Split sizes:
Train       : 2100
Calibration : 300
Validation  : 300
Test        : 300

Target:
Target variable : log_price
Target min      : 4.913813852383717
Target max      : 6.560504415195057

Coordinates:
Coordinate system : UTM EPSG:32610
X range: 540518.0798962074 to 569985.211179947
Y range: 5233999.549414068 to 5291784.21364018

XGBoost Test performance:
R²  = 0.891180
MAE = 0.056048

✓ Official data preparation reproduced.


In [8]:
# ============================================================
# Step 5 — Official GeoCP bandwidth tuning
# 完全按照 geoconformal 0.3.0 Seattle tutorial
#
# 重要：
# 1. 只使用 Validation set 调参
# 2. 不使用 Test set
# 3. include_test_atom=False，保持原始 GeoCP 行为
# ============================================================

import numpy as np
import pandas as pd

from geoconformal import (
    GeoConformalSpatialRegression
)


# ------------------------------------------------------------
# 1. Interval Score
#    与官方 Seattle tutorial 保持一致
# ------------------------------------------------------------

def interval_score(
    upper,
    lower,
    y_true,
    alpha=0.1
):
    width = upper - lower

    penalty = (
        2.0 / alpha
    ) * (
        np.maximum(
            lower - y_true,
            0
        )
        +
        np.maximum(
            y_true - upper,
            0
        )
    )

    return np.mean(
        width + penalty
    )


# ------------------------------------------------------------
# 2. Official GeoCP bandwidth grid
# ------------------------------------------------------------

bandwidths_geocp = np.linspace(
    500,
    30000,
    30
)

geocp_validation_results = []


print(
    "===== GEOCP VALIDATION TUNING ====="
)

print(
    "Number of bandwidth candidates:",
    len(bandwidths_geocp)
)

print(
    "Validation samples:",
    len(y_val)
)

print(
    "Test samples are NOT used for tuning:",
    len(y_test)
)


# ------------------------------------------------------------
# 3. Grid search on Validation set only
# ------------------------------------------------------------

for bw in bandwidths_geocp:

    geocp_val = GeoConformalSpatialRegression(

        predict_f=model.predict,

        miscoverage_level=0.1,

        bandwidth=bw,

        coord_calib=coords_calib,
        coord_test=coords_val,

        X_calib=X_calib_spatial,
        y_calib=y_calib,

        X_test=X_val_spatial,
        y_test=y_val,

        # Explicitly preserve original GeoCP behavior
        include_test_atom=False
    )

    result = geocp_val.analyze()

    mean_width = np.mean(
        result.upper_bound
        -
        result.lower_bound
    )

    score = interval_score(
        result.upper_bound,
        result.lower_bound,
        y_val,
        alpha=0.1
    )

    geocp_validation_results.append({

        "bandwidth": bw,

        "coverage":
            result.coverage_probability,

        "mean_interval":
            mean_width,

        "interval_score":
            score
    })


# ------------------------------------------------------------
# 4. Convert to DataFrame
# ------------------------------------------------------------

df_geocp_val = pd.DataFrame(
    geocp_validation_results
)


# ------------------------------------------------------------
# 5. Official parameter-selection rule
#
# If Coverage >= 0.90 candidates exist:
#     select minimum Interval Score
#
# Otherwise:
#     select minimum Interval Score over all candidates
# ------------------------------------------------------------

valid_geocp = df_geocp_val[
    df_geocp_val["coverage"] >= 0.90
].copy()


if len(valid_geocp) > 0:

    selection_mode_geocp = (
        "Coverage >= 0.90 -> minimum Interval Score"
    )

    best_geocp = valid_geocp.loc[
        valid_geocp["interval_score"].idxmin()
    ]

else:

    selection_mode_geocp = (
        "No Coverage >= 0.90 candidate "
        "-> minimum Interval Score over all candidates"
    )

    best_geocp = df_geocp_val.loc[
        df_geocp_val["interval_score"].idxmin()
    ]


# ------------------------------------------------------------
# 6. Print results
# ------------------------------------------------------------

print(
    "\n===== GEOCP TUNING RESULT ====="
)

print(
    "Selection mode:",
    selection_mode_geocp
)

print(
    "Candidates with Coverage >= 0.90:",
    len(valid_geocp)
)

print(
    "\nBest bandwidth:"
)

print(
    f"{best_geocp['bandwidth']:.6f}"
)

print(
    "\nValidation metrics:"
)

print(
    f"Coverage       = "
    f"{best_geocp['coverage']:.6f}"
)

print(
    f"Mean Interval  = "
    f"{best_geocp['mean_interval']:.6f}"
)

print(
    f"Interval Score = "
    f"{best_geocp['interval_score']:.6f}"
)


# ------------------------------------------------------------
# 7. Show top 10 valid candidates
# ------------------------------------------------------------

if len(valid_geocp) > 0:

    print(
        "\nTop 10 candidates satisfying Coverage >= 0.90:"
    )

    display(
        valid_geocp
        .sort_values(
            [
                "interval_score",
                "mean_interval"
            ]
        )
        .head(10)
        .reset_index(drop=True)
    )

else:

    print(
        "\nNo candidate reached 0.90 Coverage."
    )

    display(
        df_geocp_val
        .sort_values(
            "interval_score"
        )
        .head(10)
        .reset_index(drop=True)
    )


print(
    "\n✓ GeoCP Validation tuning completed."
)

print(
    "✓ Test set has NOT been used for parameter selection."
)

===== GEOCP VALIDATION TUNING =====
Number of bandwidth candidates: 30
Validation samples: 300
Test samples are NOT used for tuning: 300

===== GEOCP TUNING RESULT =====
Selection mode: Coverage >= 0.90 -> minimum Interval Score
Candidates with Coverage >= 0.90: 27

Best bandwidth:
28982.758621

Validation metrics:
Coverage       = 0.923333
Mean Interval  = 0.257507
Interval Score = 0.322355

Top 10 candidates satisfying Coverage >= 0.90:


,bandwidth,coverage,mean_interval,interval_score
0,28982.758621,0.923333,0.257507,0.322355
1,30000.000000,0.923333,0.257427,0.322360
2,27965.517241,0.923333,0.257551,0.322509
3,26948.275862,0.923333,0.257593,0.322552
4,23896.551724,0.923333,0.257557,0.322591
5,24913.793103,0.923333,0.257589,0.322597
6,25931.034483,0.923333,0.257643,0.322626
7,22879.310345,0.923333,0.257551,0.322818
8,20844.827586,0.923333,0.257367,0.322873
9,21862.068966,0.923333,0.257429,0.322942



✓ GeoCP Validation tuning completed.
✓ Test set has NOT been used for parameter selection.


In [9]:
# ============================================================
# Step 6 — Official GeoSIMCP-EUC Validation tuning
# 完全按照 geoconformal 0.3.0 Seattle tutorial
#
# 重要：
# 1. 只使用 Validation set 调参
# 2. Test set 不参与参数选择
# 3. distance_metric = "euclidean"
# 4. standardize_weights = True
# 5. include_test_atom = False
# ============================================================

import numpy as np
import pandas as pd

from geoconformal import (
    GeoSIMConformalSpatialRegression
)


# ------------------------------------------------------------
# 1. Official GeoSIMCP grids
# ------------------------------------------------------------

bandwidths_sim = np.linspace(
    500,
    30000,
    15
)

lambdas_sim = np.arange(
    0,
    1.05,
    0.1
)

geosimcp_euc_validation_results = []


print(
    "===== GEOSIMCP-EUC VALIDATION TUNING ====="
)

print(
    "Bandwidth candidates:",
    len(bandwidths_sim)
)

print(
    "Lambda candidates:",
    len(lambdas_sim)
)

print(
    "Total configurations:",
    len(bandwidths_sim) * len(lambdas_sim)
)

print(
    "Validation samples:",
    len(y_val)
)

print(
    "Test set is NOT used for tuning."
)


# ------------------------------------------------------------
# 2. Grid search on Validation set only
# ------------------------------------------------------------

for bw in bandwidths_sim:

    for lam in lambdas_sim:

        geosimcp_val = GeoSIMConformalSpatialRegression(

            predict_f=model.predict,

            miscoverage_level=0.1,

            bandwidth=bw,

            lambda_weight=lam,

            distance_metric="euclidean",

            standardize_weights=True,

            coord_calib=coords_calib,
            coord_test=coords_val,

            X_calib=X_calib_spatial,
            y_calib=y_calib,

            X_test=X_val_spatial,
            y_test=y_val,

            # Only 8 non-spatial attributes
            # are used for feature similarity.
            X_calib_weight=X_calib,
            X_test_weight=X_val,

            # Reproduce original GeoSIMCP behaviour
            include_test_atom=False
        )

        result = geosimcp_val.analyze()

        mean_width = np.mean(
            result.upper_bound
            -
            result.lower_bound
        )

        score = interval_score(
            result.upper_bound,
            result.lower_bound,
            y_val,
            alpha=0.1
        )

        geosimcp_euc_validation_results.append({

            "bandwidth": bw,

            "lambda": lam,

            "coverage":
                result.coverage_probability,

            "mean_interval":
                mean_width,

            "interval_score":
                score
        })


# ------------------------------------------------------------
# 3. Convert to DataFrame
# ------------------------------------------------------------

df_geosimcp_euc_val = pd.DataFrame(
    geosimcp_euc_validation_results
)


# ------------------------------------------------------------
# 4. Official parameter-selection rule
#
# If Coverage >= 0.90 candidates exist:
#     minimum Interval Score among them
#
# Otherwise:
#     minimum Interval Score over all candidates
# ------------------------------------------------------------

valid_sim = df_geosimcp_euc_val[
    df_geosimcp_euc_val["coverage"] >= 0.90
].copy()


if len(valid_sim) > 0:

    selection_mode_sim = (
        "Coverage >= 0.90 -> minimum Interval Score"
    )

    best_simcp_euc = valid_sim.loc[
        valid_sim["interval_score"].idxmin()
    ]

else:

    selection_mode_sim = (
        "No Coverage >= 0.90 candidate "
        "-> minimum Interval Score over all candidates"
    )

    best_simcp_euc = df_geosimcp_euc_val.loc[
        df_geosimcp_euc_val["interval_score"].idxmin()
    ]


# ------------------------------------------------------------
# 5. Print best result
# ------------------------------------------------------------

print(
    "\n===== GEOSIMCP-EUC TUNING RESULT ====="
)

print(
    "Selection mode:",
    selection_mode_sim
)

print(
    "Candidates with Coverage >= 0.90:",
    len(valid_sim)
)

print(
    "\nBest parameters:"
)

print(
    f"Bandwidth = "
    f"{best_simcp_euc['bandwidth']:.6f}"
)

print(
    f"Lambda    = "
    f"{best_simcp_euc['lambda']:.6f}"
)

print(
    "\nValidation metrics:"
)

print(
    f"Coverage       = "
    f"{best_simcp_euc['coverage']:.6f}"
)

print(
    f"Mean Interval  = "
    f"{best_simcp_euc['mean_interval']:.6f}"
)

print(
    f"Interval Score = "
    f"{best_simcp_euc['interval_score']:.6f}"
)


# ------------------------------------------------------------
# 6. Show top 15 valid candidates
# ------------------------------------------------------------

if len(valid_sim) > 0:

    print(
        "\nTop 15 candidates satisfying Coverage >= 0.90:"
    )

    display(
        valid_sim
        .sort_values(
            [
                "interval_score",
                "mean_interval"
            ]
        )
        .head(15)
        .reset_index(drop=True)
    )

else:

    print(
        "\nNo candidate reached 0.90 Coverage."
    )

    display(
        df_geosimcp_euc_val
        .sort_values(
            "interval_score"
        )
        .head(15)
        .reset_index(drop=True)
    )


print(
    "\n✓ GeoSIMCP-EUC Validation tuning completed."
)

print(
    "✓ Test set has NOT been used for parameter selection."
)

===== GEOSIMCP-EUC VALIDATION TUNING =====
Bandwidth candidates: 15
Lambda candidates: 11
Total configurations: 165
Validation samples: 300
Test set is NOT used for tuning.

===== GEOSIMCP-EUC TUNING RESULT =====
Selection mode: Coverage >= 0.90 -> minimum Interval Score
Candidates with Coverage >= 0.90: 153

Best parameters:
Bandwidth = 500.000000
Lambda    = 0.000000

Validation metrics:
Coverage       = 0.920000
Mean Interval  = 0.254190
Interval Score = 0.320995

Top 15 candidates satisfying Coverage >= 0.90:


,bandwidth,lambda,coverage,mean_interval,interval_score
0,500.000000,0.0,0.92,0.254190,0.320995
1,8928.571429,0.0,0.92,0.254136,0.321077
2,11035.714286,0.0,0.92,0.254136,0.321077
3,13142.857143,0.0,0.92,0.254136,0.321077
4,15250.000000,0.0,0.92,0.254136,0.321077
5,17357.142857,0.0,0.92,0.254136,0.321077
6,19464.285714,0.0,0.92,0.254136,0.321077
7,21571.428571,0.0,0.92,0.254136,0.321077
8,23678.571429,0.0,0.92,0.254136,0.321077
9,25785.714286,0.0,0.92,0.254136,0.321077



✓ GeoSIMCP-EUC Validation tuning completed.
✓ Test set has NOT been used for parameter selection.


In [10]:
# ============================================================
# Step 7 — GeoSIMCP distance-scale diagnostic
# 检查 geographic distance 与 EUC feature distance 的实际尺度
#
# 只使用 Calibration + Validation
# 不使用 Test set
# 不修改模型
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances


print(
    "===== GEOSIMCP DISTANCE SCALE DIAGNOSTIC ====="
)


# ------------------------------------------------------------
# 1. Geographic distance
#    Official tutorial uses raw UTM coordinates (meters)
#
# Shape:
# Validation x Calibration = 300 x 300
# ------------------------------------------------------------

d_geo_val = pairwise_distances(
    coords_val,
    coords_calib,
    metric="euclidean"
)


# ------------------------------------------------------------
# 2. EUC feature distance
#
# Reproduce official GeoSIMCP:
# - standardization parameters fitted ONLY on Calibration
# - validation uses the same Calibration mean/std
# ------------------------------------------------------------

calib_std_raw = np.std(
    X_calib,
    axis=0
)

valid_feature_cols = (
    calib_std_raw > 1e-8
)


X_calib_valid = X_calib[
    :,
    valid_feature_cols
]

X_val_valid = X_val[
    :,
    valid_feature_cols
]


feature_mean = np.mean(
    X_calib_valid,
    axis=0
)

feature_std = np.std(
    X_calib_valid,
    axis=0
)

feature_std[
    feature_std == 0
] = 1.0


Z_calib = (
    X_calib_valid
    -
    feature_mean
) / feature_std


Z_val = (
    X_val_valid
    -
    feature_mean
) / feature_std


d_euc_val = pairwise_distances(
    Z_val,
    Z_calib,
    metric="euclidean"
)


# ------------------------------------------------------------
# 3. Summary helper
# ------------------------------------------------------------

def distance_summary(
    name,
    matrix
):

    flat = matrix.ravel()

    return {
        "distance": name,
        "min": np.min(flat),
        "p01": np.quantile(flat, 0.01),
        "p25": np.quantile(flat, 0.25),
        "median": np.quantile(flat, 0.50),
        "mean": np.mean(flat),
        "p75": np.quantile(flat, 0.75),
        "p99": np.quantile(flat, 0.99),
        "max": np.max(flat),
    }


distance_table = pd.DataFrame(
    [
        distance_summary(
            "Geographic distance (meters)",
            d_geo_val
        ),
        distance_summary(
            "EUC feature distance (z-score)",
            d_euc_val
        ),
    ]
)


print(
    "\n[1] Raw distance distributions"
)

display(
    distance_table
)


# ------------------------------------------------------------
# 4. Direct scale ratio
# ------------------------------------------------------------

eps = 1e-12

ratio_geo_to_euc = (
    d_geo_val
    /
    (d_euc_val + eps)
)


print(
    "\n[2] Geographic / EUC distance ratio"
)

print(
    "Median ratio :",
    np.median(
        ratio_geo_to_euc
    )
)

print(
    "Mean ratio   :",
    np.mean(
        ratio_geo_to_euc
    )
)

print(
    "P25 ratio    :",
    np.quantile(
        ratio_geo_to_euc,
        0.25
    )
)

print(
    "P75 ratio    :",
    np.quantile(
        ratio_geo_to_euc,
        0.75
    )
)


# ------------------------------------------------------------
# 5. Contribution diagnostic
#
# Actual GeoSIMCP formula:
#
# lambda * d_geo^2
# +
# (1-lambda) * d_feat^2
#
# Check which component actually dominates.
# ------------------------------------------------------------

print(
    "\n[3] Actual contribution inside joint distance"
)


contribution_rows = []


for lam in [
    0.1,
    0.3,
    0.5,
    0.7,
    0.9
]:

    geo_component = (
        lam
        *
        d_geo_val ** 2
    )

    feature_component = (
        (1.0 - lam)
        *
        d_euc_val ** 2
    )

    component_ratio = (
        geo_component
        /
        (
            feature_component
            +
            eps
        )
    )

    contribution_rows.append(
        {
            "lambda": lam,

            "median_geo_component":
                np.median(
                    geo_component
                ),

            "median_feature_component":
                np.median(
                    feature_component
                ),

            "median_geo/feature":
                np.median(
                    component_ratio
                ),

            "fraction_geo_component_larger":
                np.mean(
                    geo_component
                    >
                    feature_component
                )
        }
    )


contribution_table = pd.DataFrame(
    contribution_rows
)


display(
    contribution_table
)


# ------------------------------------------------------------
# 6. Inspect actual kernel weights for the OFFICIAL
#    best GeoSIMCP parameters:
#
# bandwidth = 500
# lambda = 0
# ------------------------------------------------------------

official_bw = 500.0
official_lambda = 0.0


d_joint_official = np.sqrt(

    official_lambda
    *
    d_geo_val ** 2

    +

    (1.0 - official_lambda)
    *
    d_euc_val ** 2
)


weights_official = np.exp(

    -0.5
    *
    (
        d_joint_official
        /
        official_bw
    ) ** 2
)


print(
    "\n[4] Kernel weights at official best "
    "(bandwidth=500, lambda=0)"
)


print(
    "Weight min    :",
    weights_official.min()
)

print(
    "Weight median :",
    np.median(
        weights_official
    )
)

print(
    "Weight mean   :",
    weights_official.mean()
)

print(
    "Weight max    :",
    weights_official.max()
)


# ------------------------------------------------------------
# 7. Effective sample size
#
# n_eff = (sum w)^2 / sum(w^2)
#
# If all weights are almost equal,
# n_eff will be close to n_calibration = 300.
# ------------------------------------------------------------

weight_sum = np.sum(
    weights_official,
    axis=1
)

weight_sq_sum = np.sum(
    weights_official ** 2,
    axis=1
)

n_eff = (
    weight_sum ** 2
    /
    weight_sq_sum
)


print(
    "\n[5] Effective Calibration sample size"
)

print(
    "Calibration n :",
    len(X_calib)
)

print(
    "n_eff min     :",
    n_eff.min()
)

print(
    "n_eff median  :",
    np.median(
        n_eff
    )
)

print(
    "n_eff mean    :",
    n_eff.mean()
)

print(
    "n_eff max     :",
    n_eff.max()
)


# ------------------------------------------------------------
# 8. Within-test-point weight variation
# ------------------------------------------------------------

row_weight_cv = (
    np.std(
        weights_official,
        axis=1
    )
    /
    np.mean(
        weights_official,
        axis=1
    )
)


print(
    "\n[6] Within-test-point weight variation"
)

print(
    "Median coefficient of variation:",
    np.median(
        row_weight_cv
    )
)

print(
    "Maximum coefficient of variation:",
    np.max(
        row_weight_cv
    )
)


print(
    "\n✓ Distance-scale diagnostic completed."
)

print(
    "✓ Only Calibration and Validation were used."
)

print(
    "✓ Test set remains unused for this diagnosis."
)

===== GEOSIMCP DISTANCE SCALE DIAGNOSTIC =====

[1] Raw distance distributions


,distance,min,p01,p25,median,mean,p75,p99,max
0,Geographic distance (meters),33.342798,1474.136155,11017.339368,18262.275617,19845.652004,26602.183883,49292.694649,57393.586202
1,EUC feature distance (z-score),0.029169,0.865050,2.373884,3.260002,3.408772,4.190165,7.710845,11.904204



[2] Geographic / EUC distance ratio
Median ratio : 5454.950802461497
Mean ratio   : 7214.539181068913
P25 ratio    : 3062.765283306647
P75 ratio    : 9303.012329629473

[3] Actual contribution inside joint distance


,lambda,median_geo_component,median_feature_component,median_geo/feature,fraction_geo_component_larger
0,0.1,3.335107e+07,9.564854,3.306276e+06,1.0
1,0.3,1.000532e+08,7.439331,1.275278e+07,1.0
2,0.5,1.667554e+08,5.313808,2.975649e+07,1.0
3,0.7,2.334575e+08,3.188285,6.943181e+07,1.0
4,0.9,3.001596e+08,1.062762,2.678084e+08,1.0



[4] Kernel weights at official best (bandwidth=500, lambda=0)
Weight min    : 0.9997166200296302
Weight median : 0.9999787449939721
Weight mean   : 0.9999726527802993
Weight max    : 0.9999999982983301

[5] Effective Calibration sample size
Calibration n : 300
n_eff min     : 299.99999959044106
n_eff median  : 299.99999989844946
n_eff mean    : 299.9999998841567
n_eff max     : 299.9999999564963

[6] Within-test-point weight variation
Median coefficient of variation: 1.8398397202115737e-05
Maximum coefficient of variation: 3.694856310665833e-05

✓ Distance-scale diagnostic completed.
✓ Only Calibration and Validation were used.
✓ Test set remains unused for this diagnosis.


In [11]:
# ============================================================
# Step 7B — Verify two key findings
#
# 1. Manual kernel weights == official geoconformal weights
# 2. GeoSIMCP(lambda=0, bw=500) ~= global equal-weight CP
#
# Only Calibration + Validation are used.
# Test set remains untouched.
# ============================================================

import numpy as np

from geoconformal import (
    GeoSIMConformalSpatialRegression
)

from geoconformal.geocp.weights import (
    joint_geo_feature_weights
)

from geoconformal.geocp import (
    weighted_quantile
)


print("===== STEP 7B: IMPLEMENTATION VERIFICATION =====")


# ------------------------------------------------------------
# Part A — Compare manual weights with official weight function
# ------------------------------------------------------------

official_weight_fn = joint_geo_feature_weights(

    coord_calib=coords_calib,

    feat_calib=X_calib,

    bandwidth=500.0,

    lambda_weight=0.0,

    distance_metric="euclidean",

    standardize=True
)


official_input_val = np.hstack([
    coords_val,
    X_val
])


weights_from_library = official_weight_fn(
    official_input_val,
    None
)


max_weight_difference = np.max(
    np.abs(
        weights_from_library
        -
        weights_official
    )
)


print("\n[1] Manual vs official library weights")

print(
    "Shape manual  :",
    weights_official.shape
)

print(
    "Shape library :",
    weights_from_library.shape
)

print(
    "Maximum absolute difference:",
    max_weight_difference
)


# ------------------------------------------------------------
# Part B — Global equal-weight conformal quantile
# ------------------------------------------------------------

calibration_predictions = model.predict(
    X_calib_spatial
)

calibration_scores = np.abs(
    y_calib
    -
    calibration_predictions
)


equal_weights = np.ones(
    (
        1,
        len(calibration_scores)
    )
)


q_global = weighted_quantile(

    scores=calibration_scores,

    weights=equal_weights,

    q=0.9,

    self_weights=None,

    interpolate=False
)[0]


validation_predictions = model.predict(
    X_val_spatial
)


global_lower = (
    validation_predictions
    -
    q_global
)

global_upper = (
    validation_predictions
    +
    q_global
)


global_coverage = np.mean(
    (y_val >= global_lower)
    &
    (y_val <= global_upper)
)


global_mean_interval = np.mean(
    global_upper
    -
    global_lower
)


global_interval_score = interval_score(

    global_upper,

    global_lower,

    y_val,

    alpha=0.1
)


# ------------------------------------------------------------
# Part C — Official GeoSIMCP at lambda=0, bandwidth=500
# ------------------------------------------------------------

simcp_feature_only = GeoSIMConformalSpatialRegression(

    predict_f=model.predict,

    miscoverage_level=0.1,

    bandwidth=500.0,

    lambda_weight=0.0,

    distance_metric="euclidean",

    standardize_weights=True,

    coord_calib=coords_calib,
    coord_test=coords_val,

    X_calib=X_calib_spatial,
    y_calib=y_calib,

    X_test=X_val_spatial,
    y_test=y_val,

    X_calib_weight=X_calib,
    X_test_weight=X_val,

    include_test_atom=False
)


result_feature_only = (
    simcp_feature_only.analyze()
)


simcp_half_width = (
    np.asarray(
        result_feature_only.upper_bound
    )
    -
    np.asarray(
        result_feature_only.lower_bound
    )
) / 2.0


simcp_mean_interval = np.mean(
    result_feature_only.upper_bound
    -
    result_feature_only.lower_bound
)


simcp_interval_score = interval_score(

    result_feature_only.upper_bound,

    result_feature_only.lower_bound,

    y_val,

    alpha=0.1
)


# ------------------------------------------------------------
# Part D — Compare the two
# ------------------------------------------------------------

print(
    "\n[2] Global equal-weight CP"
)

print(
    "q*             =",
    q_global
)

print(
    "Coverage       =",
    global_coverage
)

print(
    "Mean Interval  =",
    global_mean_interval
)

print(
    "Interval Score =",
    global_interval_score
)


print(
    "\n[3] GeoSIMCP feature-only"
)

print(
    "Coverage       =",
    result_feature_only.coverage_probability
)

print(
    "Mean Interval  =",
    simcp_mean_interval
)

print(
    "Interval Score =",
    simcp_interval_score
)


print(
    "\n[4] Local q* variation in GeoSIMCP"
)

print(
    "q* min         =",
    simcp_half_width.min()
)

print(
    "q* median      =",
    np.median(
        simcp_half_width
    )
)

print(
    "q* mean        =",
    simcp_half_width.mean()
)

print(
    "q* max         =",
    simcp_half_width.max()
)

print(
    "Number of unique q* values:",
    len(
        np.unique(
            np.round(
                simcp_half_width,
                12
            )
        )
    )
)


print(
    "\n[5] Difference from global CP"
)

print(
    "Maximum |q*_GeoSIMCP - q*_global| =",
    np.max(
        np.abs(
            simcp_half_width
            -
            q_global
        )
    )
)


print(
    "\n✓ Verification completed."
)

print(
    "✓ Calibration + Validation only."
)

print(
    "✓ Test set remains untouched."
)

===== STEP 7B: IMPLEMENTATION VERIFICATION =====

[1] Manual vs official library weights
Shape manual  : (300, 300)
Shape library : (300, 300)
Maximum absolute difference: 4.647393581080905e-13

[2] Global equal-weight CP
q*             = 0.12706775548098825
Coverage       = 0.92
Mean Interval  = 0.2541355109619765
Interval Score = 0.3210766044099667

[3] GeoSIMCP feature-only
Coverage       = 0.92
Mean Interval  = 0.254190041466038
Interval Score = 0.3209948086538744

[4] Local q* variation in GeoSIMCP
q* min         = 0.12706775548098825
q* median      = 0.12706775548098825
q* mean        = 0.127095020733019
q* max         = 0.12911264938329392
Number of unique q* values: 2

[5] Difference from global CP
Maximum |q*_GeoSIMCP - q*_global| = 0.0020448939023056667

✓ Verification completed.
✓ Calibration + Validation only.
✓ Test set remains untouched.


In [12]:
# ============================================================
# Step 8 — Check whether current official Train split
# is identical to the previous frozen Outer Train split
#
# 只比较样本索引，不运行 GCCM
# ============================================================

from pathlib import Path
import numpy as np


PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OLD_SPLIT_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "seattle_split_seed42.npz"
)


print("===== TRAIN SPLIT CONSISTENCY CHECK =====")


# ------------------------------------------------------------
# 1. Reconstruct current official split WITH row indices
# ------------------------------------------------------------

all_indices = np.arange(
    len(data)
)


idx_train_current, idx_temp_current = (
    train_test_split(
        all_indices,
        test_size=0.3,
        random_state=42
    )
)


# ------------------------------------------------------------
# 2. Load previous frozen Outer Train indices
# ------------------------------------------------------------

old_split = np.load(
    OLD_SPLIT_PATH
)

idx_train_old = np.asarray(
    old_split["train_indices"],
    dtype=int
)


# ------------------------------------------------------------
# 3. Compare
# ------------------------------------------------------------

same_length = (
    len(idx_train_current)
    ==
    len(idx_train_old)
)

same_set = (
    set(idx_train_current.tolist())
    ==
    set(idx_train_old.tolist())
)

same_order = np.array_equal(
    idx_train_current,
    idx_train_old
)


print(
    "Current official Train n :",
    len(idx_train_current)
)

print(
    "Previous Outer Train n   :",
    len(idx_train_old)
)

print(
    "\nSame length :",
    same_length
)

print(
    "Same samples:",
    same_set
)

print(
    "Same order  :",
    same_order
)


# ------------------------------------------------------------
# 4. Difference diagnostic
# ------------------------------------------------------------

only_current = sorted(
    set(idx_train_current.tolist())
    -
    set(idx_train_old.tolist())
)

only_old = sorted(
    set(idx_train_old.tolist())
    -
    set(idx_train_current.tolist())
)


print(
    "\nSamples only in current Train:",
    len(only_current)
)

print(
    "Samples only in old Train    :",
    len(only_old)
)


print(
    "\n✓ Split comparison finished."
)
print(
    "No GCCM model was run."
)

===== TRAIN SPLIT CONSISTENCY CHECK =====
Current official Train n : 2100
Previous Outer Train n   : 2100

Same length : True
Same samples: True
Same order  : True

Samples only in current Train: 0
Samples only in old Train    : 0

✓ Split comparison finished.
No GCCM model was run.


In [13]:
# ============================================================
# Step 9 — Build author-aligned GCCM Train input
#
# 关键原则：
# 1. 只使用官方 70% Train（2100）
# 2. Target = log_price
# 3. Calibration / Validation / Test 不参与 GCCM
# 4. 重新构建 Delaunay，但不覆盖旧 GCCM 文件
# 5. 暂时不运行 GCCM
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.spatial import Delaunay


print(
    "===== AUTHOR-ALIGNED GCCM TRAIN PREPARATION ====="
)


# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

GCCM_ALIGNED_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
)

GCCM_ALIGNED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Variables
# ------------------------------------------------------------

causal_feature_cols = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]

gccm_target = "log_price"


# ------------------------------------------------------------
# 3. Build record-level Train table
#
# idx_train_current = official 70% Train indices
# already verified identical to previous Outer Train
# ------------------------------------------------------------

gccm_train_records = (
    data.iloc[
        idx_train_current
    ][
        [
            "UTM_X",
            "UTM_Y",
            *causal_feature_cols,
            gccm_target
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "\n[1] Record-level Train"
)

print(
    "Shape:",
    gccm_train_records.shape
)

print(
    "Missing values:",
    gccm_train_records.isna().sum().sum()
)

print(
    "Target:",
    gccm_target
)


assert len(gccm_train_records) == 2100
assert gccm_train_records.isna().sum().sum() == 0


# ------------------------------------------------------------
# 4. Aggregate duplicate coordinates within Train only
#
# This reproduces the previous point-GCCM adaptation:
# houses at exactly the same spatial location are averaged.
# ------------------------------------------------------------

gccm_train_sites = (
    gccm_train_records
    .groupby(
        [
            "UTM_X",
            "UTM_Y"
        ],
        sort=False,
        as_index=False
    )
    .mean(
        numeric_only=True
    )
)


print(
    "\n[2] Site-level Train"
)

print(
    "Unique spatial sites:",
    len(gccm_train_sites)
)

print(
    "Records aggregated:",
    len(gccm_train_records)
    -
    len(gccm_train_sites)
)

print(
    "Remaining duplicated coordinates:",
    gccm_train_sites
    .duplicated(
        subset=[
            "UTM_X",
            "UTM_Y"
        ]
    )
    .sum()
)


assert len(gccm_train_sites) == 2088

assert (
    gccm_train_sites
    .duplicated(
        subset=[
            "UTM_X",
            "UTM_Y"
        ]
    )
    .sum()
    ==
    0
)


# ------------------------------------------------------------
# 5. Rebuild Delaunay graph
# ------------------------------------------------------------

site_coords = (
    gccm_train_sites[
        [
            "UTM_X",
            "UTM_Y"
        ]
    ]
    .to_numpy(
        dtype=float
    )
)

n_sites = len(
    site_coords
)


tri = Delaunay(
    site_coords
)

neighbor_sets = [
    set()
    for _ in range(
        n_sites
    )
]

edge_set = set()


for triangle in tri.simplices:

    a, b, c = map(
        int,
        triangle
    )

    for i, j in [
        (a, b),
        (b, c),
        (c, a)
    ]:

        if i == j:
            continue

        u, v = sorted(
            (
                i,
                j
            )
        )

        edge_set.add(
            (
                u,
                v
            )
        )

        neighbor_sets[i].add(j)
        neighbor_sets[j].add(i)


degrees = np.array(
    [
        len(neigh)
        for neigh in neighbor_sets
    ],
    dtype=int
)


print(
    "\n[3] Delaunay graph"
)

print(
    "Delaunay triangles:",
    len(
        tri.simplices
    )
)

print(
    "Undirected edges:",
    len(
        edge_set
    )
)

print(
    "Mean degree:",
    degrees.mean()
)

print(
    "Min degree:",
    degrees.min()
)

print(
    "Max degree:",
    degrees.max()
)


zero_neighbor_sites = np.sum(
    degrees == 0
)

symmetric_graph = all(

    i in neighbor_sets[j]

    for i in range(
        n_sites
    )

    for j in neighbor_sets[i]
)

self_neighbor = any(

    i in neighbor_sets[i]

    for i in range(
        n_sites
    )
)


print(
    "Zero-neighbor sites:",
    zero_neighbor_sites
)

print(
    "Adjacency symmetric:",
    symmetric_graph
)

print(
    "Any self-neighbors:",
    self_neighbor
)


assert len(tri.simplices) == 4158
assert len(edge_set) == 6245
assert zero_neighbor_sites == 0
assert symmetric_graph
assert not self_neighbor


# ------------------------------------------------------------
# 6. Compare reconstructed graph with OLD saved graph
#
# This is only a topology consistency check.
# Old causal weights are NOT reused.
# ------------------------------------------------------------

OLD_EDGE_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
    / "seattle_gccm_outer_train_delaunay_edges.csv"
)


old_graph_exists = (
    OLD_EDGE_PATH.exists()
)


print(
    "\n[4] Graph consistency with previous GCCM"
)

print(
    "Old edge file exists:",
    old_graph_exists
)


if old_graph_exists:

    old_edges_df = pd.read_csv(
        OLD_EDGE_PATH
    )

    old_undirected_edges = set()

    for row in old_edges_df.itertuples(
        index=False
    ):

        # old file is 1-based
        i = int(
            row.from_site
        ) - 1

        j = int(
            row.to_site
        ) - 1

        u, v = sorted(
            (
                i,
                j
            )
        )

        old_undirected_edges.add(
            (
                u,
                v
            )
        )

    same_graph = (
        edge_set
        ==
        old_undirected_edges
    )

    print(
        "Old undirected edges:",
        len(
            old_undirected_edges
        )
    )

    print(
        "Exact same graph:",
        same_graph
    )

    assert same_graph

else:

    same_graph = None

    print(
        "Old graph unavailable; "
        "structural checks still passed."
    )


# ------------------------------------------------------------
# 7. Spatial detrending
#
# Each variable:
#
# Z = beta0 + beta_x * UTM_X + beta_y * UTM_Y + residual
#
# GCCM uses residuals.
# ------------------------------------------------------------

site_variables = (
    causal_feature_cols
    +
    [
        gccm_target
    ]
)


site_xy_centered = (
    site_coords
    -
    site_coords.mean(
        axis=0
    )
)


design = np.column_stack(
    [
        np.ones(
            len(
                site_xy_centered
            )
        ),
        site_xy_centered
    ]
)


gccm_train_detrended = (
    gccm_train_sites.copy()
)


detrend_diagnostics = []


for var in site_variables:

    z = (
        gccm_train_sites[
            var
        ]
        .to_numpy(
            dtype=float
        )
    )

    beta, *_ = np.linalg.lstsq(
        design,
        z,
        rcond=None
    )

    fitted = (
        design
        @
        beta
    )

    residual = (
        z
        -
        fitted
    )

    residual_col = (
        f"{var}_resid"
    )

    gccm_train_detrended[
        residual_col
    ] = residual


    ss_total = np.sum(
        (
            z
            -
            z.mean()
        ) ** 2
    )

    ss_resid = np.sum(
        residual ** 2
    )

    trend_r2 = (
        1
        -
        ss_resid
        /
        ss_total

        if ss_total > 0

        else np.nan
    )


    detrend_diagnostics.append(
        {
            "variable":
                var,

            "raw_mean":
                z.mean(),

            "raw_std":
                z.std(
                    ddof=1
                ),

            "residual_mean":
                residual.mean(),

            "residual_std":
                residual.std(
                    ddof=1
                ),

            "spatial_trend_R2":
                trend_r2,
        }
    )


detrend_diagnostics = pd.DataFrame(
    detrend_diagnostics
)


print(
    "\n[5] Spatial detrending"
)

display(
    detrend_diagnostics.round(
        6
    )
)


# ------------------------------------------------------------
# 8. Verify residuals are orthogonal to UTM X/Y
# ------------------------------------------------------------

spatial_check = []


for var in site_variables:

    residual = (
        gccm_train_detrended[
            f"{var}_resid"
        ]
        .to_numpy(
            dtype=float
        )
    )

    spatial_check.append(
        {
            "variable":
                var,

            "corr_resid_UTM_X":
                np.corrcoef(
                    residual,
                    site_coords[:, 0]
                )[0, 1],

            "corr_resid_UTM_Y":
                np.corrcoef(
                    residual,
                    site_coords[:, 1]
                )[0, 1],
        }
    )


spatial_check = pd.DataFrame(
    spatial_check
)


max_abs_corr = np.nanmax(
    np.abs(
        spatial_check[
            [
                "corr_resid_UTM_X",
                "corr_resid_UTM_Y"
            ]
        ]
        .to_numpy()
    )
)


print(
    "\nResidual-coordinate check:"
)

display(
    spatial_check.round(
        12
    )
)

print(
    "Maximum absolute correlation:",
    max_abs_corr
)


residual_cols = [
    f"{var}_resid"
    for var in site_variables
]


assert np.isfinite(
    gccm_train_detrended[
        residual_cols
    ].to_numpy()
).all()

assert all(
    gccm_train_detrended[
        col
    ].std(
        ddof=1
    ) > 0
    for col in residual_cols
)


# ------------------------------------------------------------
# 9. Export NEW author-aligned preprocessing files
#    Do NOT overwrite old GCCM results
# ------------------------------------------------------------

sites_export = (
    gccm_train_detrended.copy()
)

sites_export.insert(
    0,
    "site_id",
    np.arange(
        1,
        len(
            sites_export
        ) + 1
    )
)


SITE_OUTPUT_PATH = (
    GCCM_ALIGNED_DIR
    /
    "seattle_gccm_train_logprice_sites_v1.csv"
)


EDGE_OUTPUT_PATH = (
    GCCM_ALIGNED_DIR
    /
    "seattle_gccm_train_delaunay_edges_v1.csv"
)


DETREND_OUTPUT_PATH = (
    GCCM_ALIGNED_DIR
    /
    "seattle_gccm_train_logprice_detrend_diagnostics_v1.csv"
)


sites_export.to_csv(
    SITE_OUTPUT_PATH,
    index=False
)


edge_rows = []

for i, neighs in enumerate(
    neighbor_sets
):

    for j in sorted(
        neighs
    ):

        edge_rows.append(
            {
                "from_site":
                    i + 1,

                "to_site":
                    int(j) + 1,
            }
        )


edges_export = pd.DataFrame(
    edge_rows
)


edges_export.to_csv(
    EDGE_OUTPUT_PATH,
    index=False
)


detrend_diagnostics.to_csv(
    DETREND_OUTPUT_PATH,
    index=False
)


print(
    "\n[6] Saved NEW author-aligned files"
)

print(
    SITE_OUTPUT_PATH
)

print(
    EDGE_OUTPUT_PATH
)

print(
    DETREND_OUTPUT_PATH
)


print(
    "\n===== FINAL CHECK ====="
)

print(
    "Train records       :",
    len(
        gccm_train_records
    )
)

print(
    "Unique sites        :",
    len(
        gccm_train_sites
    )
)

print(
    "Delaunay triangles  :",
    len(
        tri.simplices
    )
)

print(
    "Undirected edges    :",
    len(
        edge_set
    )
)

print(
    "Target used by GCCM :",
    gccm_target
)

print(
    "Max residual-coordinate correlation:",
    f"{max_abs_corr:.3e}"
)


print(
    "\n✓ Author-aligned GCCM Train preprocessing complete."
)

print(
    "✓ Calibration / Validation / Test were NOT used."
)

print(
    "✓ Old GCCM outputs were NOT overwritten."
)

print(
    "✓ GCCM itself has NOT been run yet."
)

===== AUTHOR-ALIGNED GCCM TRAIN PREPARATION =====

[1] Record-level Train
Shape: (2100, 11)
Missing values: 0
Target: log_price

[2] Site-level Train
Unique spatial sites: 2088
Records aggregated: 12
Remaining duplicated coordinates: 0

[3] Delaunay graph
Delaunay triangles: 4158
Undirected edges: 6245
Mean degree: 5.981800766283525
Min degree: 3
Max degree: 13
Zero-neighbor sites: 0
Adjacency symmetric: True
Any self-neighbors: False

[4] Graph consistency with previous GCCM
Old edge file exists: True
Old undirected edges: 6245
Exact same graph: True

[5] Spatial detrending


,variable,raw_mean,raw_std,residual_mean,residual_std,spatial_trend_R2
0,bathrooms,2.053400,0.764773,-0.0,0.750506,0.036961
1,sqft_living,1974.029215,860.934689,0.0,834.837177,0.059707
2,sqft_lot,7460.718151,3716.876372,0.0,3435.312990,0.145767
3,grade,7.514847,1.119916,-0.0,1.092752,0.047922
4,condition,3.444205,0.683592,0.0,0.683321,0.000791
5,waterfront,0.005268,0.072408,0.0,0.072400,0.000237
6,view,0.246648,0.770807,-0.0,0.765958,0.012540
7,age,48.553161,29.421369,0.0,27.608672,0.119427
8,log_price,5.655894,0.232491,-0.0,0.207186,0.205839



Residual-coordinate check:


,variable,corr_resid_UTM_X,corr_resid_UTM_Y
0,bathrooms,-0.0,-0.0
1,sqft_living,-0.0,-0.0
2,sqft_lot,-0.0,0.0
3,grade,-0.0,0.0
4,condition,0.0,-0.0
5,waterfront,0.0,-0.0
6,view,0.0,0.0
7,age,0.0,0.0
8,log_price,0.0,-0.0


Maximum absolute correlation: 8.963354198094182e-16

[6] Saved NEW author-aligned files
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\seattle_gccm_train_logprice_sites_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\seattle_gccm_train_delaunay_edges_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\seattle_gccm_train_logprice_detrend_diagnostics_v1.csv

===== FINAL CHECK =====
Train records       : 2100
Unique sites        : 2088
Delaunay triangles  : 4158
Undirected edges    : 6245
Target used by GCCM : log_price
Max residual-coordinate correlation: 8.963e-16

✓ Author-aligned GCCM Train preprocessing complete.
✓ Calibration / Validation / Test were NOT used.
✓ Old GCCM outputs were NOT overwritten.
✓ GCCM itself has NOT been run yet.


In [14]:
# ============================================================
# Step 10 — Audit existing GCCM R scripts before rerunning
#
# 目的：
# 1. 找到电脑里之前真正使用的 GCCM R 脚本
# 2. 检查 library size、E、GCCMLattice、target 等设置
# 3. 暂时不运行 R，不修改任何脚本
# ============================================================

from pathlib import Path
import os
import shutil


print("===== GCCM R SCRIPT AUDIT =====")


# ------------------------------------------------------------
# 1. Possible GCCM directories
# ------------------------------------------------------------

GCCM_WORK_ROOT = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "02_gccm"
)

print(
    "\n[1] GCCM working root:"
)

print(
    GCCM_WORK_ROOT
)

print(
    "Exists:",
    GCCM_WORK_ROOT.exists()
)


# ------------------------------------------------------------
# 2. Locate R / Rscript
# ------------------------------------------------------------

print(
    "\n[2] R environment"
)

rscript_candidates = [

    shutil.which("Rscript"),
    shutil.which("Rscript.exe"),

    Path(
        r"C:\Program Files\R\R-4.6.1\bin\Rscript.exe"
    ),

    Path(
        r"C:\Program Files\R\R-4.6.1\bin\x64\Rscript.exe"
    ),
]


valid_rscript_paths = []

for candidate in rscript_candidates:

    if candidate is None:
        continue

    candidate = Path(
        candidate
    )

    if (
        candidate.exists()
        and
        candidate not in valid_rscript_paths
    ):
        valid_rscript_paths.append(
            candidate
        )


if len(valid_rscript_paths) == 0:

    print(
        "Rscript: NOT FOUND"
    )

else:

    for p in valid_rscript_paths:

        print(
            "Rscript FOUND:",
            p
        )


# ------------------------------------------------------------
# 3. Find all R scripts under 02_gccm
# ------------------------------------------------------------

print(
    "\n[3] R scripts under 02_gccm"
)


r_files = []


if GCCM_WORK_ROOT.exists():

    for pattern in [
        "*.R",
        "*.r"
    ]:

        r_files.extend(
            GCCM_WORK_ROOT.rglob(
                pattern
            )
        )


# remove duplicates
r_files = sorted(
    set(
        r_files
    ),
    key=lambda p: str(p).lower()
)


print(
    "Number of R scripts found:",
    len(r_files)
)


for i, path in enumerate(
    r_files,
    start=1
):

    print(
        f"{i:02d}.",
        path
    )


# ------------------------------------------------------------
# 4. Search scripts for GCCM-related keywords
# ------------------------------------------------------------

keywords = [

    "GCCMLattice",
    "GCCMSingle4Lattice",

    "lib_sizes",
    "libSize",
    "library",

    "E=",
    "E =",

    "price_resid",
    "log_price_resid",

    "rho",
    "spearman",
    "converged",

    "p_two_sided",
    "CI_lower",

    "delaunay",
    "edges",

    "seattle_gccm",
]


print(
    "\n[4] Relevant lines inside R scripts"
)


script_hits = {}


for path in r_files:

    try:

        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception as e:

        print(
            "\nCould not read:",
            path,
            repr(e)
        )

        continue


    lines = text.splitlines()

    hits = []


    for line_no, line in enumerate(
        lines,
        start=1
    ):

        if any(
            kw.lower() in line.lower()
            for kw in keywords
        ):

            hits.append(
                (
                    line_no,
                    line
                )
            )


    if len(hits) > 0:

        script_hits[path] = hits


        print(
            "\n" + "=" * 90
        )

        print(
            path
        )

        print(
            "=" * 90
        )


        # Print at most first 80 relevant lines
        for line_no, line in hits[:80]:

            print(
                f"{line_no:4d}: {line}"
            )


# ------------------------------------------------------------
# 5. Specifically identify likely Seattle runner scripts
# ------------------------------------------------------------

print(
    "\n[5] Likely Seattle GCCM runner scripts"
)


likely_runner_scripts = []


for path, hits in script_hits.items():

    full_text_lower = (
        path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
        .lower()
    )

    score = 0

    for marker in [
        "gccmlattice",
        "seattle",
        "price_resid",
        "delaunay",
        "lib"
    ]:

        if marker in full_text_lower:

            score += 1


    if score >= 2:

        likely_runner_scripts.append(
            (
                score,
                path
            )
        )


likely_runner_scripts = sorted(
    likely_runner_scripts,
    key=lambda x: (
        -x[0],
        str(x[1]).lower()
    )
)


if len(likely_runner_scripts) == 0:

    print(
        "No obvious custom Seattle runner identified."
    )

else:

    for score, path in likely_runner_scripts:

        print(
            f"score={score}:",
            path
        )


# ------------------------------------------------------------
# 6. Check previous GCCM output files
# ------------------------------------------------------------

OLD_GCCM_OUTPUT = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
)


print(
    "\n[6] Previous GCCM diagnostic/output files"
)


if OLD_GCCM_OUTPUT.exists():

    old_files = sorted(
        [
            p
            for p in OLD_GCCM_OUTPUT.iterdir()
            if p.is_file()
        ],
        key=lambda p: p.name.lower()
    )

    for p in old_files:

        if any(
            token in p.name.lower()
            for token in [
                "convergence",
                "gccm",
                "causal",
                "outer_train"
            ]
        ):

            print(
                p.name
            )


print(
    "\n✓ R/GCCM audit completed."
)

print(
    "✓ No R script was executed."
)

print(
    "✓ No existing file was modified."
)

===== GCCM R SCRIPT AUDIT =====

[1] GCCM working root:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm
Exists: True

[2] R environment
Rscript FOUND: C:\Program Files\R\R-4.6.1\bin\Rscript.exe
Rscript FOUND: C:\Program Files\R\R-4.6.1\bin\x64\Rscript.exe

[3] R scripts under 02_gccm
Number of R scripts found: 15
01. E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm\GCCM_updated 2\GUI\GCCMRcode\coderaster\DeletethelinearityofXandY.r
02. E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm\GCCM_updated 2\GUI\GCCMRcode\coderaster\DeletethelinearityofXdetrendwithY.r
03. E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm\GCCM_updated 2\GUI\GCCMRcode\coderaster\DeletethelinearityofYdetrendwithX.r
04. E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm\GCCM_updated 2\GUI\GCCMRcode\coderaster\Noneedtodeletelinearize.r
05. E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm\GCCM_updated 2\GUI\GCCMRcode\codeshp\DeletethelinearityofXand

In [15]:
# ============================================================
# Step 10B — Recover the exact previous GCCM run specification
#
# 从旧的 Outer-Train GCCM 原始结果中恢复：
# 1. 实际 library sizes
# 2. 每个 library size 有多少重复结果
# 3. raw result 的列结构
# 4. 8 个 feature 是否使用完全相同的 L grid
#
# 不运行 GCCM
# 不修改任何旧文件
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "===== RECOVER PREVIOUS GCCM RUN SPECIFICATION ====="
)


# ------------------------------------------------------------
# 1. Previous GCCM output directory
# ------------------------------------------------------------

OLD_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
)


feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


print(
    "\n[1] Old GCCM output directory"
)

print(
    OLD_GCCM_DIR
)

print(
    "Exists:",
    OLD_GCCM_DIR.exists()
)


# ------------------------------------------------------------
# 2. Locate the 8 previous Outer-Train raw convergence files
#
# Scientific direction:
# feature -> price
#
# GCCM reverse cross-map evidence:
# price xmap feature
# ------------------------------------------------------------

raw_files = {}


for feature in feature_order:

    path = (
        OLD_GCCM_DIR
        /
        f"seattle_gccm_outer_train_price_xmap_{feature}_convergence_raw.csv"
    )

    raw_files[feature] = path


print(
    "\n[2] Raw convergence files"
)


all_exist = True


for feature, path in raw_files.items():

    exists = path.exists()

    all_exist = (
        all_exist
        and
        exists
    )

    print(
        f"{feature:15s}:",
        "FOUND" if exists else "MISSING"
    )


print(
    "\nAll 8 files found:",
    all_exist
)


assert all_exist


# ------------------------------------------------------------
# 3. Inspect every raw file
# ------------------------------------------------------------

audit_rows = []

L_grids = {}


for feature, path in raw_files.items():

    df_raw = pd.read_csv(
        path
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        "FEATURE:",
        feature
    )

    print(
        "FILE:",
        path.name
    )

    print(
        "Shape:",
        df_raw.shape
    )

    print(
        "Columns:",
        df_raw.columns.tolist()
    )


    # ----------------------------------------------
    # Identify likely library-size column
    # ----------------------------------------------

    L_candidates = [
        col
        for col in df_raw.columns
        if col.lower() in [
            "l",
            "lib_size",
            "library_size",
            "libsize"
        ]
    ]


    if len(L_candidates) == 0:

        print(
            "WARNING: no obvious L column found."
        )

        display(
            df_raw.head(10)
        )

        continue


    L_col = L_candidates[0]


    L_values = (
        pd.to_numeric(
            df_raw[L_col],
            errors="coerce"
        )
        .dropna()
        .unique()
    )


    L_values = np.sort(
        L_values
    )


    L_grids[feature] = (
        L_values.copy()
    )


    print(
        "Library-size column:",
        L_col
    )

    print(
        "Unique library sizes:"
    )

    print(
        L_values
    )


    # ----------------------------------------------
    # Number of raw results per L
    # ----------------------------------------------

    counts_by_L = (
        df_raw
        .groupby(
            L_col
        )
        .size()
        .reset_index(
            name="n_rows"
        )
    )


    print(
        "\nRows per library size:"
    )

    display(
        counts_by_L
    )


    # ----------------------------------------------
    # rho summary, if present
    # ----------------------------------------------

    rho_candidates = [
        col
        for col in df_raw.columns
        if "rho" in col.lower()
    ]


    if len(rho_candidates) > 0:

        rho_col = rho_candidates[0]

        rho_summary = (
            df_raw
            .groupby(
                L_col
            )[rho_col]
            .agg(
                [
                    "count",
                    "mean",
                    "std",
                    "min",
                    "max"
                ]
            )
            .reset_index()
        )


        print(
            "\nRho summary by L:"
        )

        display(
            rho_summary
        )


    # ----------------------------------------------
    # Save audit information
    # ----------------------------------------------

    audit_rows.append(
        {
            "feature":
                feature,

            "n_rows":
                len(
                    df_raw
                ),

            "n_columns":
                df_raw.shape[1],

            "L_column":
                L_col,

            "n_unique_L":
                len(
                    L_values
                ),

            "L_min":
                np.min(
                    L_values
                ),

            "L_max":
                np.max(
                    L_values
                ),

            "L_values":
                ",".join(
                    str(
                        int(x)
                        if float(x).is_integer()
                        else float(x)
                    )
                    for x in L_values
                )
        }
    )


# ------------------------------------------------------------
# 4. Check whether all 8 features used the same L grid
# ------------------------------------------------------------

print(
    "\n===== L-GRID CONSISTENCY ====="
)


reference_feature = (
    feature_order[0]
)

reference_grid = (
    L_grids[
        reference_feature
    ]
)


same_L_grid = True


for feature in feature_order[1:]:

    same = np.array_equal(
        reference_grid,
        L_grids[
            feature
        ]
    )

    print(
        f"{feature:15s}:",
        "SAME" if same else "DIFFERENT"
    )

    same_L_grid = (
        same_L_grid
        and
        same
    )


print(
    "\nAll 8 use identical library-size grid:",
    same_L_grid
)


if same_L_grid:

    print(
        "\nFrozen previous L grid:"
    )

    print(
        reference_grid
    )


# ------------------------------------------------------------
# 5. Summary table
# ------------------------------------------------------------

audit_table = pd.DataFrame(
    audit_rows
)


print(
    "\n===== PREVIOUS GCCM RAW OUTPUT SUMMARY ====="
)

display(
    audit_table
)


# ------------------------------------------------------------
# 6. Check old convergence diagnostics
# ------------------------------------------------------------

OLD_DIAG_PATH = (
    OLD_GCCM_DIR
    /
    "seattle_gccm_outer_train_convergence_diagnostics.csv"
)


print(
    "\n[3] Previous convergence diagnostics"
)

print(
    "Exists:",
    OLD_DIAG_PATH.exists()
)


if OLD_DIAG_PATH.exists():

    old_diag = pd.read_csv(
        OLD_DIAG_PATH
    )

    display(
        old_diag
    )

    print(
        "\nDiagnostic columns:"
    )

    print(
        old_diag.columns.tolist()
    )


print(
    "\n✓ Previous GCCM specification audit complete."
)

print(
    "✓ GCCM was NOT rerun."
)

print(
    "✓ No file was modified."
)

===== RECOVER PREVIOUS GCCM RUN SPECIFICATION =====

[1] Old GCCM output directory
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm
Exists: True

[2] Raw convergence files
bathrooms      : FOUND
sqft_living    : FOUND
sqft_lot       : FOUND
grade          : FOUND
condition      : FOUND
waterfront     : FOUND
view           : FOUND
age            : FOUND

All 8 files found: True

FEATURE: bathrooms
FILE: seattle_gccm_outer_train_price_xmap_bathrooms_convergence_raw.csv
Shape: (10431, 2)
Columns: ['L', 'rho']
Library-size column: L
Unique library sizes:
[  10   50  100  200  400 2086]

Rows per library size:


,L,n_rows
0,10,2086
1,50,2086
2,100,2086
3,200,2086
4,400,2086
5,2086,1



Rho summary by L:


,L,count,mean,std,min,max
0,10,2086,0.093654,0.127377,-0.266889,0.281514
1,50,2086,0.125428,0.059726,-0.136010,0.248862
2,100,2086,0.133224,0.044170,-0.004837,0.208055
3,200,2086,0.139426,0.034481,0.041696,0.206183
4,400,2086,0.147415,0.024601,0.089921,0.198636
5,2086,1,0.143220,NaN,0.143220,0.143220



FEATURE: sqft_living
FILE: seattle_gccm_outer_train_price_xmap_sqft_living_convergence_raw.csv
Shape: (10431, 2)
Columns: ['L', 'rho']
Library-size column: L
Unique library sizes:
[  10   50  100  200  400 2086]

Rows per library size:


...


===== L-GRID CONSISTENCY =====
sqft_living    : SAME
sqft_lot       : SAME
grade          : SAME
condition      : SAME
waterfront     : SAME
view           : SAME
age            : SAME

All 8 use identical library-size grid: True

Frozen previous L grid:
[  10   50  100  200  400 2086]

===== PREVIOUS GCCM RAW OUTPUT SUMMARY =====


,feature,n_rows,n_columns,L_column,n_unique_L,L_min,L_max,L_values
0,bathrooms,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
1,sqft_living,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
2,sqft_lot,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
3,grade,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
4,condition,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
5,waterfront,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
6,view,10431,2,L,6,10,2086,"10,50,100,200,400,2086"
7,age,10431,2,L,6,10,2086,"10,50,100,200,400,2086"



[3] Previous convergence diagnostics
Exists: True


,feature,rho_L10,rho_Lmax,gain,spearman,p_two_sided,CI_lower,CI_upper,converged
0,age,0.005681,0.055566,0.049885,1.000000,1.113930e-02,0.012679,0.098250,True
1,bathrooms,0.093654,0.143220,0.049566,0.942857,4.990586e-11,0.100922,0.185000,True
2,condition,0.014780,0.025398,0.010618,0.657143,2.462664e-01,-0.017539,0.068241,False
3,grade,0.247137,0.310584,0.063447,0.657143,0.000000e+00,0.271282,0.348851,True
4,sqft_living,0.175639,0.270903,0.095264,0.942857,0.000000e+00,0.230668,0.310214,True
5,sqft_lot,0.049763,0.171303,0.121540,1.000000,3.330669e-15,0.129337,0.212658,True
6,view,0.098185,0.296150,0.197965,1.000000,0.000000e+00,0.256492,0.334812,True
7,waterfront,0.021254,0.164455,0.143201,1.000000,4.085621e-14,0.122401,0.205919,True



Diagnostic columns:
['feature', 'rho_L10', 'rho_Lmax', 'gain', 'spearman', 'p_two_sided', 'CI_lower', 'CI_upper', 'converged']

✓ Previous GCCM specification audit complete.
✓ GCCM was NOT rerun.
✓ No file was modified.


In [16]:
# ============================================================
# Step 11A — Reproduce one OLD GCCM terminal result
#
# 目的：
# 使用旧 price 数据重新运行：
#
#     price xmap sqft_living
#
# 但只运行最大 library size L = 2086。
#
# 如果得到 rho ≈ 0.270903，
# 说明新的 R runner 与旧 GCCM 主实验计算链一致。
#
# 这一步：
# - 不使用 log_price
# - 不使用 Calibration / Validation / Test
# - 不修改旧结果
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import pandas as pd
import numpy as np


print(
    "===== OLD GCCM TERMINAL REPRODUCTION SMOKE TEST ====="
)


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

OLD_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
)

OLD_SITES_PATH = (
    OLD_GCCM_DIR
    / "seattle_gccm_outer_train_sites.csv"
)

OLD_EDGES_PATH = (
    OLD_GCCM_DIR
    / "seattle_gccm_outer_train_delaunay_edges.csv"
)

R_CORE_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "02_gccm"
    / "GCCM_updated 2"
    / "R"
    / "GCCM"
)

GCCM4_PATH = (
    R_CORE_DIR
    / "GCCM4Lattice.r"
)

BASIC_PATH = (
    R_CORE_DIR
    / "basic.r"
)

R_SCRIPT_EXE = Path(
    r"C:\Program Files\R\R-4.6.1\bin\Rscript.exe"
)


SMOKE_DIR = (
    GCCM_ALIGNED_DIR
    / "smoke_test"
)

SMOKE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


R_RUNNER_PATH = (
    SMOKE_DIR
    / "reproduce_old_price_xmap_sqft_living_Lmax.R"
)

SMOKE_OUTPUT_PATH = (
    SMOKE_DIR
    / "old_price_xmap_sqft_living_L2086.csv"
)


# ------------------------------------------------------------
# 2. Preflight
# ------------------------------------------------------------

required_paths = [
    OLD_SITES_PATH,
    OLD_EDGES_PATH,
    GCCM4_PATH,
    BASIC_PATH,
    R_SCRIPT_EXE,
]


print(
    "\n[1] Preflight"
)

for p in required_paths:

    print(
        p.name,
        ":",
        "FOUND" if p.exists() else "MISSING"
    )


assert all(
    p.exists()
    for p in required_paths
)


# ------------------------------------------------------------
# 3. Inspect old site columns
# ------------------------------------------------------------

old_sites_preview = pd.read_csv(
    OLD_SITES_PATH,
    nrows=5
)


print(
    "\n[2] Required residual columns"
)

print(
    "price_resid:",
    "FOUND"
    if "price_resid" in old_sites_preview.columns
    else "MISSING"
)

print(
    "sqft_living_resid:",
    "FOUND"
    if "sqft_living_resid" in old_sites_preview.columns
    else "MISSING"
)


assert "price_resid" in old_sites_preview.columns
assert "sqft_living_resid" in old_sites_preview.columns


# ------------------------------------------------------------
# 4. Helper: convert Windows path for R
# ------------------------------------------------------------

def r_path(path):
    return Path(path).resolve().as_posix()


# ------------------------------------------------------------
# 5. Write a NEW independent R runner
#
# Important:
# - official GCCM4Lattice.r is sourced unchanged
# - E = 3
# - b = E + 1 = 4
# - lib = pred = all 2088 sites
# - L = 2086 only
#
# Direction:
#
# feature -> price hypothesis
# is assessed by reverse cross-map:
#
# price manifold xmap feature
#
# Therefore:
# embedding variable = price_resid
# target variable    = sqft_living_resid
# ------------------------------------------------------------

r_code = f"""
library(parallel)
library(foreach)
library(doParallel)

source("{r_path(GCCM4_PATH)}")
source("{r_path(BASIC_PATH)}")

sites <- read.csv(
    "{r_path(OLD_SITES_PATH)}",
    check.names = FALSE
)

edges <- read.csv(
    "{r_path(OLD_EDGES_PATH)}"
)

n <- nrow(sites)

# ------------------------------------------------------------
# Reconstruct adjacency list from directed edge table
# ------------------------------------------------------------

neighborMatrix <- vector(
    "list",
    n
)

for (i in 1:n) {{

    neighborMatrix[[i]] <- as.integer(
        edges$to_site[
            edges$from_site == i
        ]
    )
}}

# Basic topology checks
stopifnot(
    length(neighborMatrix) == n
)

stopifnot(
    all(
        sapply(
            neighborMatrix,
            length
        ) > 0
    )
)

# ------------------------------------------------------------
# Frozen GCCM settings
# ------------------------------------------------------------

E <- 3

b <- E + 1

lib <- c(
    1,
    n
)

pred <- c(
    1,
    n
)

# GCCMLattice internally removes E-1 initial positions,
# giving max library = 2088 - 2 = 2086.
lib_sizes <- c(
    2086
)

# ------------------------------------------------------------
# Reverse cross-map for hypothesis:
#
# sqft_living -> price
#
# Therefore:
# price xmap sqft_living
# ------------------------------------------------------------

x <- as.numeric(
    sites$price_resid
)

y <- as.numeric(
    sites$sqft_living_resid
)

# Build the price spatial manifold
embeddings <- generateEmbedings(
    neighborMatrix,
    x,
    E
)

# Reconstruct sqft_living from the price manifold
result <- GCCMLattice(
    embeddings,
    y,
    lib_sizes,
    lib,
    pred,
    E,
    b = b,
    cores = NULL
)

write.csv(
    result,
    "{r_path(SMOKE_OUTPUT_PATH)}",
    row.names = FALSE
)

cat(
    "\\n===== R SMOKE TEST RESULT =====\\n"
)

print(
    result
)

cat(
    "\\nn =", n,
    "\\nE =", E,
    "\\nb =", b,
    "\\nL =", lib_sizes,
    "\\n"
)
"""


R_RUNNER_PATH.write_text(
    textwrap.dedent(
        r_code
    ),
    encoding="utf-8"
)


print(
    "\n[3] New smoke-test R script saved:"
)

print(
    R_RUNNER_PATH
)


# ------------------------------------------------------------
# 6. Execute R runner
# ------------------------------------------------------------

print(
    "\n[4] Running R..."
)

completed = subprocess.run(

    [
        str(R_SCRIPT_EXE),
        str(R_RUNNER_PATH)
    ],

    capture_output=True,
    text=True,
    errors="ignore"
)


print(
    "\nR return code:",
    completed.returncode
)


print(
    "\n----- R STDOUT -----"
)

print(
    completed.stdout
)


if completed.stderr.strip():

    print(
        "\n----- R STDERR -----"
    )

    print(
        completed.stderr
    )


assert completed.returncode == 0


# ------------------------------------------------------------
# 7. Read result and compare with frozen old result
# ------------------------------------------------------------

smoke_result = pd.read_csv(
    SMOKE_OUTPUT_PATH
)


print(
    "\n[5] Reproduction comparison"
)

display(
    smoke_result
)


rho_new = float(
    smoke_result.loc[
        smoke_result["L"] == 2086,
        "rho"
    ].iloc[0]
)


# Frozen previous raw result
old_raw_path = (
    OLD_GCCM_DIR
    / "seattle_gccm_outer_train_price_xmap_sqft_living_convergence_raw.csv"
)

old_raw = pd.read_csv(
    old_raw_path
)


rho_old = float(
    old_raw.loc[
        old_raw["L"] == 2086,
        "rho"
    ].iloc[0]
)


abs_difference = abs(
    rho_new
    -
    rho_old
)


print(
    "Old rho(L=2086) :",
    rho_old
)

print(
    "New rho(L=2086) :",
    rho_new
)

print(
    "Absolute difference:",
    abs_difference
)


if abs_difference < 1e-10:

    print(
        "\n✓ EXACT numerical reproduction passed."
    )

elif abs_difference < 1e-6:

    print(
        "\n✓ Numerical reproduction passed "
        "(tiny floating-point difference)."
    )

else:

    print(
        "\n✗ Reproduction mismatch."
    )

    print(
        "STOP here. Do NOT run log_price GCCM yet."
    )


print(
    "\n✓ Old files were not modified."
)

print(
    "✓ Calibration / Validation / Test were not used."
)

===== OLD GCCM TERMINAL REPRODUCTION SMOKE TEST =====

[1] Preflight
seattle_gccm_outer_train_sites.csv : FOUND
seattle_gccm_outer_train_delaunay_edges.csv : FOUND
GCCM4Lattice.r : FOUND
basic.r : FOUND
Rscript.exe : FOUND

[2] Required residual columns
price_resid: FOUND
sqft_living_resid: FOUND

[3] New smoke-test R script saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\smoke_test\reproduce_old_price_xmap_sqft_living_Lmax.R

[4] Running R...

R return code: 0

----- R STDOUT -----

===== R SMOKE TEST RESULT =====
     L       rho
1 2086 0.2709034

n = 2088 
E = 3 
b = 4 
L = 2086 


----- R STDERR -----
Loading required package: iterators


[5] Reproduction comparison


,L,rho
0,2086,0.270903


Old rho(L=2086) : 0.27090336995428
New rho(L=2086) : 0.27090336995428
Absolute difference: 0.0

✓ EXACT numerical reproduction passed.

✓ Old files were not modified.
✓ Calibration / Validation / Test were not used.


In [17]:
# ============================================================
# Step 11B — FULL old GCCM reproduction
#
# 重新计算旧：
#
#     price xmap sqft_living
#
# 完整 library grid:
# [10, 50, 100, 200, 400, 2086]
#
# 目标：
# 验证 cyclic library mechanism（循环库机制）
# 以及完整 raw convergence output（原始收敛输出）。
#
# 仍然不使用 log_price。
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import numpy as np
import pandas as pd


print(
    "===== FULL OLD GCCM REPRODUCTION ====="
)


# ------------------------------------------------------------
# 1. Output files
# ------------------------------------------------------------

FULL_REPRO_R_PATH = (
    SMOKE_DIR
    / "reproduce_old_price_xmap_sqft_living_full.R"
)

FULL_REPRO_OUTPUT_PATH = (
    SMOKE_DIR
    / "old_price_xmap_sqft_living_full_reproduced.csv"
)


# ------------------------------------------------------------
# 2. Frozen GCCM settings
# ------------------------------------------------------------

FULL_L_GRID = [
    10,
    50,
    100,
    200,
    400,
    2086,
]

E_GCCM = 3
B_GCCM = E_GCCM + 1


print(
    "\nFrozen settings:"
)

print(
    "E:",
    E_GCCM
)

print(
    "b:",
    B_GCCM
)

print(
    "Library sizes:",
    FULL_L_GRID
)


# ------------------------------------------------------------
# 3. Build R runner
# ------------------------------------------------------------

r_code_full = f"""
library(parallel)
library(foreach)
library(doParallel)

source("{r_path(GCCM4_PATH)}")
source("{r_path(BASIC_PATH)}")

sites <- read.csv(
    "{r_path(OLD_SITES_PATH)}",
    check.names = FALSE
)

edges <- read.csv(
    "{r_path(OLD_EDGES_PATH)}"
)

n <- nrow(sites)


# ------------------------------------------------------------
# Delaunay adjacency
# ------------------------------------------------------------

neighborMatrix <- vector(
    "list",
    n
)

for (i in 1:n) {{

    neighborMatrix[[i]] <- as.integer(
        edges$to_site[
            edges$from_site == i
        ]
    )
}}


stopifnot(
    length(neighborMatrix) == n
)

stopifnot(
    all(
        sapply(
            neighborMatrix,
            length
        ) > 0
    )
)


# ------------------------------------------------------------
# Frozen GCCM parameters
# ------------------------------------------------------------

E <- 3
b <- E + 1

lib <- c(
    1,
    n
)

pred <- c(
    1,
    n
)

lib_sizes <- c(
    10,
    50,
    100,
    200,
    400,
    2086
)


# ------------------------------------------------------------
# Scientific hypothesis:
#
# sqft_living -> price
#
# GCCM reverse map:
#
# price xmap sqft_living
#
# ------------------------------------------------------------

x <- as.numeric(
    sites$price_resid
)

y <- as.numeric(
    sites$sqft_living_resid
)


# Price manifold
embeddings <- generateEmbedings(
    neighborMatrix,
    x,
    E
)


cat(
    "\\nStarting full GCCM run...\\n"
)

start_time <- Sys.time()


# ------------------------------------------------------------
# Use official GCCMLattice unchanged.
#
# cores = 8:
# official polygon example also uses 8 cores.
# No random sampling is introduced by this function.
# ------------------------------------------------------------

result <- GCCMLattice(
    embeddings,
    y,
    lib_sizes,
    lib,
    pred,
    E,
    b = b,
    cores = 8
)


end_time <- Sys.time()


write.csv(
    result,
    "{r_path(FULL_REPRO_OUTPUT_PATH)}",
    row.names = FALSE
)


cat(
    "\\n===== FULL GCCM RUN FINISHED =====\\n"
)

cat(
    "Runtime minutes:",
    as.numeric(
        difftime(
            end_time,
            start_time,
            units = "mins"
        )
    ),
    "\\n"
)

cat(
    "Rows:",
    nrow(result),
    "\\n"
)

print(
    table(
        result$L
    )
)

cat(
    "\\nMean rho by L:\\n"
)

print(
    aggregate(
        rho ~ L,
        data = result,
        FUN = mean,
        na.rm = TRUE
    )
)
"""


FULL_REPRO_R_PATH.write_text(
    textwrap.dedent(
        r_code_full
    ),
    encoding="utf-8"
)


print(
    "\nR runner saved:"
)

print(
    FULL_REPRO_R_PATH
)


# ------------------------------------------------------------
# 4. Execute R
# ------------------------------------------------------------

print(
    "\nRunning full GCCM reproduction..."
)

print(
    "This is substantially heavier than the L=2086 smoke test."
)


completed_full = subprocess.run(

    [
        str(R_SCRIPT_EXE),
        str(FULL_REPRO_R_PATH)
    ],

    capture_output=True,
    text=True,
    errors="ignore"
)


print(
    "\nR return code:",
    completed_full.returncode
)


print(
    "\n----- R STDOUT -----"
)

print(
    completed_full.stdout
)


if completed_full.stderr.strip():

    print(
        "\n----- R STDERR -----"
    )

    print(
        completed_full.stderr
    )


assert completed_full.returncode == 0


# ------------------------------------------------------------
# 5. Load new and old raw outputs
# ------------------------------------------------------------

new_full = pd.read_csv(
    FULL_REPRO_OUTPUT_PATH
)


old_full_path = (
    OLD_GCCM_DIR
    /
    "seattle_gccm_outer_train_price_xmap_sqft_living_convergence_raw.csv"
)


old_full = pd.read_csv(
    old_full_path
)


print(
    "\n===== RAW OUTPUT STRUCTURE ====="
)

print(
    "Old shape:",
    old_full.shape
)

print(
    "New shape:",
    new_full.shape
)


print(
    "\nOld columns:",
    old_full.columns.tolist()
)

print(
    "New columns:",
    new_full.columns.tolist()
)


# ------------------------------------------------------------
# 6. Compare row counts by L
# ------------------------------------------------------------

old_counts = (
    old_full
    .groupby("L")
    .size()
)

new_counts = (
    new_full
    .groupby("L")
    .size()
)


count_comparison = pd.DataFrame(
    {
        "old_n":
            old_counts,

        "new_n":
            new_counts,
    }
)


count_comparison[
    "same"
] = (

    count_comparison[
        "old_n"
    ]

    ==

    count_comparison[
        "new_n"
    ]
)


print(
    "\n[1] Rows per library size"
)

display(
    count_comparison
)


# ------------------------------------------------------------
# 7. Direct raw rho comparison
#
# GCCMSingle4Lattice is deterministic.
# If row order is preserved, values should match directly.
# ------------------------------------------------------------

same_shape = (
    old_full.shape
    ==
    new_full.shape
)


if same_shape:

    direct_diff = np.abs(

        old_full[
            "rho"
        ].to_numpy()

        -

        new_full[
            "rho"
        ].to_numpy()
    )


    print(
        "\n[2] Direct raw-row comparison"
    )

    print(
        "Maximum absolute rho difference:",
        np.nanmax(
            direct_diff
        )
    )

    print(
        "Mean absolute rho difference:",
        np.nanmean(
            direct_diff
        )
    )

else:

    direct_diff = None

    print(
        "\nDirect comparison unavailable "
        "because shapes differ."
    )


# ------------------------------------------------------------
# 8. Compare mean convergence curve
# ------------------------------------------------------------

old_curve = (
    old_full
    .groupby(
        "L"
    )["rho"]
    .mean()
)


new_curve = (
    new_full
    .groupby(
        "L"
    )["rho"]
    .mean()
)


curve_compare = pd.DataFrame(
    {
        "old_mean_rho":
            old_curve,

        "new_mean_rho":
            new_curve,
    }
)


curve_compare[
    "absolute_difference"
] = np.abs(

    curve_compare[
        "old_mean_rho"
    ]

    -

    curve_compare[
        "new_mean_rho"
    ]
)


print(
    "\n[3] Mean convergence curve comparison"
)

display(
    curve_compare
)


# ------------------------------------------------------------
# 9. Final reproduction decision
# ------------------------------------------------------------

structure_pass = (

    same_shape

    and

    count_comparison[
        "same"
    ].all()
)


curve_pass = (

    curve_compare[
        "absolute_difference"
    ].max()

    < 1e-10
)


raw_pass = (

    direct_diff is not None

    and

    np.nanmax(
        direct_diff
    )

    < 1e-10
)


print(
    "\n===== FINAL REPRODUCTION CHECK ====="
)

print(
    "Structure pass:",
    structure_pass
)

print(
    "Mean curve pass:",
    curve_pass
)

print(
    "Raw row-by-row pass:",
    raw_pass
)


if (
    structure_pass
    and
    curve_pass
    and
    raw_pass
):

    print(
        "\n✓ FULL EXACT GCCM REPRODUCTION PASSED."
    )

    print(
        "✓ The R runner can now be frozen."
    )

else:

    print(
        "\n⚠ Full reproduction is not exact."
    )

    print(
        "Do NOT switch to log_price yet."
    )


print(
    "\n✓ Old GCCM files were not modified."
)

print(
    "✓ Calibration / Validation / Test were not used."
)

===== FULL OLD GCCM REPRODUCTION =====

Frozen settings:
E: 3
b: 4
Library sizes: [10, 50, 100, 200, 400, 2086]

R runner saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\smoke_test\reproduce_old_price_xmap_sqft_living_full.R

Running full GCCM reproduction...
This is substantially heavier than the L=2086 smoke test.

R return code: 0

----- R STDOUT -----

Starting full GCCM run...

===== FULL GCCM RUN FINISHED =====
Runtime minutes: 5.695585 
Rows: 10431 

  10   50  100  200  400 2086 
2086 2086 2086 2086 2086    1 

Mean rho by L:
     L       rho
1   10 0.1756389
2   50 0.2364875
3  100 0.2530547
4  200 0.2648696
5  400 0.2746333
6 2086 0.2709034


----- R STDERR -----
Loading required package: iterators


===== RAW OUTPUT STRUCTURE =====
Old shape: (10431, 2)
New shape: (10431, 2)

Old columns: ['L', 'rho']
New columns: ['L', 'rho']

[1] Rows per library size


,old_n,new_n,same
L,,,
10,2086,2086,True
50,2086,2086,True
100,2086,2086,True
200,2086,2086,True
400,2086,2086,True
2086,1,1,True



[2] Direct raw-row comparison
Maximum absolute rho difference: 0.0
Mean absolute rho difference: 0.0

[3] Mean convergence curve comparison


,old_mean_rho,new_mean_rho,absolute_difference
L,,,
10,0.175639,0.175639,0.0
50,0.236488,0.236488,0.0
100,0.253055,0.253055,0.0
200,0.264870,0.264870,0.0
400,0.274633,0.274633,0.0
2086,0.270903,0.270903,0.0



===== FINAL REPRODUCTION CHECK =====
Structure pass: True
Mean curve pass: True
Raw row-by-row pass: True

✓ FULL EXACT GCCM REPRODUCTION PASSED.
✓ The R runner can now be frozen.

✓ Old GCCM files were not modified.
✓ Calibration / Validation / Test were not used.


In [18]:
# ============================================================
# Step 12A — Author-aligned log_price GCCM
#
# 正式重新计算 8 个 causal feature 的 GCCM：
#
# Scientific hypothesis:
#
#     feature -> log_price
#
# GCCM reverse cross-map evidence:
#
#     log_price xmap feature
#
# 已冻结并验证的设置：
#
# Train          = official 70% Train only
# Unique sites   = 2088
# Graph          = Delaunay
# E              = 3
# b              = E + 1 = 4
# L              = [10, 50, 100, 200, 400, 2086]
# cores          = 8
#
# 唯一相对旧 GCCM 的实质改变：
#
#     price_resid -> log_price_resid
#
# Calibration / Validation / Test 均不参与 GCCM。
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import time

import numpy as np
import pandas as pd


print(
    "===== AUTHOR-ALIGNED LOG_PRICE GCCM ====="
)


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

GCCM_ALIGNED_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
)

GCCM_ALIGNED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SITES_PATH = (
    GCCM_ALIGNED_DIR
    / "seattle_gccm_train_logprice_sites_v1.csv"
)

EDGES_PATH = (
    GCCM_ALIGNED_DIR
    / "seattle_gccm_train_delaunay_edges_v1.csv"
)


R_CORE_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "02_gccm"
    / "GCCM_updated 2"
    / "R"
    / "GCCM"
)

GCCM4_PATH = (
    R_CORE_DIR
    / "GCCM4Lattice.r"
)

BASIC_PATH = (
    R_CORE_DIR
    / "basic.r"
)


R_SCRIPT_EXE = Path(
    r"C:\Program Files\R\R-4.6.1\bin\Rscript.exe"
)


R_RUNNER_PATH = (
    GCCM_ALIGNED_DIR
    / "run_seattle_logprice_gccm_v1.R"
)


# ------------------------------------------------------------
# 2. Frozen feature order
# ------------------------------------------------------------

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


# ------------------------------------------------------------
# 3. Preflight
# ------------------------------------------------------------

print(
    "\n[1] Preflight"
)


required_paths = [
    SITES_PATH,
    EDGES_PATH,
    GCCM4_PATH,
    BASIC_PATH,
    R_SCRIPT_EXE,
]


for path in required_paths:

    print(
        f"{path.name:55s}:",
        "FOUND"
        if path.exists()
        else "MISSING"
    )


assert all(
    path.exists()
    for path in required_paths
)


# ------------------------------------------------------------
# 4. Check author-aligned site file
# ------------------------------------------------------------

sites_preview = pd.read_csv(
    SITES_PATH,
    nrows=5
)


print(
    "\n[2] Residual columns"
)

print(
    "log_price_resid:",
    "FOUND"
    if "log_price_resid" in sites_preview.columns
    else "MISSING"
)


assert "log_price_resid" in sites_preview.columns


for feature in feature_order:

    residual_col = (
        f"{feature}_resid"
    )

    print(
        f"{residual_col:25s}:",
        "FOUND"
        if residual_col in sites_preview.columns
        else "MISSING"
    )

    assert residual_col in sites_preview.columns


# ------------------------------------------------------------
# 5. Helper for R paths
# ------------------------------------------------------------

def r_path(path):

    return (
        Path(path)
        .resolve()
        .as_posix()
    )


# ------------------------------------------------------------
# 6. Build output paths
# ------------------------------------------------------------

raw_output_paths = {}


for feature in feature_order:

    raw_output_paths[
        feature
    ] = (
        GCCM_ALIGNED_DIR
        /
        (
            "seattle_gccm_train_"
            f"logprice_xmap_{feature}_"
            "convergence_raw_v1.csv"
        )
    )


# ------------------------------------------------------------
# 7. Build R vector strings
# ------------------------------------------------------------

r_features = (
    "c("
    +
    ",".join(
        f'"{feature}"'
        for feature in feature_order
    )
    +
    ")"
)


# R named output vector
r_output_pairs = []

for feature in feature_order:

    output_path = raw_output_paths[
        feature
    ]

    r_output_pairs.append(
        f'"{feature}"="{r_path(output_path)}"'
    )


r_outputs = (
    "c("
    +
    ",".join(
        r_output_pairs
    )
    +
    ")"
)


# ------------------------------------------------------------
# 8. Create frozen R runner
# ------------------------------------------------------------

r_code = f"""
library(parallel)
library(foreach)
library(doParallel)

# ============================================================
# OFFICIAL GCCM CORE
#
# These source files are used unchanged.
# ============================================================

source("{r_path(GCCM4_PATH)}")
source("{r_path(BASIC_PATH)}")


# ============================================================
# SEATTLE AUTHOR-ALIGNED INPUT
# ============================================================

sites <- read.csv(
    "{r_path(SITES_PATH)}",
    check.names = FALSE
)

edges <- read.csv(
    "{r_path(EDGES_PATH)}"
)

n <- nrow(sites)


cat(
    "\\n============================================\\n"
)

cat(
    "Seattle author-aligned log_price GCCM\\n"
)

cat(
    "============================================\\n"
)

cat(
    "Sites:",
    n,
    "\\n"
)


stopifnot(
    n == 2088
)


# ============================================================
# SEATTLE POINT-GRAPH ADAPTATION
#
# Reconstruct Delaunay adjacency list.
# ============================================================

neighborMatrix <- vector(
    "list",
    n
)


for (i in 1:n) {{

    neighborMatrix[[i]] <- as.integer(
        edges$to_site[
            edges$from_site == i
        ]
    )
}}


stopifnot(
    length(neighborMatrix) == n
)


stopifnot(
    all(
        sapply(
            neighborMatrix,
            length
        ) > 0
    )
)


# ============================================================
# FROZEN GCCM SETTINGS
# ============================================================

E <- 3

b <- E + 1

lib <- c(
    1,
    n
)

pred <- c(
    1,
    n
)

lib_sizes <- c(
    10,
    50,
    100,
    200,
    400,
    2086
)


cat(
    "E:",
    E,
    "\\n"
)

cat(
    "b:",
    b,
    "\\n"
)

cat(
    "Library sizes:",
    paste(
        lib_sizes,
        collapse = ", "
    ),
    "\\n"
)


# ============================================================
# CAUSAL DIRECTION
#
# Scientific hypothesis:
#
#     feature -> log_price
#
# Under CCM/GCCM reverse mapping:
#
#     log_price xmap feature
#
# Therefore:
#
# embedding/manifold variable = log_price_resid
# reconstruction target       = feature_resid
# ============================================================

x_effect <- as.numeric(
    sites$log_price_resid
)


# Build log_price spatial manifold ONCE.
#
# It is identical for all 8 feature targets.
effect_embeddings <- generateEmbedings(
    neighborMatrix,
    x_effect,
    E
)


features <- {r_features}

output_paths <- {r_outputs}


# ============================================================
# RUN 8 FEATURES SEQUENTIALLY
#
# Each GCCMLattice call internally uses 8 cores.
# ============================================================

runtime_summary <- data.frame(
    feature = character(),
    runtime_minutes = numeric(),
    rows = integer(),
    rho_Lmax = numeric(),
    stringsAsFactors = FALSE
)


for (
    feature_index
    in seq_along(
        features
    )
) {{

    feature <- features[
        feature_index
    ]


    target_col <- paste0(
        feature,
        "_resid"
    )


    cat(
        "\\n--------------------------------------------\\n"
    )

    cat(
        "[",
        feature_index,
        "/",
        length(features),
        "] ",
        feature,
        "\\n",
        sep = ""
    )

    cat(
        "Direction: log_price xmap ",
        feature,
        "\\n",
        sep = ""
    )


    y_feature <- as.numeric(
        sites[
            [
                target_col
            ]
        ]
    )


    start_time <- Sys.time()


    result <- GCCMLattice(

        effect_embeddings,

        y_feature,

        lib_sizes,

        lib,

        pred,

        E,

        b = b,

        cores = 8
    )


    end_time <- Sys.time()


    runtime_minutes <- as.numeric(
        difftime(
            end_time,
            start_time,
            units = "mins"
        )
    )


    write.csv(

        result,

        output_paths[
            feature
        ],

        row.names = FALSE
    )


    rho_Lmax <- result$rho[
        result$L == 2086
    ][1]


    runtime_summary <- rbind(

        runtime_summary,

        data.frame(

            feature =
                feature,

            runtime_minutes =
                runtime_minutes,

            rows =
                nrow(result),

            rho_Lmax =
                rho_Lmax,

            stringsAsFactors = FALSE
        )
    )


    cat(
        "Runtime minutes:",
        runtime_minutes,
        "\\n"
    )

    cat(
        "Rows:",
        nrow(result),
        "\\n"
    )

    cat(
        "rho(L=2086):",
        rho_Lmax,
        "\\n"
    )


    cat(
        "Mean rho by L:\\n"
    )

    print(
        aggregate(
            rho ~ L,
            data = result,
            FUN = mean,
            na.rm = TRUE
        )
    )
}}


# ============================================================
# SAVE RUNTIME SUMMARY
# ============================================================

runtime_path <- file.path(

    dirname(
        output_paths[
            features[1]
        ]
    ),

    "seattle_gccm_train_logprice_runtime_summary_v1.csv"
)


write.csv(
    runtime_summary,
    runtime_path,
    row.names = FALSE
)


cat(
    "\\n============================================\\n"
)

cat(
    "ALL 8 GCCM RUNS FINISHED\\n"
)

cat(
    "============================================\\n"
)

print(
    runtime_summary
)

cat(
    "\\nTotal runtime minutes:",
    sum(
        runtime_summary$runtime_minutes
    ),
    "\\n"
)
"""


R_RUNNER_PATH.write_text(
    textwrap.dedent(
        r_code
    ),
    encoding="utf-8"
)


print(
    "\n[3] Frozen author-aligned R runner"
)

print(
    R_RUNNER_PATH
)


# ------------------------------------------------------------
# 9. Execute R WITH LIVE OUTPUT
# ------------------------------------------------------------

print(
    "\n[4] Starting formal log_price GCCM..."
)

print(
    "Expected total runtime: roughly 40–50 minutes."
)

print(
    "Do not run another GCCM job at the same time."
)

print()


start_python_time = time.time()


process = subprocess.Popen(

    [
        str(R_SCRIPT_EXE),
        str(R_RUNNER_PATH)
    ],

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    errors="ignore",

    bufsize=1
)


# Print R output live
for line in process.stdout:

    print(
        line,
        end=""
    )


return_code = process.wait()


elapsed_python_minutes = (
    time.time()
    -
    start_python_time
) / 60.0


print(
    "\n===== R PROCESS FINISHED ====="
)

print(
    "Return code:",
    return_code
)

print(
    "Python-observed runtime minutes:",
    elapsed_python_minutes
)


assert return_code == 0


# ------------------------------------------------------------
# 10. Validate all 8 raw output files
# ------------------------------------------------------------

print(
    "\n[5] Raw output validation"
)


expected_counts = {
    10: 2086,
    50: 2086,
    100: 2086,
    200: 2086,
    400: 2086,
    2086: 1,
}


validation_rows = []


for feature in feature_order:

    path = raw_output_paths[
        feature
    ]


    exists = path.exists()


    if not exists:

        validation_rows.append(
            {
                "feature":
                    feature,

                "file_exists":
                    False,

                "n_rows":
                    np.nan,

                "structure_ok":
                    False,

                "rho_Lmax":
                    np.nan,
            }
        )

        continue


    df_raw = pd.read_csv(
        path
    )


    counts = (
        df_raw
        .groupby(
            "L"
        )
        .size()
        .to_dict()
    )


    structure_ok = (

        len(df_raw) == 10431

        and

        all(
            counts.get(
                L,
                0
            )
            ==
            expected_n

            for L, expected_n
            in expected_counts.items()
        )
    )


    rho_Lmax = float(

        df_raw.loc[
            df_raw["L"] == 2086,
            "rho"
        ].iloc[0]
    )


    validation_rows.append(
        {
            "feature":
                feature,

            "file_exists":
                True,

            "n_rows":
                len(df_raw),

            "structure_ok":
                structure_ok,

            "rho_Lmax":
                rho_Lmax,
        }
    )


validation_table = pd.DataFrame(
    validation_rows
)


display(
    validation_table.round(
        6
    )
)


print(
    "\nAll 8 raw files exist:",
    validation_table[
        "file_exists"
    ].all()
)

print(
    "All 8 raw structures valid:",
    validation_table[
        "structure_ok"
    ].all()
)


assert validation_table[
    "file_exists"
].all()


assert validation_table[
    "structure_ok"
].all()


print(
    "\n✓ Author-aligned log_price GCCM raw computation complete."
)

print(
    "✓ Only official Train data were used."
)

print(
    "✓ Calibration / Validation / Test were not used."
)

print(
    "✓ Original price-based GCCM outputs were not modified."
)

===== AUTHOR-ALIGNED LOG_PRICE GCCM =====

[1] Preflight
seattle_gccm_train_logprice_sites_v1.csv               : FOUND
seattle_gccm_train_delaunay_edges_v1.csv               : FOUND
GCCM4Lattice.r                                         : FOUND
basic.r                                                : FOUND
Rscript.exe                                            : FOUND

[2] Residual columns
log_price_resid: FOUND
bathrooms_resid          : FOUND
sqft_living_resid        : FOUND
sqft_lot_resid           : FOUND
grade_resid              : FOUND
condition_resid          : FOUND
waterfront_resid         : FOUND
view_resid               : FOUND
age_resid                : FOUND

[3] Frozen author-aligned R runner
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\run_seattle_logprice_gccm_v1.R

[4] Starting formal log_price GCCM...
Expected total runtime: roughly 40–50 minutes.
Do not run another GCCM job at the same time.

Loading required package: iterators

S

AssertionError: 

In [19]:
# ============================================================
# Step 12A-fix1 — Repair R indexing syntax and parse-check
#
# 只修复上一版 R runner 中一处 Python-style indexing：
#
#     sites[ [ target_col ] ]
#
# 改成合法 R 语法：
#
#     sites[[target_col]]
#
# 本格只做语法检查，不运行 GCCM。
# ============================================================

from pathlib import Path
import subprocess


print(
    "===== FIX LOG_PRICE GCCM R RUNNER ====="
)


# ------------------------------------------------------------
# 1. Existing failed runner
# ------------------------------------------------------------

BAD_RUNNER_PATH = (
    GCCM_ALIGNED_DIR
    / "run_seattle_logprice_gccm_v1.R"
)

FIXED_RUNNER_PATH = (
    GCCM_ALIGNED_DIR
    / "run_seattle_logprice_gccm_v1_fix1.R"
)


print(
    "\n[1] Runner paths"
)

print(
    "Failed runner :",
    BAD_RUNNER_PATH
)

print(
    "Exists        :",
    BAD_RUNNER_PATH.exists()
)

print(
    "\nFixed runner  :",
    FIXED_RUNNER_PATH
)


assert BAD_RUNNER_PATH.exists()


# ------------------------------------------------------------
# 2. Read failed R runner
# ------------------------------------------------------------

r_text = BAD_RUNNER_PATH.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. Locate the exact invalid block
# ------------------------------------------------------------

bad_block = """y_feature <- as.numeric(
        sites[
            [
                target_col
            ]
        ]
    )"""


good_block = """y_feature <- as.numeric(
        sites[[target_col]]
    )"""


print(
    "\n[2] Invalid block found:",
    bad_block in r_text
)


assert bad_block in r_text, (
    "Expected invalid block was not found. "
    "Stop instead of modifying the wrong text."
)


# ------------------------------------------------------------
# 4. Replace ONLY this block
# ------------------------------------------------------------

fixed_text = r_text.replace(
    bad_block,
    good_block,
    1
)


# Confirm exactly one logical change
assert fixed_text != r_text

assert bad_block not in fixed_text

assert good_block in fixed_text


# ------------------------------------------------------------
# 5. Save as NEW runner
#
# Do NOT overwrite the failed v1 runner.
# ------------------------------------------------------------

FIXED_RUNNER_PATH.write_text(
    fixed_text,
    encoding="utf-8"
)


print(
    "\n[3] Fixed runner saved"
)

print(
    FIXED_RUNNER_PATH
)


# ------------------------------------------------------------
# 6. Show repaired section for manual audit
# ------------------------------------------------------------

fixed_lines = fixed_text.splitlines()


print(
    "\n[4] Repaired code around y_feature"
)


for i, line in enumerate(
    fixed_lines
):

    if "y_feature <-" in line:

        start = max(
            0,
            i - 5
        )

        end = min(
            len(fixed_lines),
            i + 10
        )

        for j in range(
            start,
            end
        ):

            print(
                f"{j+1:4d}: {fixed_lines[j]}"
            )

        break


# ------------------------------------------------------------
# 7. Ask R itself to PARSE the entire runner
#
# Important:
# parse() checks syntax only.
# It does NOT execute GCCM.
# ------------------------------------------------------------

print(
    "\n[5] R syntax check"
)


parse_command = (
    f'parse(file="{FIXED_RUNNER_PATH.resolve().as_posix()}"); '
    f'cat("R_PARSE_OK\\n")'
)


parse_result = subprocess.run(

    [
        str(R_SCRIPT_EXE),
        "-e",
        parse_command
    ],

    capture_output=True,
    text=True,
    errors="ignore"
)


print(
    "Return code:",
    parse_result.returncode
)

print(
    "\n----- R STDOUT -----"
)

print(
    parse_result.stdout
)


if parse_result.stderr.strip():

    print(
        "\n----- R STDERR -----"
    )

    print(
        parse_result.stderr
    )


# ------------------------------------------------------------
# 8. Final decision
# ------------------------------------------------------------

parse_pass = (

    parse_result.returncode == 0

    and

    "R_PARSE_OK" in parse_result.stdout
)


print(
    "\n===== FINAL CHECK ====="
)

print(
    "R syntax valid:",
    parse_pass
)


if parse_pass:

    print(
        "\n✓ R runner syntax check PASSED."
    )

    print(
        "✓ Only one indexing statement was repaired."
    )

    print(
        "✓ GCCM has NOT been executed."
    )

else:

    print(
        "\n✗ R syntax check FAILED."
    )

    print(
        "Do NOT start the formal GCCM run."
    )


assert parse_pass

===== FIX LOG_PRICE GCCM R RUNNER =====

[1] Runner paths
Failed runner : E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\run_seattle_logprice_gccm_v1.R
Exists        : True

Fixed runner  : E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\run_seattle_logprice_gccm_v1_fix1.R

[2] Invalid block found: True

[3] Fixed runner saved
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\run_seattle_logprice_gccm_v1_fix1.R

[4] Repaired code around y_feature
 231:         "\n",
 232:         sep = ""
 233:     )
 234: 
 235: 
 236:     y_feature <- as.numeric(
 237:         sites[[target_col]]
 238:     )
 239: 
 240: 
 241:     start_time <- Sys.time()
 242: 
 243: 
 244:     result <- GCCMLattice(
 245: 

[5] R syntax check
Return code: 0

----- R STDOUT -----
expression(library(parallel), library(foreach), library(doParallel), 
    source("E:/GeoSIMCP_Reproduce/geoconformal-main/GCCP_Seattle/02_gcc

In [20]:
# ============================================================
# Step 12B — Run the syntax-checked log_price GCCM runner
#
# 直接运行已经审核通过的：
# run_seattle_logprice_gccm_v1_fix1.R
#
# 不再修改任何 GCCM 参数。
# ============================================================

from pathlib import Path
import subprocess
import time
import numpy as np
import pandas as pd


print(
    "===== START FORMAL LOG_PRICE GCCM ====="
)


# ------------------------------------------------------------
# 1. Frozen runner
# ------------------------------------------------------------

FIXED_RUNNER_PATH = (
    GCCM_ALIGNED_DIR
    / "run_seattle_logprice_gccm_v1_fix1.R"
)


print(
    "\nRunner:"
)

print(
    FIXED_RUNNER_PATH
)

print(
    "Exists:",
    FIXED_RUNNER_PATH.exists()
)


assert FIXED_RUNNER_PATH.exists()


# ------------------------------------------------------------
# 2. Final frozen settings reminder
# ------------------------------------------------------------

print(
    "\n===== FROZEN GCCM SETTINGS ====="
)

print(
    "Train            : official 70% Train only"
)

print(
    "Unique sites     : 2088"
)

print(
    "Graph            : Delaunay"
)

print(
    "E                : 3"
)

print(
    "b                : 4"
)

print(
    "Library sizes    : [10, 50, 100, 200, 400, 2086]"
)

print(
    "Direction        : log_price xmap feature"
)

print(
    "Target manifold  : log_price_resid"
)

print(
    "Cores per feature: 8"
)

print(
    "\nCalibration / Validation / Test are NOT used."
)


# ------------------------------------------------------------
# 3. Run R with live output
# ------------------------------------------------------------

print(
    "\n===== RUNNING R ====="
)

print(
    "Expected runtime: roughly 40–50 minutes."
)

print(
    "Please do not start another GCCM job meanwhile.\n"
)


start_time = time.time()


process = subprocess.Popen(

    [
        str(R_SCRIPT_EXE),
        str(FIXED_RUNNER_PATH)
    ],

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    errors="ignore",

    bufsize=1
)


for line in process.stdout:

    print(
        line,
        end=""
    )


return_code = process.wait()


runtime_minutes = (
    time.time()
    -
    start_time
) / 60.0


print(
    "\n===== R PROCESS FINISHED ====="
)

print(
    "Return code:",
    return_code
)

print(
    "Observed runtime minutes:",
    runtime_minutes
)


assert return_code == 0


# ------------------------------------------------------------
# 4. Validate the 8 raw GCCM outputs
# ------------------------------------------------------------

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


expected_counts = {
    10: 2086,
    50: 2086,
    100: 2086,
    200: 2086,
    400: 2086,
    2086: 1,
}


validation_rows = []


for feature in feature_order:

    path = (
        GCCM_ALIGNED_DIR
        /
        (
            "seattle_gccm_train_"
            f"logprice_xmap_{feature}_"
            "convergence_raw_v1.csv"
        )
    )


    if not path.exists():

        validation_rows.append(
            {
                "feature": feature,
                "file_exists": False,
                "n_rows": np.nan,
                "structure_ok": False,
                "rho_Lmax": np.nan,
            }
        )

        continue


    df_raw = pd.read_csv(
        path
    )


    counts = (
        df_raw
        .groupby("L")
        .size()
        .to_dict()
    )


    structure_ok = (

        len(df_raw) == 10431

        and

        all(
            counts.get(L, 0) == n_expected
            for L, n_expected
            in expected_counts.items()
        )
    )


    rho_Lmax = float(
        df_raw.loc[
            df_raw["L"] == 2086,
            "rho"
        ].iloc[0]
    )


    validation_rows.append(
        {
            "feature": feature,
            "file_exists": True,
            "n_rows": len(df_raw),
            "structure_ok": structure_ok,
            "rho_Lmax": rho_Lmax,
        }
    )


validation_table = pd.DataFrame(
    validation_rows
)


print(
    "\n===== RAW OUTPUT VALIDATION ====="
)

display(
    validation_table.round(6)
)


print(
    "\nAll 8 files exist:",
    validation_table[
        "file_exists"
    ].all()
)

print(
    "All 8 structures valid:",
    validation_table[
        "structure_ok"
    ].all()
)


assert validation_table[
    "file_exists"
].all()


assert validation_table[
    "structure_ok"
].all()


print(
    "\n✓ Formal log_price GCCM raw computation completed."
)

print(
    "✓ GCCM runner remained frozen."
)

print(
    "✓ Calibration / Validation / Test were not used."
)

===== START FORMAL LOG_PRICE GCCM =====

Runner:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\run_seattle_logprice_gccm_v1_fix1.R
Exists: True

===== FROZEN GCCM SETTINGS =====
Train            : official 70% Train only
Unique sites     : 2088
Graph            : Delaunay
E                : 3
b                : 4
Library sizes    : [10, 50, 100, 200, 400, 2086]
Direction        : log_price xmap feature
Target manifold  : log_price_resid
Cores per feature: 8

Calibration / Validation / Test are NOT used.

===== RUNNING R =====
Expected runtime: roughly 40–50 minutes.
Please do not start another GCCM job meanwhile.

Loading required package: iterators

Seattle author-aligned log_price GCCM
Sites: 2088 
E: 3 
b: 4 
Library sizes: 10, 50, 100, 200, 400, 2086 

--------------------------------------------
[1/8] bathrooms
Direction: log_price xmap bathrooms
Runtime minutes: 5.417587 
Rows: 10431 
rho(L=2086): 0.1647397 
Mean rho by L:
     L        rho
1   

,feature,file_exists,n_rows,structure_ok,rho_Lmax
0,bathrooms,True,10431,True,0.164740
1,sqft_living,True,10431,True,0.241593
2,sqft_lot,True,10431,True,0.181311
3,grade,True,10431,True,0.339297
4,condition,True,10431,True,0.049273
5,waterfront,True,10431,True,0.095433
6,view,True,10431,True,0.212148
7,age,True,10431,True,0.029956



All 8 files exist: True
All 8 structures valid: True

✓ Formal log_price GCCM raw computation completed.
✓ GCCM runner remained frozen.
✓ Calibration / Validation / Test were not used.


In [21]:
# ============================================================
# Step 13 — Reproduce old GCCM diagnostics,
#           then apply EXACTLY the same rules to log_price
#
# 目的：
# A. 从旧 raw GCCM CSV 重新计算 diagnostics
#    并与旧保存结果核对，证明诊断公式无变化。
#
# B. 同一公式应用到新的 log_price GCCM。
#
# C. 计算新的 causal strength / causal weight。
#
# D. 比较 price vs log_price 的 GCCM causal structure。
#
# 不使用 Calibration / Validation / Test。
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    spearmanr,
    norm
)


print(
    "===== GCCM CONVERGENCE DIAGNOSTICS ====="
)


# ------------------------------------------------------------
# 1. Frozen settings
# ------------------------------------------------------------

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


L_GRID = np.array(
    [
        10,
        50,
        100,
        200,
        400,
        2086
    ],
    dtype=int
)


N_TERMINAL = 2086

SPEARMAN_THRESHOLD = 0.60

P_THRESHOLD = 0.05

Z_975 = norm.ppf(
    0.975
)


OLD_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
)


NEW_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
)


# ------------------------------------------------------------
# 2. Diagnostic function
#
# This reproduces the frozen previous rule:
#
# 1. rho_Lmax > rho_L10
# 2. Spearman(L, mean rho) >= 0.60
# 3. terminal two-sided p < 0.05
# 4. terminal Fisher 95% CI lower > 0
#
# causal_strength =
#     rho_Lmax if converged
#     0 otherwise
# ------------------------------------------------------------

def compute_gccm_diagnostic(
    feature,
    raw_df
):

    # ----------------------------------------------
    # Mean rho at each library size
    # ----------------------------------------------

    mean_curve = (
        raw_df
        .groupby(
            "L"
        )["rho"]
        .mean()
        .reindex(
            L_GRID
        )
    )


    assert (
        mean_curve.notna().all()
    )


    rho_L10 = float(
        mean_curve.loc[
            10
        ]
    )


    # At Lmax there is exactly one raw result.
    terminal_values = (
        raw_df.loc[
            raw_df["L"] == 2086,
            "rho"
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert len(
        terminal_values
    ) == 1


    rho_Lmax = float(
        terminal_values[0]
    )


    gain = (
        rho_Lmax
        -
        rho_L10
    )


    # ----------------------------------------------
    # Spearman convergence trend
    #
    # Use all six library sizes and their mean rho.
    # ----------------------------------------------

    spearman_value = float(
        spearmanr(
            L_GRID,
            mean_curve.to_numpy()
        ).statistic
    )


    # ----------------------------------------------
    # Terminal significance:
    # Fisher z transformation
    #
    # z = atanh(rho)
    # SE = 1 / sqrt(N - 3)
    #
    # two-sided p:
    # 2 * P(Z > |z / SE|)
    # ----------------------------------------------

    rho_safe = np.clip(
        rho_Lmax,
        -0.999999999999,
        0.999999999999
    )


    fisher_z = np.arctanh(
        rho_safe
    )


    fisher_se = (
        1.0
        /
        np.sqrt(
            N_TERMINAL
            -
            3
        )
    )


    z_stat = (
        fisher_z
        /
        fisher_se
    )


    p_two_sided = float(
        2.0
        *
        norm.sf(
            abs(
                z_stat
            )
        )
    )


    # ----------------------------------------------
    # Fisher 95% confidence interval
    # ----------------------------------------------

    CI_lower = float(
        np.tanh(
            fisher_z
            -
            Z_975
            *
            fisher_se
        )
    )


    CI_upper = float(
        np.tanh(
            fisher_z
            +
            Z_975
            *
            fisher_se
        )
    )


    # ----------------------------------------------
    # Frozen convergence screening
    # ----------------------------------------------

    converged = bool(

        (gain > 0)

        and

        (
            spearman_value
            >=
            SPEARMAN_THRESHOLD
        )

        and

        (
            p_two_sided
            <
            P_THRESHOLD
        )

        and

        (
            CI_lower
            >
            0
        )
    )


    causal_strength = (

        rho_Lmax

        if converged

        else 0.0
    )


    return {

        "feature":
            feature,

        "rho_L10":
            rho_L10,

        "rho_Lmax":
            rho_Lmax,

        "gain":
            gain,

        "spearman":
            spearman_value,

        "p_two_sided":
            p_two_sided,

        "CI_lower":
            CI_lower,

        "CI_upper":
            CI_upper,

        "converged":
            converged,

        "causal_strength":
            causal_strength,
    }


# ============================================================
# PART A
# Reconstruct OLD price diagnostics
# ============================================================

print(
    "\n[1] Reconstructing OLD price diagnostics..."
)


old_reconstructed_rows = []


for feature in feature_order:

    raw_path = (
        OLD_GCCM_DIR
        /
        (
            "seattle_gccm_outer_train_"
            f"price_xmap_{feature}_"
            "convergence_raw.csv"
        )
    )


    assert raw_path.exists()


    raw_df = pd.read_csv(
        raw_path
    )


    old_reconstructed_rows.append(
        compute_gccm_diagnostic(
            feature,
            raw_df
        )
    )


old_reconstructed = pd.DataFrame(
    old_reconstructed_rows
)


# ------------------------------------------------------------
# Load OLD saved diagnostics
# ------------------------------------------------------------

OLD_SAVED_DIAG_PATH = (
    OLD_GCCM_DIR
    /
    "seattle_gccm_outer_train_convergence_diagnostics.csv"
)


old_saved = pd.read_csv(
    OLD_SAVED_DIAG_PATH
)


old_saved = (
    old_saved
    .set_index(
        "feature"
    )
    .loc[
        feature_order
    ]
    .reset_index()
)


if old_saved[
    "converged"
].dtype == object:

    old_saved[
        "converged"
    ] = (

        old_saved[
            "converged"
        ]
        .astype(str)
        .str.lower()
        .map(
            {
                "true": True,
                "false": False
            }
        )
    )


old_reconstructed = (
    old_reconstructed
    .set_index(
        "feature"
    )
    .loc[
        feature_order
    ]
    .reset_index()
)


# ------------------------------------------------------------
# Compare old reconstructed vs old saved
# ------------------------------------------------------------

numeric_compare_cols = [
    "rho_L10",
    "rho_Lmax",
    "gain",
    "spearman",
    "p_two_sided",
    "CI_lower",
    "CI_upper",
]


old_check_rows = []


for col in numeric_compare_cols:

    diff = np.abs(

        old_reconstructed[
            col
        ].to_numpy(
            dtype=float
        )

        -

        old_saved[
            col
        ].to_numpy(
            dtype=float
        )
    )


    old_check_rows.append(
        {
            "metric":
                col,

            "max_abs_difference":
                np.nanmax(
                    diff
                )
        }
    )


old_check = pd.DataFrame(
    old_check_rows
)


boolean_match = np.array_equal(

    old_reconstructed[
        "converged"
    ].to_numpy(
        dtype=bool
    ),

    old_saved[
        "converged"
    ].to_numpy(
        dtype=bool
    )
)


print(
    "\nOld diagnostic reproduction check:"
)

display(
    old_check
)


print(
    "Converged flags identical:",
    boolean_match
)


old_reproduction_pass = (

    old_check[
        "max_abs_difference"
    ].max()

    <
    1e-6

    and

    boolean_match
)


print(
    "Old diagnostic reproduction passed:",
    old_reproduction_pass
)


assert old_reproduction_pass


# ============================================================
# PART B
# Compute NEW log_price diagnostics
# ============================================================

print(
    "\n[2] Computing NEW log_price diagnostics..."
)


new_rows = []


for feature in feature_order:

    raw_path = (
        NEW_GCCM_DIR
        /
        (
            "seattle_gccm_train_"
            f"logprice_xmap_{feature}_"
            "convergence_raw_v1.csv"
        )
    )


    assert raw_path.exists()


    raw_df = pd.read_csv(
        raw_path
    )


    new_rows.append(
        compute_gccm_diagnostic(
            feature,
            raw_df
        )
    )


new_diag = pd.DataFrame(
    new_rows
)


# ============================================================
# PART C
# Build NEW causal weights
# ============================================================

strength_sum_new = (
    new_diag[
        "causal_strength"
    ].sum()
)


assert strength_sum_new > 0


new_diag[
    "causal_weight"
] = (

    new_diag[
        "causal_strength"
    ]

    /

    strength_sum_new
)


assert np.isclose(

    new_diag[
        "causal_weight"
    ].sum(),

    1.0
)


print(
    "\n===== NEW LOG_PRICE GCCM DIAGNOSTICS ====="
)


display(
    new_diag[
        [
            "feature",
            "rho_L10",
            "rho_Lmax",
            "gain",
            "spearman",
            "p_two_sided",
            "CI_lower",
            "CI_upper",
            "converged",
            "causal_strength",
            "causal_weight",
        ]
    ].round(6)
)


print(
    "\nNew causal strength sum:",
    strength_sum_new
)

print(
    "New causal weight sum:",
    new_diag[
        "causal_weight"
    ].sum()
)


# ============================================================
# PART D
# Reconstruct OLD price weights for fair comparison
# ============================================================

old_reconstructed[
    "causal_strength"
] = np.where(

    old_reconstructed[
        "converged"
    ],

    old_reconstructed[
        "rho_Lmax"
    ],

    0.0
)


strength_sum_old = (
    old_reconstructed[
        "causal_strength"
    ].sum()
)


old_reconstructed[
    "causal_weight"
] = (

    old_reconstructed[
        "causal_strength"
    ]

    /

    strength_sum_old
)


# ============================================================
# PART E
# Compare price vs log_price causal structure
# ============================================================

comparison = (

    old_reconstructed[
        [
            "feature",
            "rho_Lmax",
            "converged",
            "causal_strength",
            "causal_weight",
        ]
    ]

    .rename(
        columns={
            "rho_Lmax":
                "price_rho_Lmax",

            "converged":
                "price_converged",

            "causal_strength":
                "price_strength",

            "causal_weight":
                "price_weight",
        }
    )

    .merge(

        new_diag[
            [
                "feature",
                "rho_Lmax",
                "converged",
                "causal_strength",
                "causal_weight",
            ]
        ]
        .rename(
            columns={
                "rho_Lmax":
                    "logprice_rho_Lmax",

                "converged":
                    "logprice_converged",

                "causal_strength":
                    "logprice_strength",

                "causal_weight":
                    "logprice_weight",
            }
        ),

        on="feature"
    )
)


comparison[
    "rho_change_log_minus_price"
] = (

    comparison[
        "logprice_rho_Lmax"
    ]

    -

    comparison[
        "price_rho_Lmax"
    ]
)


comparison[
    "weight_change_log_minus_price"
] = (

    comparison[
        "logprice_weight"
    ]

    -

    comparison[
        "price_weight"
    ]
)


print(
    "\n===== PRICE vs LOG_PRICE GCCM COMPARISON ====="
)


display(
    comparison.round(
        6
    )
)


# ------------------------------------------------------------
# Weight stability correlations
# ------------------------------------------------------------

pearson_weight_corr = float(
    np.corrcoef(

        comparison[
            "price_weight"
        ],

        comparison[
            "logprice_weight"
        ]
    )[0, 1]
)


spearman_weight_corr = float(

    comparison[
        [
            "price_weight",
            "logprice_weight"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


print(
    "\nWeight stability:"
)

print(
    "Pearson correlation :",
    pearson_weight_corr
)

print(
    "Spearman correlation:",
    spearman_weight_corr
)


# ============================================================
# PART F
# Save new author-aligned diagnostics / weights
# ============================================================

NEW_DIAG_PATH = (
    NEW_GCCM_DIR
    /
    "seattle_gccm_train_logprice_convergence_diagnostics_v1.csv"
)


NEW_WEIGHT_PATH = (
    NEW_GCCM_DIR
    /
    "seattle_gccm_train_logprice_causal_weights_v1.csv"
)


COMPARISON_PATH = (
    NEW_GCCM_DIR
    /
    "seattle_gccm_price_vs_logprice_weights_v1.csv"
)


new_diag.to_csv(
    NEW_DIAG_PATH,
    index=False
)


new_diag[
    [
        "feature",
        "causal_strength",
        "causal_weight"
    ]
].to_csv(
    NEW_WEIGHT_PATH,
    index=False
)


comparison.to_csv(
    COMPARISON_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    NEW_DIAG_PATH
)

print(
    NEW_WEIGHT_PATH
)

print(
    COMPARISON_PATH
)


print(
    "\n✓ Old diagnostic logic reproduced first."
)

print(
    "✓ Same frozen rules applied to log_price GCCM."
)

print(
    "✓ New causal weights sum to 1."
)

print(
    "✓ Calibration / Validation / Test were not used."
)

===== GCCM CONVERGENCE DIAGNOSTICS =====

[1] Reconstructing OLD price diagnostics...

Old diagnostic reproduction check:


,metric,max_abs_difference
0,rho_L10,1.942890e-16
1,rho_Lmax,0.000000e+00
2,gain,8.881784e-16
3,spearman,2.220446e-16
4,p_two_sided,3.129713e-05
5,CI_lower,7.771561e-16
6,CI_upper,3.885781e-16


Converged flags identical: True
Old diagnostic reproduction passed: False


AssertionError: 

In [22]:
# ============================================================
# Step 13A-fix — Exact reproduction of OLD GCCM diagnostics
#
# 修正：
# p_two_sided 使用 Pearson correlation t-test
# CI 继续使用 Fisher z transformation
#
# 本格只验证旧 price GCCM diagnostics。
# 暂时不计算新的 log_price causal weights。
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    spearmanr,
    norm,
    t
)


print(
    "===== EXACT OLD GCCM DIAGNOSTIC REPRODUCTION ====="
)


# ------------------------------------------------------------
# 1. Frozen settings
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OLD_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
)


feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


L_GRID = np.array(
    [
        10,
        50,
        100,
        200,
        400,
        2086
    ],
    dtype=int
)


N_TERMINAL = 2086

SPEARMAN_THRESHOLD = 0.60

P_THRESHOLD = 0.05


# ------------------------------------------------------------
# 2. Correct diagnostic function
# ------------------------------------------------------------

def compute_gccm_diagnostic_exact(
    feature,
    raw_df
):

    # --------------------------------------------------------
    # Mean convergence curve
    # --------------------------------------------------------

    mean_curve = (
        raw_df
        .groupby("L")["rho"]
        .mean()
        .reindex(L_GRID)
    )

    assert mean_curve.notna().all()


    rho_L10 = float(
        mean_curve.loc[10]
    )


    terminal_values = (
        raw_df.loc[
            raw_df["L"] == 2086,
            "rho"
        ]
        .to_numpy(
            dtype=float
        )
    )

    assert len(terminal_values) == 1


    rho_Lmax = float(
        terminal_values[0]
    )


    gain = (
        rho_Lmax
        -
        rho_L10
    )


    # --------------------------------------------------------
    # Spearman convergence trend
    # --------------------------------------------------------

    spearman_value = float(
        spearmanr(
            L_GRID,
            mean_curve.to_numpy()
        ).statistic
    )


    # --------------------------------------------------------
    # OLD p-value formula:
    # Pearson correlation significance t-test
    #
    # t = r * sqrt((N-2)/(1-r^2))
    # df = N-2
    # --------------------------------------------------------

    rho_safe = np.clip(
        rho_Lmax,
        -0.999999999999,
        0.999999999999
    )


    t_stat = (
        rho_safe
        *
        np.sqrt(
            (N_TERMINAL - 2)
            /
            (
                1.0
                -
                rho_safe ** 2
            )
        )
    )


    p_two_sided = float(
        2.0
        *
        t.sf(
            abs(t_stat),
            df=N_TERMINAL - 2
        )
    )


    # --------------------------------------------------------
    # 95% CI:
    # Fisher z transformation
    # --------------------------------------------------------

    fisher_z = np.arctanh(
        rho_safe
    )


    fisher_se = (
        1.0
        /
        np.sqrt(
            N_TERMINAL - 3
        )
    )


    z_975 = norm.ppf(
        0.975
    )


    CI_lower = float(
        np.tanh(
            fisher_z
            -
            z_975
            *
            fisher_se
        )
    )


    CI_upper = float(
        np.tanh(
            fisher_z
            +
            z_975
            *
            fisher_se
        )
    )


    # --------------------------------------------------------
    # Frozen convergence rule
    # --------------------------------------------------------

    converged = bool(

        (gain > 0)

        and

        (
            spearman_value
            >=
            SPEARMAN_THRESHOLD
        )

        and

        (
            p_two_sided
            <
            P_THRESHOLD
        )

        and

        (
            CI_lower
            >
            0
        )
    )


    return {

        "feature":
            feature,

        "rho_L10":
            rho_L10,

        "rho_Lmax":
            rho_Lmax,

        "gain":
            gain,

        "spearman":
            spearman_value,

        "p_two_sided":
            p_two_sided,

        "CI_lower":
            CI_lower,

        "CI_upper":
            CI_upper,

        "converged":
            converged,
    }


# ------------------------------------------------------------
# 3. Reconstruct OLD diagnostics from raw results
# ------------------------------------------------------------

reconstructed_rows = []


for feature in feature_order:

    raw_path = (
        OLD_GCCM_DIR
        /
        (
            "seattle_gccm_outer_train_"
            f"price_xmap_{feature}_"
            "convergence_raw.csv"
        )
    )

    assert raw_path.exists()


    raw_df = pd.read_csv(
        raw_path
    )


    reconstructed_rows.append(
        compute_gccm_diagnostic_exact(
            feature,
            raw_df
        )
    )


reconstructed = pd.DataFrame(
    reconstructed_rows
)


# ------------------------------------------------------------
# 4. Load saved OLD diagnostics
# ------------------------------------------------------------

saved_path = (
    OLD_GCCM_DIR
    /
    "seattle_gccm_outer_train_convergence_diagnostics.csv"
)


saved = pd.read_csv(
    saved_path
)


saved = (
    saved
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)


if saved["converged"].dtype == object:

    saved["converged"] = (
        saved["converged"]
        .astype(str)
        .str.lower()
        .map(
            {
                "true": True,
                "false": False
            }
        )
    )


# ------------------------------------------------------------
# 5. Compare reconstructed vs saved
# ------------------------------------------------------------

numeric_cols = [
    "rho_L10",
    "rho_Lmax",
    "gain",
    "spearman",
    "p_two_sided",
    "CI_lower",
    "CI_upper",
]


comparison_rows = []


for col in numeric_cols:

    diff = np.abs(

        reconstructed[col]
        .to_numpy(dtype=float)

        -

        saved[col]
        .to_numpy(dtype=float)
    )


    comparison_rows.append(
        {
            "metric":
                col,

            "max_abs_difference":
                np.nanmax(diff)
        }
    )


comparison_check = pd.DataFrame(
    comparison_rows
)


boolean_match = np.array_equal(

    reconstructed[
        "converged"
    ].to_numpy(dtype=bool),

    saved[
        "converged"
    ].to_numpy(dtype=bool)
)


print(
    "\n[1] Numeric reproduction"
)

display(
    comparison_check
)


print(
    "\n[2] Converged flags identical:",
    boolean_match
)


print(
    "\n[3] Reconstructed OLD diagnostics"
)

display(
    reconstructed.round(6)
)


max_difference = (
    comparison_check[
        "max_abs_difference"
    ].max()
)


reproduction_pass = (

    max_difference
    <
    1e-10

    and

    boolean_match
)


print(
    "\n===== FINAL CHECK ====="
)

print(
    "Maximum numeric difference:",
    max_difference
)

print(
    "Exact diagnostic reproduction passed:",
    reproduction_pass
)


assert reproduction_pass


print(
    "\n✓ OLD GCCM diagnostics reproduced exactly."
)

print(
    "✓ p_two_sided formula is now frozen."
)

print(
    "✓ No new log_price causal weights were calculated."
)

===== EXACT OLD GCCM DIAGNOSTIC REPRODUCTION =====

[1] Numeric reproduction


,metric,max_abs_difference
0,rho_L10,1.942890e-16
1,rho_Lmax,0.000000e+00
2,gain,8.881784e-16
3,spearman,2.220446e-16
4,p_two_sided,1.144917e-16
5,CI_lower,7.771561e-16
6,CI_upper,3.885781e-16



[2] Converged flags identical: True

[3] Reconstructed OLD diagnostics


,feature,rho_L10,rho_Lmax,gain,spearman,p_two_sided,CI_lower,CI_upper,converged
0,bathrooms,0.093654,0.143220,0.049566,0.942857,0.000000,0.100922,0.185000,True
1,sqft_living,0.175639,0.270903,0.095264,0.942857,0.000000,0.230668,0.310214,True
2,sqft_lot,0.049763,0.171303,0.121540,1.000000,0.000000,0.129337,0.212658,True
3,grade,0.247137,0.310584,0.063447,0.657143,0.000000,0.271282,0.348851,True
4,condition,0.014780,0.025398,0.010618,0.657143,0.246266,-0.017539,0.068241,False
5,waterfront,0.021254,0.164455,0.143201,1.000000,0.000000,0.122401,0.205919,True
6,view,0.098185,0.296150,0.197965,1.000000,0.000000,0.256492,0.334812,True
7,age,0.005681,0.055566,0.049885,1.000000,0.011139,0.012679,0.098250,True



===== FINAL CHECK =====
Maximum numeric difference: 8.881784197001252e-16
Exact diagnostic reproduction passed: True

✓ OLD GCCM diagnostics reproduced exactly.
✓ p_two_sided formula is now frozen.
✓ No new log_price causal weights were calculated.


In [23]:
compute_gccm_diagnostic_exact

<function __main__.compute_gccm_diagnostic_exact(feature, raw_df)>

In [24]:
# ============================================================
# Step 13B — Final author-aligned log_price GCCM diagnostics
#            + causal strengths + causal weights
#
# 使用刚刚已经通过 EXACT reproduction 的
# compute_gccm_diagnostic_exact()
#
# 不改变任何诊断公式。
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "===== FINAL LOG_PRICE GCCM CAUSAL WEIGHTS ====="
)


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

NEW_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
)

OLD_GCCM_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm"
)


feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


# ============================================================
# PART A — NEW log_price convergence diagnostics
# ============================================================

new_rows = []


for feature in feature_order:

    raw_path = (
        NEW_GCCM_DIR
        /
        (
            "seattle_gccm_train_"
            f"logprice_xmap_{feature}_"
            "convergence_raw_v1.csv"
        )
    )

    assert raw_path.exists()


    raw_df = pd.read_csv(
        raw_path
    )


    diagnostic = compute_gccm_diagnostic_exact(
        feature,
        raw_df
    )


    new_rows.append(
        diagnostic
    )


new_diag = pd.DataFrame(
    new_rows
)


# ------------------------------------------------------------
# 2. Causal strength
#
# Only features passing ALL convergence conditions
# receive terminal rho as causal strength.
# ------------------------------------------------------------

new_diag[
    "causal_strength"
] = np.where(

    new_diag[
        "converged"
    ],

    new_diag[
        "rho_Lmax"
    ],

    0.0
)


strength_sum_new = (
    new_diag[
        "causal_strength"
    ].sum()
)


assert strength_sum_new > 0


# ------------------------------------------------------------
# 3. Normalize causal strengths into causal weights
#
# w_k = s_k / sum_j s_j
# ------------------------------------------------------------

new_diag[
    "causal_weight"
] = (

    new_diag[
        "causal_strength"
    ]

    /

    strength_sum_new
)


weight_sum_new = (
    new_diag[
        "causal_weight"
    ].sum()
)


assert np.isclose(
    weight_sum_new,
    1.0
)


print(
    "\n[1] NEW log_price GCCM diagnostics"
)


display(
    new_diag[
        [
            "feature",
            "rho_L10",
            "rho_Lmax",
            "gain",
            "spearman",
            "p_two_sided",
            "CI_lower",
            "CI_upper",
            "converged",
            "causal_strength",
            "causal_weight",
        ]
    ].round(6)
)


print(
    "\nConverged features:",
    int(
        new_diag[
            "converged"
        ].sum()
    ),
    "/",
    len(
        new_diag
    )
)


print(
    "Causal strength sum:",
    strength_sum_new
)

print(
    "Causal weight sum:",
    weight_sum_new
)


# ============================================================
# PART B — Reconstruct OLD price causal weights
# using the same exact diagnostic function
# ============================================================

old_rows = []


for feature in feature_order:

    old_raw_path = (
        OLD_GCCM_DIR
        /
        (
            "seattle_gccm_outer_train_"
            f"price_xmap_{feature}_"
            "convergence_raw.csv"
        )
    )

    assert old_raw_path.exists()


    old_raw = pd.read_csv(
        old_raw_path
    )


    old_rows.append(
        compute_gccm_diagnostic_exact(
            feature,
            old_raw
        )
    )


old_diag_exact = pd.DataFrame(
    old_rows
)


old_diag_exact[
    "causal_strength"
] = np.where(

    old_diag_exact[
        "converged"
    ],

    old_diag_exact[
        "rho_Lmax"
    ],

    0.0
)


strength_sum_old = (
    old_diag_exact[
        "causal_strength"
    ].sum()
)


old_diag_exact[
    "causal_weight"
] = (

    old_diag_exact[
        "causal_strength"
    ]

    /

    strength_sum_old
)


assert np.isclose(
    old_diag_exact[
        "causal_weight"
    ].sum(),
    1.0
)


# ============================================================
# PART C — price vs log_price comparison
# ============================================================

old_compare = (
    old_diag_exact[
        [
            "feature",
            "rho_Lmax",
            "converged",
            "causal_strength",
            "causal_weight",
        ]
    ]
    .rename(
        columns={
            "rho_Lmax":
                "price_rho",

            "converged":
                "price_converged",

            "causal_strength":
                "price_strength",

            "causal_weight":
                "price_weight",
        }
    )
)


new_compare = (
    new_diag[
        [
            "feature",
            "rho_Lmax",
            "converged",
            "causal_strength",
            "causal_weight",
        ]
    ]
    .rename(
        columns={
            "rho_Lmax":
                "logprice_rho",

            "converged":
                "logprice_converged",

            "causal_strength":
                "logprice_strength",

            "causal_weight":
                "logprice_weight",
        }
    )
)


comparison = old_compare.merge(
    new_compare,
    on="feature"
)


comparison[
    "rho_change"
] = (

    comparison[
        "logprice_rho"
    ]

    -

    comparison[
        "price_rho"
    ]
)


comparison[
    "weight_change"
] = (

    comparison[
        "logprice_weight"
    ]

    -

    comparison[
        "price_weight"
    ]
)


comparison[
    "status_changed"
] = (

    comparison[
        "price_converged"
    ]

    !=

    comparison[
        "logprice_converged"
    ]
)


print(
    "\n[2] PRICE vs LOG_PRICE comparison"
)


display(
    comparison.round(
        6
    )
)


# ============================================================
# PART D — Weight stability
# ============================================================

pearson_weight_corr = float(
    np.corrcoef(

        comparison[
            "price_weight"
        ],

        comparison[
            "logprice_weight"
        ]

    )[0, 1]
)


spearman_weight_corr = float(
    comparison[
        [
            "price_weight",
            "logprice_weight"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


print(
    "\n[3] Weight stability"
)

print(
    "Pearson correlation :",
    pearson_weight_corr
)

print(
    "Spearman correlation:",
    spearman_weight_corr
)


print(
    "\nFeatures whose convergence status changed:"
)


status_changed_table = (
    comparison[
        comparison[
            "status_changed"
        ]
    ][
        [
            "feature",
            "price_converged",
            "logprice_converged",
            "price_weight",
            "logprice_weight",
        ]
    ]
)


display(
    status_changed_table.round(
        6
    )
)


# ============================================================
# PART E — Rank causal weights
# ============================================================

weight_ranking = (
    new_diag[
        [
            "feature",
            "causal_strength",
            "causal_weight"
        ]
    ]
    .sort_values(
        "causal_weight",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


weight_ranking.insert(
    0,
    "rank",
    np.arange(
        1,
        len(
            weight_ranking
        ) + 1
    )
)


print(
    "\n[4] NEW log_price causal-weight ranking"
)

display(
    weight_ranking.round(
        6
    )
)


# ============================================================
# PART F — Save frozen author-aligned weights
# ============================================================

NEW_DIAG_PATH = (
    NEW_GCCM_DIR
    /
    "seattle_gccm_train_logprice_convergence_diagnostics_v1.csv"
)


NEW_WEIGHT_PATH = (
    NEW_GCCM_DIR
    /
    "seattle_gccm_train_logprice_causal_weights_v1.csv"
)


COMPARISON_PATH = (
    NEW_GCCM_DIR
    /
    "seattle_gccm_price_vs_logprice_comparison_v1.csv"
)


new_diag.to_csv(
    NEW_DIAG_PATH,
    index=False
)


new_diag[
    [
        "feature",
        "causal_strength",
        "causal_weight"
    ]
].to_csv(
    NEW_WEIGHT_PATH,
    index=False
)


comparison.to_csv(
    COMPARISON_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    NEW_DIAG_PATH
)

print(
    NEW_WEIGHT_PATH
)

print(
    COMPARISON_PATH
)


print(
    "\n✓ Author-aligned log_price GCCM diagnostics completed."
)

print(
    "✓ Causal weights were normalized to sum to 1."
)

print(
    "✓ The diagnostic formula was not changed."
)

print(
    "✓ Calibration / Validation / Test were not used."
)

===== FINAL LOG_PRICE GCCM CAUSAL WEIGHTS =====

[1] NEW log_price GCCM diagnostics


,feature,rho_L10,rho_Lmax,gain,spearman,p_two_sided,CI_lower,CI_upper,converged,causal_strength,causal_weight
0,bathrooms,0.096822,0.164740,0.067917,1.000000,0.000000,0.122689,0.206200,True,0.164740,0.128323
1,sqft_living,0.167613,0.241593,0.073979,0.657143,0.000000,0.200756,0.281591,True,0.241593,0.188186
2,sqft_lot,0.060458,0.181311,0.120852,1.000000,0.000000,0.139478,0.222497,True,0.181311,0.141230
3,grade,0.245583,0.339297,0.093714,1.000000,0.000000,0.300758,0.376728,True,0.339297,0.264292
4,condition,0.016546,0.049273,0.032728,1.000000,0.024420,0.006369,0.091996,True,0.049273,0.038381
5,waterfront,0.010610,0.095433,0.084823,1.000000,0.000013,0.052731,0.137786,True,0.095433,0.074337
6,view,0.102757,0.212148,0.109392,0.942857,0.000000,0.170786,0.252765,True,0.212148,0.165251
7,age,0.006343,0.029956,0.023613,0.828571,0.171417,-0.012978,0.072780,False,0.000000,0.000000



Converged features: 7 / 8
Causal strength sum: 1.2837939337435706
Causal weight sum: 1.0

[2] PRICE vs LOG_PRICE comparison


,feature,price_rho,price_converged,price_strength,price_weight,logprice_rho,logprice_converged,logprice_strength,logprice_weight,rho_change,weight_change,status_changed
0,bathrooms,0.143220,True,0.143220,0.101417,0.164740,True,0.164740,0.128323,0.021520,0.026905,False
1,sqft_living,0.270903,True,0.270903,0.191833,0.241593,True,0.241593,0.188186,-0.029311,-0.003647,False
2,sqft_lot,0.171303,True,0.171303,0.121304,0.181311,True,0.181311,0.141230,0.010007,0.019926,False
3,grade,0.310584,True,0.310584,0.219932,0.339297,True,0.339297,0.264292,0.028713,0.044360,False
4,condition,0.025398,False,0.000000,0.000000,0.049273,True,0.049273,0.038381,0.023876,0.038381,True
5,waterfront,0.164455,True,0.164455,0.116455,0.095433,True,0.095433,0.074337,-0.069022,-0.042118,False
6,view,0.296150,True,0.296150,0.209711,0.212148,True,0.212148,0.165251,-0.084002,-0.044460,False
7,age,0.055566,True,0.055566,0.039348,0.029956,False,0.000000,0.000000,-0.025610,-0.039348,True



[3] Weight stability
Pearson correlation : 0.8985708799491449
Spearman correlation: 0.9285714285714286

Features whose convergence status changed:


,feature,price_converged,logprice_converged,price_weight,logprice_weight
4,condition,False,True,0.000000,0.038381
7,age,True,False,0.039348,0.000000



[4] NEW log_price causal-weight ranking


,rank,feature,causal_strength,causal_weight
0,1,grade,0.339297,0.264292
1,2,sqft_living,0.241593,0.188186
2,3,view,0.212148,0.165251
3,4,sqft_lot,0.181311,0.141230
4,5,bathrooms,0.164740,0.128323
5,6,waterfront,0.095433,0.074337
6,7,condition,0.049273,0.038381
7,8,age,0.000000,0.000000



===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\seattle_gccm_train_logprice_convergence_diagnostics_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\seattle_gccm_train_logprice_causal_weights_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm_author_aligned\seattle_gccm_price_vs_logprice_comparison_v1.csv

✓ Author-aligned log_price GCCM diagnostics completed.
✓ Causal weights were normalized to sum to 1.
✓ The diagnostic formula was not changed.
✓ Calibration / Validation / Test were not used.


In [25]:
# ============================================================
# Step 14 — GCCP causal-distance unit tests & scale diagnostic
#
# 目的：
# 1. 读取正式 log_price GCCM causal weights
# 2. 构造 EUC-anchored causal distance
# 3. 验证 equal weights -> exactly EUC
# 4. 检查真实 causal distance 与 EUC 有多不同
# 5. 检查 causal weighting 是否真正改变局部邻居
# 6. 检查 d_geo 与 d_causal 的尺度
#
# 只使用 Calibration + Validation。
# Test 不参与。
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances


print(
    "===== GCCP CAUSAL DISTANCE DIAGNOSTIC ====="
)


# ------------------------------------------------------------
# 1. Load frozen author-aligned causal weights
# ------------------------------------------------------------

CAUSAL_WEIGHT_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
    / "seattle_gccm_train_logprice_causal_weights_v1.csv"
)


causal_weight_df = pd.read_csv(
    CAUSAL_WEIGHT_PATH
)


feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


causal_weight_df = (
    causal_weight_df
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)


causal_weights = (
    causal_weight_df[
        "causal_weight"
    ]
    .to_numpy(
        dtype=float
    )
)


p_features = len(
    causal_weights
)


print(
    "\n[1] Frozen causal weights"
)

display(
    causal_weight_df
)


print(
    "Weight sum:",
    causal_weights.sum()
)

print(
    "Number of features:",
    p_features
)


assert p_features == 8

assert np.isclose(
    causal_weights.sum(),
    1.0
)


# ------------------------------------------------------------
# 2. Reproduce official GeoSIMCP EUC standardization
#
# Standardization is fitted on Calibration ONLY.
# ------------------------------------------------------------

calib_std_raw = np.std(
    X_calib,
    axis=0
)


valid_feature_cols = (
    calib_std_raw > 1e-8
)


print(
    "\n[2] Feature standardization"
)

print(
    "Features retained:",
    valid_feature_cols.sum(),
    "/",
    len(valid_feature_cols)
)


assert valid_feature_cols.sum() == 8


X_calib_valid = (
    X_calib[
        :,
        valid_feature_cols
    ]
)


X_val_valid = (
    X_val[
        :,
        valid_feature_cols
    ]
)


feature_mean = np.mean(
    X_calib_valid,
    axis=0
)


feature_std = np.std(
    X_calib_valid,
    axis=0
)


feature_std[
    feature_std == 0
] = 1.0


Z_calib = (
    X_calib_valid
    -
    feature_mean
) / feature_std


Z_val = (
    X_val_valid
    -
    feature_mean
) / feature_std


# ------------------------------------------------------------
# 3. Ordinary GeoSIMCP-EUC distance
# ------------------------------------------------------------

d_euc = pairwise_distances(
    Z_val,
    Z_calib,
    metric="euclidean"
)


# ------------------------------------------------------------
# 4. GCCP causal distance
#
# causal weights sum to 1.
#
# Distance coefficients:
#
# gamma_k = p * w_k
#
# Therefore equal causal weights (1/p)
# give gamma_k = 1 exactly,
# reproducing ordinary Euclidean distance.
# ------------------------------------------------------------

distance_coefficients = (
    p_features
    *
    causal_weights
)


print(
    "\n[3] Distance coefficients gamma_k = p * causal_weight"
)


coefficient_table = pd.DataFrame(
    {
        "feature":
            feature_order,

        "causal_weight":
            causal_weights,

        "distance_coefficient":
            distance_coefficients,
    }
)


display(
    coefficient_table.round(6)
)


print(
    "Mean distance coefficient:",
    distance_coefficients.mean()
)

print(
    "Coefficient sum:",
    distance_coefficients.sum()
)


# Weighted squared differences
diff = (
    Z_val[:, None, :]
    -
    Z_calib[None, :, :]
)


d_causal = np.sqrt(

    np.sum(

        distance_coefficients[
            None,
            None,
            :
        ]

        *

        diff ** 2,

        axis=2
    )
)


# ============================================================
# UNIT TEST 1
# Equal causal weights must reproduce EUC exactly
# ============================================================

equal_causal_weights = np.ones(
    p_features
) / p_features


equal_coefficients = (
    p_features
    *
    equal_causal_weights
)


d_causal_equal = np.sqrt(

    np.sum(

        equal_coefficients[
            None,
            None,
            :
        ]

        *

        diff ** 2,

        axis=2
    )
)


equal_weight_max_diff = np.max(
    np.abs(
        d_causal_equal
        -
        d_euc
    )
)


print(
    "\n===== UNIT TEST 1 ====="
)

print(
    "Equal causal weights -> EUC"
)

print(
    "Maximum absolute difference:",
    equal_weight_max_diff
)


assert equal_weight_max_diff < 1e-10


print(
    "✓ PASSED"
)


# ============================================================
# 5. Distance distribution comparison
# ============================================================

def summarize_distance(
    name,
    matrix
):

    x = matrix.ravel()

    return {
        "distance":
            name,

        "min":
            np.min(x),

        "p25":
            np.quantile(
                x,
                0.25
            ),

        "median":
            np.median(x),

        "mean":
            np.mean(x),

        "p75":
            np.quantile(
                x,
                0.75
            ),

        "p99":
            np.quantile(
                x,
                0.99
            ),

        "max":
            np.max(x),
    }


distance_summary = pd.DataFrame(
    [
        summarize_distance(
            "EUC",
            d_euc
        ),

        summarize_distance(
            "Causal",
            d_causal
        ),
    ]
)


print(
    "\n[4] EUC vs causal distance distribution"
)

display(
    distance_summary.round(6)
)


# ------------------------------------------------------------
# 6. EUC vs causal distance correlation
# ------------------------------------------------------------

distance_pearson = float(
    np.corrcoef(
        d_euc.ravel(),
        d_causal.ravel()
    )[0, 1]
)


distance_spearman = float(
    pd.Series(
        d_euc.ravel()
    )
    .corr(
        pd.Series(
            d_causal.ravel()
        ),
        method="spearman"
    )
)


print(
    "\n[5] EUC vs causal-distance similarity"
)

print(
    "Pearson correlation :",
    distance_pearson
)

print(
    "Spearman correlation:",
    distance_spearman
)


# ============================================================
# 7. Neighbor-overlap diagnostic
#
# If causal weighting really changes local similarity,
# nearest Calibration samples should change.
# ============================================================

neighbor_rows = []


for k in [
    10,
    30,
    50
]:

    overlaps = []


    for i in range(
        len(X_val)
    ):

        euc_neighbors = set(
            np.argsort(
                d_euc[i]
            )[:k]
        )


        causal_neighbors = set(
            np.argsort(
                d_causal[i]
            )[:k]
        )


        overlaps.append(

            len(
                euc_neighbors
                &
                causal_neighbors
            )

            /
            k
        )


    neighbor_rows.append(
        {
            "k":
                k,

            "mean_overlap":
                np.mean(
                    overlaps
                ),

            "median_overlap":
                np.median(
                    overlaps
                ),

            "min_overlap":
                np.min(
                    overlaps
                ),

            "max_overlap":
                np.max(
                    overlaps
                ),
        }
    )


neighbor_overlap = pd.DataFrame(
    neighbor_rows
)


print(
    "\n[6] Nearest-neighbor overlap: EUC vs causal"
)

display(
    neighbor_overlap.round(6)
)


# ============================================================
# 8. Geographic vs causal scale
# ============================================================

d_geo = pairwise_distances(
    coords_val,
    coords_calib,
    metric="euclidean"
)


geo_causal_summary = pd.DataFrame(
    [
        summarize_distance(
            "Geographic (meters)",
            d_geo
        ),

        summarize_distance(
            "EUC",
            d_euc
        ),

        summarize_distance(
            "Causal",
            d_causal
        ),
    ]
)


print(
    "\n[7] Geographic vs EUC vs causal scales"
)

display(
    geo_causal_summary.round(6)
)


# ------------------------------------------------------------
# 9. Joint-distance contribution diagnostic
#
# Same original GeoSIMCP formula:
#
# lambda*d_geo^2 + (1-lambda)*d_causal^2
# ------------------------------------------------------------

contribution_rows = []


eps = 1e-12


for lam in [
    0.1,
    0.3,
    0.5,
    0.7,
    0.9
]:

    geo_component = (
        lam
        *
        d_geo ** 2
    )


    causal_component = (
        (1.0 - lam)
        *
        d_causal ** 2
    )


    ratio = (
        geo_component
        /
        (
            causal_component
            +
            eps
        )
    )


    contribution_rows.append(
        {
            "lambda":
                lam,

            "median_geo_component":
                np.median(
                    geo_component
                ),

            "median_causal_component":
                np.median(
                    causal_component
                ),

            "median_geo/causal":
                np.median(
                    ratio
                ),

            "fraction_geo_larger":
                np.mean(
                    geo_component
                    >
                    causal_component
                ),
        }
    )


contribution_table = pd.DataFrame(
    contribution_rows
)


print(
    "\n[8] Actual contribution inside original GeoSIMCP joint distance"
)

display(
    contribution_table
)


print(
    "\n✓ Causal-distance diagnostics completed."
)

print(
    "✓ Equal weights reproduce EUC exactly."
)

print(
    "✓ Calibration + Validation only."
)

print(
    "✓ Test remains untouched."
)

===== GCCP CAUSAL DISTANCE DIAGNOSTIC =====

[1] Frozen causal weights


,feature,causal_strength,causal_weight
0,bathrooms,0.164740,0.128323
1,sqft_living,0.241593,0.188186
2,sqft_lot,0.181311,0.141230
3,grade,0.339297,0.264292
4,condition,0.049273,0.038381
5,waterfront,0.095433,0.074337
6,view,0.212148,0.165251
7,age,0.000000,0.000000


Weight sum: 0.9999999999999998
Number of features: 8

[2] Feature standardization
Features retained: 7 / 8


AssertionError: 

In [26]:
# ============================================================
# Step 14A — Diagnose zero-variance feature in Calibration
#
# 目的：
# 确认官方 GeoSIMCP-EUC 为什么只保留 7/8 个属性。
# 不计算 GCCP，不使用 Test 做参数选择。
# ============================================================

import numpy as np
import pandas as pd


print(
    "===== FEATURE VARIANCE DIAGNOSTIC ====="
)


feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


rows = []


for j, feature in enumerate(
    feature_order
):

    rows.append(
        {
            "feature":
                feature,

            "train_min":
                np.min(X_train[:, j]),

            "train_max":
                np.max(X_train[:, j]),

            "train_std":
                np.std(X_train[:, j]),

            "calib_min":
                np.min(X_calib[:, j]),

            "calib_max":
                np.max(X_calib[:, j]),

            "calib_std":
                np.std(X_calib[:, j]),

            "calib_unique":
                len(
                    np.unique(
                        X_calib[:, j]
                    )
                ),

            "val_min":
                np.min(X_val[:, j]),

            "val_max":
                np.max(X_val[:, j]),

            "val_unique":
                len(
                    np.unique(
                        X_val[:, j]
                    )
                ),
        }
    )


variance_table = pd.DataFrame(
    rows
)


variance_table[
    "retained_by_official_EUC"
] = (

    variance_table[
        "calib_std"
    ]

    >
    1e-8
)


display(
    variance_table
)


dropped_features = (
    variance_table.loc[
        ~variance_table[
            "retained_by_official_EUC"
        ],
        "feature"
    ]
    .tolist()
)


print(
    "\nFeatures retained:",
    int(
        variance_table[
            "retained_by_official_EUC"
        ].sum()
    ),
    "/",
    len(
        feature_order
    )
)


print(
    "Features dropped:",
    dropped_features
)


# ------------------------------------------------------------
# Extra check for binary / rare features
# ------------------------------------------------------------

print(
    "\n===== RARE-FEATURE COUNTS ====="
)


for feature in [
    "waterfront",
]:

    j = feature_order.index(
        feature
    )

    print(
        f"\n{feature}:"
    )

    for split_name, X_split in [
        ("Train", X_train),
        ("Calibration", X_calib),
        ("Validation", X_val),
        ("Test", X_test),
    ]:

        values, counts = np.unique(
            X_split[:, j],
            return_counts=True
        )

        print(
            split_name,
            dict(
                zip(
                    values,
                    counts
                )
            )
        )


print(
    "\n✓ Diagnostic complete."
)

===== FEATURE VARIANCE DIAGNOSTIC =====


,feature,train_min,train_max,train_std,calib_min,calib_max,calib_std,calib_unique,val_min,val_max,val_unique,retained_by_official_EUC
0,bathrooms,0.0,5.5,0.764270,0.75,6.0,0.780375,15,0.75,4.5,14,True
1,sqft_living,410.0,7400.0,859.758897,730.00,5490.0,802.276267,179,520.00,5220.0,180,True
2,sqft_lot,651.0,20000.0,3713.356713,834.00,19897.0,3564.417292,254,705.00,17861.0,243,True
3,grade,4.0,13.0,1.118800,4.00,12.0,1.128475,9,5.00,11.0,7,True
4,condition,1.0,5.0,0.683955,2.00,5.0,0.644041,4,1.00,5.0,5,True
5,waterfront,0.0,1.0,0.072185,0.00,0.0,0.000000,1,0.00,1.0,2,False
6,view,0.0,4.0,0.768642,0.00,4.0,0.694230,5,0.00,4.0,5,True
7,age,0.0,115.0,29.490585,0.00,115.0,30.970349,99,0.00,115.0,96,True



Features retained: 7 / 8
Features dropped: ['waterfront']

===== RARE-FEATURE COUNTS =====

waterfront:
Train {np.float64(0.0): np.int64(2089), np.float64(1.0): np.int64(11)}
Calibration {np.float64(0.0): np.int64(300)}
Validation {np.float64(0.0): np.int64(299), np.float64(1.0): np.int64(1)}
Test {np.float64(0.0): np.int64(296), np.float64(1.0): np.int64(4)}

✓ Diagnostic complete.


In [27]:
# ============================================================
# Step 14B — Corrected GCCP causal-distance diagnostic
#
# 与官方 GeoSIMCP-EUC 严格对齐：
#
# 1. Calibration 中零方差属性先删除
# 2. 因果权重限制到同一有效特征空间
# 3. 对剩余 causal weights 重新归一化
# 4. gamma_k = p_eff * w_k
# 5. equal weights 必须 EXACTLY reproduce EUC
#
# Calibration + Validation only.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances


print(
    "===== CORRECTED GCCP CAUSAL DISTANCE DIAGNOSTIC ====="
)


# ------------------------------------------------------------
# 1. Feature order
# ------------------------------------------------------------

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


# ------------------------------------------------------------
# 2. Load frozen Train-GCCM causal weights
# ------------------------------------------------------------

CAUSAL_WEIGHT_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
    / "seattle_gccm_train_logprice_causal_weights_v1.csv"
)


causal_weight_df = pd.read_csv(
    CAUSAL_WEIGHT_PATH
)


causal_weight_df = (
    causal_weight_df
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)


all_causal_weights = (
    causal_weight_df[
        "causal_weight"
    ]
    .to_numpy(
        dtype=float
    )
)


assert np.isclose(
    all_causal_weights.sum(),
    1.0
)


# ------------------------------------------------------------
# 3. Official GeoSIMCP Calibration-based feature filtering
# ------------------------------------------------------------

calib_std_raw = np.std(
    X_calib,
    axis=0
)


valid_feature_mask = (
    calib_std_raw > 1e-8
)


active_indices = np.where(
    valid_feature_mask
)[0]


active_features = [
    feature_order[i]
    for i in active_indices
]


dropped_features = [
    feature_order[i]
    for i in range(
        len(feature_order)
    )
    if not valid_feature_mask[i]
]


p_eff = len(
    active_features
)


print(
    "\n[1] Effective GeoSIMCP feature space"
)

print(
    "Active features:",
    active_features
)

print(
    "Dropped features:",
    dropped_features
)

print(
    "Effective dimension:",
    p_eff
)


assert p_eff == 7

assert dropped_features == [
    "waterfront"
]


# ------------------------------------------------------------
# 4. Restrict causal weights to SAME active feature space
# ------------------------------------------------------------

active_weights_raw = (
    all_causal_weights[
        active_indices
    ]
)


active_weight_mass = (
    active_weights_raw.sum()
)


# Conditional renormalization
active_causal_weights = (
    active_weights_raw
    /
    active_weight_mass
)


assert np.isclose(
    active_causal_weights.sum(),
    1.0
)


# Distance coefficients
#
# gamma_k = p_eff * normalized causal weight
#
# Equal weights -> gamma=1
distance_coefficients = (
    p_eff
    *
    active_causal_weights
)


active_weight_table = pd.DataFrame(
    {
        "feature":
            active_features,

        "original_GCCM_weight":
            active_weights_raw,

        "renormalized_weight":
            active_causal_weights,

        "distance_coefficient":
            distance_coefficients,
    }
)


print(
    "\n[2] Effective causal weights"
)

display(
    active_weight_table.round(
        6
    )
)


print(
    "Original retained weight mass:",
    active_weight_mass
)

print(
    "Renormalized weight sum:",
    active_causal_weights.sum()
)

print(
    "Mean distance coefficient:",
    distance_coefficients.mean()
)


assert np.isclose(
    distance_coefficients.mean(),
    1.0
)


# ------------------------------------------------------------
# 5. Standardize EXACTLY like official GeoSIMCP
#
# Fit mean/std on Calibration only.
# ------------------------------------------------------------

X_calib_active = (
    X_calib[
        :,
        active_indices
    ]
)


X_val_active = (
    X_val[
        :,
        active_indices
    ]
)


feature_mean = np.mean(
    X_calib_active,
    axis=0
)


feature_std = np.std(
    X_calib_active,
    axis=0
)


assert np.all(
    feature_std > 1e-8
)


Z_calib = (
    X_calib_active
    -
    feature_mean
) / feature_std


Z_val = (
    X_val_active
    -
    feature_mean
) / feature_std


# ------------------------------------------------------------
# 6. Official 7-dimensional EUC distance
# ------------------------------------------------------------

d_euc = pairwise_distances(
    Z_val,
    Z_calib,
    metric="euclidean"
)


# ------------------------------------------------------------
# 7. Causal-weighted distance
# ------------------------------------------------------------

diff = (
    Z_val[
        :,
        None,
        :
    ]
    -
    Z_calib[
        None,
        :,
        :
    ]
)


d_causal = np.sqrt(

    np.sum(

        distance_coefficients[
            None,
            None,
            :
        ]

        *

        diff ** 2,

        axis=2
    )
)


# ============================================================
# UNIT TEST 1
#
# Equal weights over SAME 7-dimensional space
# must reproduce official EUC exactly.
# ============================================================

equal_active_weights = (
    np.ones(
        p_eff
    )
    /
    p_eff
)


equal_coefficients = (
    p_eff
    *
    equal_active_weights
)


d_equal = np.sqrt(

    np.sum(

        equal_coefficients[
            None,
            None,
            :
        ]

        *

        diff ** 2,

        axis=2
    )
)


max_equal_euc_diff = np.max(
    np.abs(
        d_equal
        -
        d_euc
    )
)


print(
    "\n===== UNIT TEST 1 ====="
)

print(
    "Equal causal weights -> official EUC"
)

print(
    "Maximum absolute difference:",
    max_equal_euc_diff
)


assert max_equal_euc_diff < 1e-10


print(
    "✓ PASSED"
)


# ============================================================
# 8. Distance distributions
# ============================================================

def summarize_distance(
    name,
    matrix
):

    x = matrix.ravel()

    return {
        "distance":
            name,

        "min":
            np.min(x),

        "p01":
            np.quantile(
                x,
                0.01
            ),

        "p25":
            np.quantile(
                x,
                0.25
            ),

        "median":
            np.median(x),

        "mean":
            np.mean(x),

        "p75":
            np.quantile(
                x,
                0.75
            ),

        "p99":
            np.quantile(
                x,
                0.99
            ),

        "max":
            np.max(x),
    }


distance_summary = pd.DataFrame(
    [
        summarize_distance(
            "EUC",
            d_euc
        ),

        summarize_distance(
            "Causal",
            d_causal
        ),
    ]
)


print(
    "\n[3] EUC vs causal distance distributions"
)

display(
    distance_summary.round(
        6
    )
)


# ============================================================
# 9. Distance correlation
# ============================================================

distance_pearson = float(
    np.corrcoef(
        d_euc.ravel(),
        d_causal.ravel()
    )[0, 1]
)


distance_spearman = float(
    pd.Series(
        d_euc.ravel()
    )
    .corr(
        pd.Series(
            d_causal.ravel()
        ),
        method="spearman"
    )
)


print(
    "\n[4] EUC vs causal-distance correlation"
)

print(
    "Pearson :",
    distance_pearson
)

print(
    "Spearman:",
    distance_spearman
)


# ============================================================
# 10. Neighbor-overlap diagnostic
# ============================================================

neighbor_rows = []


for k in [
    10,
    30,
    50
]:

    overlaps = []


    for i in range(
        len(X_val)
    ):

        euc_idx = set(
            np.argsort(
                d_euc[i]
            )[:k]
        )


        causal_idx = set(
            np.argsort(
                d_causal[i]
            )[:k]
        )


        overlaps.append(

            len(
                euc_idx
                &
                causal_idx
            )

            /
            k
        )


    neighbor_rows.append(
        {
            "k":
                k,

            "mean_overlap":
                np.mean(
                    overlaps
                ),

            "median_overlap":
                np.median(
                    overlaps
                ),

            "min_overlap":
                np.min(
                    overlaps
                ),

            "max_overlap":
                np.max(
                    overlaps
                ),
        }
    )


neighbor_overlap = pd.DataFrame(
    neighbor_rows
)


print(
    "\n[5] EUC vs causal nearest-neighbor overlap"
)

display(
    neighbor_overlap.round(
        6
    )
)


# ============================================================
# 11. Geographic distance
# ============================================================

d_geo = pairwise_distances(
    coords_val,
    coords_calib,
    metric="euclidean"
)


scale_table = pd.DataFrame(
    [
        summarize_distance(
            "Geographic (meters)",
            d_geo
        ),

        summarize_distance(
            "EUC",
            d_euc
        ),

        summarize_distance(
            "Causal",
            d_causal
        ),
    ]
)


print(
    "\n[6] Geographic vs EUC vs causal scales"
)

display(
    scale_table.round(
        6
    )
)


# ============================================================
# 12. Original GeoSIMCP joint-distance contribution
#
# lambda*d_geo^2
# +
# (1-lambda)*d_causal^2
# ============================================================

eps = 1e-12


contribution_rows = []


for lam in [
    0.1,
    0.3,
    0.5,
    0.7,
    0.9
]:

    geo_component = (
        lam
        *
        d_geo ** 2
    )


    causal_component = (
        (1.0 - lam)
        *
        d_causal ** 2
    )


    ratio = (
        geo_component
        /
        (
            causal_component
            +
            eps
        )
    )


    contribution_rows.append(
        {
            "lambda":
                lam,

            "median_geo_component":
                np.median(
                    geo_component
                ),

            "median_causal_component":
                np.median(
                    causal_component
                ),

            "median_geo/causal":
                np.median(
                    ratio
                ),

            "fraction_geo_larger":
                np.mean(
                    geo_component
                    >
                    causal_component
                ),
        }
    )


contribution_table = pd.DataFrame(
    contribution_rows
)


print(
    "\n[7] Contribution inside original GeoSIMCP fusion"
)

display(
    contribution_table
)


print(
    "\n✓ Corrected causal-distance diagnostic complete."
)

print(
    "✓ Same 7-dimensional feature space as official EUC."
)

print(
    "✓ Equal weights reproduce EUC exactly."
)

print(
    "✓ Calibration + Validation only."
)

===== CORRECTED GCCP CAUSAL DISTANCE DIAGNOSTIC =====

[1] Effective GeoSIMCP feature space
Active features: ['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'view', 'age']
Dropped features: ['waterfront']
Effective dimension: 7

[2] Effective causal weights


,feature,original_GCCM_weight,renormalized_weight,distance_coefficient
0,bathrooms,0.128323,0.138628,0.970393
1,sqft_living,0.188186,0.203299,1.423093
2,sqft_lot,0.141230,0.152572,1.068005
3,grade,0.264292,0.285516,1.998615
4,condition,0.038381,0.041463,0.290241
5,view,0.165251,0.178522,1.249653
6,age,0.000000,0.000000,0.000000


Original retained weight mass: 0.925663382745887
Renormalized weight sum: 0.9999999999999998
Mean distance coefficient: 0.9999999999999998

===== UNIT TEST 1 =====
Equal causal weights -> official EUC
Maximum absolute difference: 1.0498546476611637e-14
✓ PASSED

[3] EUC vs causal distance distributions


,distance,min,p01,p25,median,mean,p75,p99,max
0,EUC,0.029169,0.86505,2.373884,3.260002,3.408772,4.190165,7.710845,11.904204
1,Causal,0.031046,0.70552,2.046423,2.915385,3.272429,4.089084,8.768216,13.802594



[4] EUC vs causal-distance correlation
Pearson : 0.9293836742247134
Spearman: 0.8965982406564439

[5] EUC vs causal nearest-neighbor overlap


...


[6] Geographic vs EUC vs causal scales


,distance,min,p01,p25,median,mean,p75,p99,max
0,Geographic (meters),33.342798,1474.136155,11017.339368,18262.275617,19845.652004,26602.183883,49292.694649,57393.586202
1,EUC,0.029169,0.865050,2.373884,3.260002,3.408772,4.190165,7.710845,11.904204
2,Causal,0.031046,0.705520,2.046423,2.915385,3.272429,4.089084,8.768216,13.802594



[7] Contribution inside original GeoSIMCP fusion


,lambda,median_geo_component,median_causal_component,median_geo/causal,fraction_geo_larger
0,0.1,3.335107e+07,7.649521,3.799133e+06,1.0
1,0.3,1.000532e+08,5.949628,1.465380e+07,1.0
2,0.5,1.667554e+08,4.249734,3.419220e+07,1.0
3,0.7,2.334575e+08,2.549840,7.978180e+07,1.0
4,0.9,3.001596e+08,0.849947,3.077298e+08,1.0



✓ Corrected causal-distance diagnostic complete.
✓ Same 7-dimensional feature space as official EUC.
✓ Equal weights reproduce EUC exactly.
✓ Calibration + Validation only.


In [28]:
# ============================================================
# Step 15 — End-to-end GCCP equivalence unit tests
#
# Test A:
# Equal causal weights -> official GeoSIMCP-EUC weights
#
# Test B:
# lambda = 1 -> official GeoCP prediction intervals
#
# Calibration + Validation only.
# ============================================================

import numpy as np

from geoconformal import (
    GeoConformalSpatialRegression
)

from geoconformal.geocp import (
    weighted_quantile
)

from geoconformal.geocp.weights import (
    joint_geo_feature_weights
)


print(
    "===== GCCP END-TO-END EQUIVALENCE TESTS ====="
)


# ============================================================
# TEST A
# Equal causal weights -> official GeoSIMCP-EUC
# ============================================================

print(
    "\n===== TEST A ====="
)

print(
    "Equal causal weights -> official GeoSIMCP-EUC"
)


# We already constructed:
#
# d_geo
# d_euc
#
# in Step 14B.
#
# Under equal causal weights:
#
# d_causal == d_euc


test_configs = [

    # Strong feature-local weighting
    (1.0, 0.0),

    # Official minimum bandwidth, feature-only
    (500.0, 0.0),

    # Mixed geographic-feature setting
    (30000.0, 0.5),

    # Pure geographic setting
    (28982.758621, 1.0),
]


test_A_rows = []


official_input_val = np.hstack(
    [
        coords_val,
        X_val
    ]
)


for bw, lam in test_configs:

    # --------------------------------------------------------
    # Our manual equal-weight version
    # --------------------------------------------------------

    d_joint_manual = np.sqrt(

        lam
        *
        d_geo ** 2

        +

        (1.0 - lam)
        *
        d_euc ** 2
    )


    weights_manual = np.exp(

        -0.5
        *
        (
            d_joint_manual
            /
            bw
        ) ** 2
    )


    # --------------------------------------------------------
    # Official geoconformal 0.3.0 implementation
    # --------------------------------------------------------

    official_weight_fn = (
        joint_geo_feature_weights(

            coord_calib=coords_calib,

            feat_calib=X_calib,

            bandwidth=bw,

            lambda_weight=lam,

            distance_metric="euclidean",

            standardize=True
        )
    )


    weights_official = (
        official_weight_fn(
            official_input_val,
            None
        )
    )


    max_diff = np.max(
        np.abs(
            weights_manual
            -
            weights_official
        )
    )


    test_A_rows.append(
        {
            "bandwidth":
                bw,

            "lambda":
                lam,

            "max_weight_difference":
                max_diff,
        }
    )


test_A = pd.DataFrame(
    test_A_rows
)


display(
    test_A
)


TEST_A_PASS = bool(
    (
        test_A[
            "max_weight_difference"
        ]
        <
        1e-10
    ).all()
)


print(
    "TEST A passed:",
    TEST_A_PASS
)


assert TEST_A_PASS


# ============================================================
# TEST B
# Actual GCCP causal distance with lambda=1
# must reproduce official GeoCP END TO END.
# ============================================================

print(
    "\n===== TEST B ====="
)

print(
    "lambda = 1 -> official GeoCP"
)


# ------------------------------------------------------------
# 1. Use frozen best GeoCP Validation bandwidth
# ------------------------------------------------------------

GEOGRAPHIC_BW = float(
    best_geocp[
        "bandwidth"
    ]
)


print(
    "GeoCP bandwidth:",
    GEOGRAPHIC_BW
)


# ------------------------------------------------------------
# 2. Calibration residuals
# ------------------------------------------------------------

calib_predictions = model.predict(
    X_calib_spatial
)


calibration_scores = np.abs(

    y_calib
    -
    calib_predictions
)


# ------------------------------------------------------------
# 3. Manual GCCP weights at lambda=1
#
# causal term disappears completely:
#
# d_joint = d_geo
# ------------------------------------------------------------

lambda_geo_only = 1.0


d_joint_geo_only = np.sqrt(

    lambda_geo_only
    *
    d_geo ** 2

    +

    (1.0 - lambda_geo_only)
    *
    d_causal ** 2
)


gccp_geo_only_weights = np.exp(

    -0.5
    *
    (
        d_joint_geo_only
        /
        GEOGRAPHIC_BW
    ) ** 2
)


# ------------------------------------------------------------
# 4. Manual localized weighted conformal quantile
# ------------------------------------------------------------

q_manual = weighted_quantile(

    scores=calibration_scores,

    weights=gccp_geo_only_weights,

    q=0.9,

    self_weights=None,

    interpolate=False
)


val_predictions = model.predict(
    X_val_spatial
)


manual_lower = (
    val_predictions
    -
    q_manual
)


manual_upper = (
    val_predictions
    +
    q_manual
)


# ------------------------------------------------------------
# 5. Official GeoCP
# ------------------------------------------------------------

official_geocp = GeoConformalSpatialRegression(

    predict_f=model.predict,

    miscoverage_level=0.1,

    bandwidth=GEOGRAPHIC_BW,

    coord_calib=coords_calib,

    coord_test=coords_val,

    X_calib=X_calib_spatial,

    y_calib=y_calib,

    X_test=X_val_spatial,

    y_test=y_val,

    include_test_atom=False
)


official_geocp_result = (
    official_geocp.analyze()
)


official_lower = np.asarray(
    official_geocp_result.lower_bound
)


official_upper = np.asarray(
    official_geocp_result.upper_bound
)


# ------------------------------------------------------------
# 6. Compare final prediction intervals
# ------------------------------------------------------------

max_lower_difference = np.max(
    np.abs(
        manual_lower
        -
        official_lower
    )
)


max_upper_difference = np.max(
    np.abs(
        manual_upper
        -
        official_upper
    )
)


manual_coverage = np.mean(

    (y_val >= manual_lower)

    &

    (y_val <= manual_upper)
)


print(
    "\nManual GCCP geo-only Coverage:",
    manual_coverage
)

print(
    "Official GeoCP Coverage:",
    official_geocp_result.coverage_probability
)


print(
    "\nMaximum lower-bound difference:",
    max_lower_difference
)

print(
    "Maximum upper-bound difference:",
    max_upper_difference
)


TEST_B_PASS = (

    max_lower_difference
    <
    1e-10

    and

    max_upper_difference
    <
    1e-10
)


print(
    "\nTEST B passed:",
    TEST_B_PASS
)


assert TEST_B_PASS


# ============================================================
# TEST C
# Clarify an easy conceptual mistake:
#
# Setting causal distance = 0 with lambda < 1
# is NOT the same as GeoCP.
# ============================================================

print(
    "\n===== TEST C ====="
)

print(
    "Zero causal distance with lambda=0.5 "
    "is NOT identical to GeoCP."
)


lam_demo = 0.5


d_joint_zero_causal = np.sqrt(

    lam_demo
    *
    d_geo ** 2
)


weights_zero_causal = np.exp(

    -0.5
    *
    (
        d_joint_zero_causal
        /
        GEOGRAPHIC_BW
    ) ** 2
)


pure_geocp_weights = np.exp(

    -0.5
    *
    (
        d_geo
        /
        GEOGRAPHIC_BW
    ) ** 2
)


zero_causal_vs_geocp_diff = np.max(

    np.abs(

        weights_zero_causal

        -

        pure_geocp_weights
    )
)


print(
    "Maximum weight difference:",
    zero_causal_vs_geocp_diff
)


print(
    "\nInterpretation:"
)

print(
    "Turning OFF the causal component "
    "means lambda=1."
)

print(
    "Simply setting d_causal=0 while lambda<1 "
    "rescales geographic distance and is not GeoCP."
)


print(
    "\n✓ TEST A passed:",
    TEST_A_PASS
)

print(
    "✓ TEST B passed:",
    TEST_B_PASS
)

print(
    "✓ Calibration + Validation only."
)

===== GCCP END-TO-END EQUIVALENCE TESTS =====

===== TEST A =====
Equal causal weights -> official GeoSIMCP-EUC


,bandwidth,lambda,max_weight_difference
0,1.000000,0.0,2.849757e-09
1,500.000000,0.0,4.647394e-13
2,30000.000000,0.5,1.085132e-12
3,28982.758621,1.0,2.325140e-12


TEST A passed: False


AssertionError: 

In [29]:
# ============================================================
# Step 15A-fix — Diagnose numerical distance implementation
#
# 比较：
# 1. sklearn pairwise_distances
# 2. direct NumPy Euclidean formula
#
# 不运行模型、不使用 Test。
# ============================================================

import numpy as np


print(
    "===== DISTANCE NUMERICAL IMPLEMENTATION CHECK ====="
)


# ------------------------------------------------------------
# 1. Direct NumPy EUC
#
# This follows the mathematical implementation used by
# GeoSIMCP's feature-distance calculation:
#
# sqrt(sum((x_test - x_calib)^2))
# ------------------------------------------------------------

d_euc_direct = np.sqrt(

    np.sum(

        (
            Z_val[:, None, :]
            -
            Z_calib[None, :, :]
        ) ** 2,

        axis=2
    )
)


# ------------------------------------------------------------
# 2. Compare with sklearn pairwise_distances result
# from Step 14B
# ------------------------------------------------------------

distance_diff = np.abs(
    d_euc_direct
    -
    d_euc
)


print(
    "\n[1] sklearn vs direct NumPy EUC"
)

print(
    "Maximum absolute distance difference:",
    distance_diff.max()
)

print(
    "Mean absolute distance difference:",
    distance_diff.mean()
)


# ------------------------------------------------------------
# 3. Repeat the most sensitive case:
# bandwidth = 1, lambda = 0
# ------------------------------------------------------------

bw = 1.0
lam = 0.0


weights_direct = np.exp(

    -0.5
    *
    (
        d_euc_direct
        /
        bw
    ) ** 2
)


official_weight_fn = (
    joint_geo_feature_weights(

        coord_calib=coords_calib,

        feat_calib=X_calib,

        bandwidth=bw,

        lambda_weight=lam,

        distance_metric="euclidean",

        standardize=True
    )
)


official_input_val = np.hstack(
    [
        coords_val,
        X_val
    ]
)


weights_official_direct_check = (
    official_weight_fn(
        official_input_val,
        None
    )
)


weight_diff_direct = np.abs(

    weights_direct
    -
    weights_official_direct_check
)


print(
    "\n[2] Direct NumPy vs official weights"
)

print(
    "Maximum absolute weight difference:",
    weight_diff_direct.max()
)

print(
    "Mean absolute weight difference:",
    weight_diff_direct.mean()
)


print(
    "\n✓ Diagnostic finished."
)

===== DISTANCE NUMERICAL IMPLEMENTATION CHECK =====

[1] sklearn vs direct NumPy EUC
Maximum absolute distance difference: 1.0498546476611637e-14
Mean absolute distance difference: 1.8260963728798805e-16

[2] Direct NumPy vs official weights
Maximum absolute weight difference: 2.849756708389606e-09
Mean absolute weight difference: 8.620335976051865e-10

✓ Diagnostic finished.


In [30]:
# ============================================================
# Step 15B — Locate the remaining ~1e-9 weight difference
#
# 目标：
# 1. 再次查看官方 joint_geo_feature_weights 源码中
#    是否存在 1e-8 / epsilon 数值稳定性处理
# 2. 手工加入 1e-8 后，看能否 EXACTLY reproduce
#    official GeoSIMCP weights
#
# Calibration + Validation only.
# ============================================================

import inspect
import numpy as np


print(
    "===== OFFICIAL EUC NUMERICAL-STABILITY CHECK ====="
)


# ------------------------------------------------------------
# 1. Inspect official source
# ------------------------------------------------------------

source_text = inspect.getsource(
    joint_geo_feature_weights
)


print(
    "\n[1] Lines related to Euclidean distance / epsilon"
)


source_lines = source_text.splitlines()


keywords = [
    "d_feat",
    "euclidean",
    "1e-8",
    "1e-08",
    "epsilon",
    "eps",
]


for line_no, line in enumerate(
    source_lines,
    start=1
):

    if any(
        keyword.lower() in line.lower()
        for keyword in keywords
    ):

        print(
            f"{line_no:4d}: {line}"
        )


# ------------------------------------------------------------
# 2. Sensitive configuration
#
# bandwidth = 1
# lambda = 0
#
# This isolates feature distance.
# ------------------------------------------------------------

bw = 1.0
lam = 0.0


official_weight_fn = (
    joint_geo_feature_weights(

        coord_calib=coords_calib,

        feat_calib=X_calib,

        bandwidth=bw,

        lambda_weight=lam,

        distance_metric="euclidean",

        standardize=True
    )
)


official_input_val = np.hstack(
    [
        coords_val,
        X_val
    ]
)


weights_official = (
    official_weight_fn(
        official_input_val,
        None
    )
)


# ------------------------------------------------------------
# 3. Manual version WITHOUT epsilon
# ------------------------------------------------------------

weights_no_eps = np.exp(

    -0.5
    *
    (
        d_euc_direct
        /
        bw
    ) ** 2
)


diff_no_eps = np.abs(
    weights_no_eps
    -
    weights_official
)


# ------------------------------------------------------------
# 4. Manual version WITH +1e-8
# ------------------------------------------------------------

EPSILON = 1e-8


d_euc_plus_eps = (
    d_euc_direct
    +
    EPSILON
)


weights_plus_eps = np.exp(

    -0.5
    *
    (
        d_euc_plus_eps
        /
        bw
    ) ** 2
)


diff_plus_eps = np.abs(
    weights_plus_eps
    -
    weights_official
)


print(
    "\n[2] WITHOUT +1e-8"
)

print(
    "Maximum weight difference:",
    diff_no_eps.max()
)

print(
    "Mean weight difference:",
    diff_no_eps.mean()
)


print(
    "\n[3] WITH +1e-8"
)

print(
    "Maximum weight difference:",
    diff_plus_eps.max()
)

print(
    "Mean weight difference:",
    diff_plus_eps.mean()
)


# ------------------------------------------------------------
# 5. Search several tiny epsilon values
#
# Diagnostic only — NOT tuning a model.
# This tells us which numerical offset matches the library.
# ------------------------------------------------------------

epsilon_candidates = [
    0.0,
    1e-12,
    1e-10,
    1e-9,
    1e-8,
    1e-7,
]


epsilon_rows = []


for eps in epsilon_candidates:

    manual_weights = np.exp(

        -0.5
        *
        (
            (
                d_euc_direct
                +
                eps
            )
            /
            bw
        ) ** 2
    )


    difference = np.abs(
        manual_weights
        -
        weights_official
    )


    epsilon_rows.append(
        {
            "epsilon":
                eps,

            "max_abs_weight_diff":
                difference.max(),

            "mean_abs_weight_diff":
                difference.mean(),
        }
    )


epsilon_table = pd.DataFrame(
    epsilon_rows
)


print(
    "\n[4] Epsilon diagnostic"
)

display(
    epsilon_table
)


best_eps_row = (
    epsilon_table
    .sort_values(
        "max_abs_weight_diff"
    )
    .iloc[0]
)


print(
    "\nBest matching epsilon:",
    best_eps_row[
        "epsilon"
    ]
)

print(
    "Best maximum difference:",
    best_eps_row[
        "max_abs_weight_diff"
    ]
)


print(
    "\n✓ Numerical-stability diagnostic completed."
)

===== OFFICIAL EUC NUMERICAL-STABILITY CHECK =====

[1] Lines related to Euclidean distance / epsilon
   6:     distance_metric: str = 'euclidean',
  15:         d_joint = sqrt( lambda * d_geo^2 + (1 - lambda) * d_feat^2 )
  36:     :param distance_metric: 'euclidean' or 'mnd' (minimum normalized difference)
  37:     :param standardize: z-score the features (euclidean only), fit on calibration
  50:         feat_ranges[feat_ranges == 0] = 1e-8
  55:             valid_cols = np.std(feat_calib, axis=0) > 1e-8
  85:         d_feat = _feat_distance(feat_test) + 1e-8
  86:         d_joint = np.sqrt(lam * d_geo ** 2 + (1.0 - lam) * d_feat ** 2)
  87:         return np.exp(-0.5 * (d_joint / (bw + 1e-8)) ** 2)               # unnormalized

[2] WITHOUT +1e-8
Maximum weight difference: 2.849756708389606e-09
Mean weight difference: 8.620335976051865e-10

[3] WITH +1e-8
Maximum weight difference: 7.357587983314318e-09
Mean weight difference: 1.8451114754534967e-09

[4] Epsilon diagnostic


,epsilon,max_abs_weight_diff,mean_abs_weight_diff
0,0.000000e+00,2.849757e-09,8.620336e-10
1,1.000000e-12,2.850112e-09,8.621177e-10
2,1.000000e-10,2.885413e-09,8.704877e-10
3,1.000000e-09,3.216947e-09,9.496633e-10
4,1.000000e-08,7.357588e-09,1.845111e-09
5,1.000000e-07,6.080663e-08,1.103016e-08



Best matching epsilon: 0.0
Best maximum difference: 2.849756708389606e-09

✓ Numerical-stability diagnostic completed.


In [31]:
# ============================================================
# Step 15C — EXACT reproduction of official GeoSIMCP weights
#
# 完全按照刚刚亲眼看到的官方源码：
#
# d_feat  = raw_feature_distance + 1e-8
# d_joint = sqrt(
#     lambda*d_geo^2
#     + (1-lambda)*d_feat^2
# )
#
# weight = exp(
#     -0.5 * (d_joint / (bandwidth + 1e-8))^2
# )
#
# Calibration + Validation only.
# ============================================================

import numpy as np
import pandas as pd


print(
    "===== EXACT OFFICIAL GEOSIMCP NUMERICAL FORMULA ====="
)


EPS = 1e-8


# ------------------------------------------------------------
# 1. Raw EUC feature distance
#
# d_euc_direct was already calculated on the same
# 7-dimensional feature space actually retained by GeoSIMCP.
# ------------------------------------------------------------

d_feat_official_style = (
    d_euc_direct
    +
    EPS
)


# ------------------------------------------------------------
# 2. Test several configurations
# ------------------------------------------------------------

configs = [
    (1.0, 0.0),
    (500.0, 0.0),
    (30000.0, 0.5),
    (28982.758621, 1.0),
]


comparison_rows = []


official_input_val = np.hstack(
    [
        coords_val,
        X_val
    ]
)


for bw, lam in configs:

    # ========================================================
    # Manual reproduction of EXACT official source
    # ========================================================

    d_joint_manual = np.sqrt(

        lam
        *
        d_geo ** 2

        +

        (1.0 - lam)
        *
        d_feat_official_style ** 2
    )


    weights_manual_exact = np.exp(

        -0.5

        *

        (
            d_joint_manual
            /
            (
                bw
                +
                EPS
            )
        ) ** 2
    )


    # ========================================================
    # Official library
    # ========================================================

    official_weight_fn = joint_geo_feature_weights(

        coord_calib=coords_calib,

        feat_calib=X_calib,

        bandwidth=bw,

        lambda_weight=lam,

        distance_metric="euclidean",

        standardize=True
    )


    weights_library = official_weight_fn(

        official_input_val,

        None
    )


    diff = np.abs(

        weights_manual_exact

        -

        weights_library
    )


    comparison_rows.append(
        {
            "bandwidth":
                bw,

            "lambda":
                lam,

            "max_abs_difference":
                diff.max(),

            "mean_abs_difference":
                diff.mean(),
        }
    )


comparison_exact = pd.DataFrame(
    comparison_rows
)


print(
    "\n[1] Exact-source-formula comparison"
)

display(
    comparison_exact
)


# ------------------------------------------------------------
# 3. Decision
# ------------------------------------------------------------

max_difference = (
    comparison_exact[
        "max_abs_difference"
    ].max()
)


print(
    "\nMaximum difference across all configurations:",
    max_difference
)


EXACT_WEIGHT_PASS = (
    max_difference
    <
    1e-10
)


print(
    "Exact official-weight reproduction passed:",
    EXACT_WEIGHT_PASS
)


assert EXACT_WEIGHT_PASS


print(
    "\n✓ Official GeoSIMCP numerical implementation reproduced."
)

print(
    "✓ d_feat + 1e-8 is preserved."
)

print(
    "✓ bandwidth + 1e-8 is preserved."
)

print(
    "✓ No parameter was tuned."
)

===== EXACT OFFICIAL GEOSIMCP NUMERICAL FORMULA =====

[1] Exact-source-formula comparison


,bandwidth,lambda,max_abs_difference,mean_abs_difference
0,1.000000,0.0,2.664535e-15,2.136514e-17
1,500.000000,0.0,1.110223e-16,4.934325e-21
2,30000.000000,0.5,1.085132e-12,4.726536e-13
3,28982.758621,1.0,2.325140e-12,8.883405e-13



Maximum difference across all configurations: 2.3251400804724653e-12
Exact official-weight reproduction passed: True

✓ Official GeoSIMCP numerical implementation reproduced.
✓ d_feat + 1e-8 is preserved.
✓ bandwidth + 1e-8 is preserved.
✓ No parameter was tuned.


In [32]:
# ============================================================
# Step 15D — FINAL end-to-end GCCP equivalence tests
#
# TEST A:
# Equal causal weights
# -> official GeoSIMCP-EUC prediction intervals
#
# TEST B:
# Actual causal distance + lambda=1
# -> official GeoCP prediction intervals
#
# Calibration + Validation only.
# Test remains untouched.
# ============================================================

import numpy as np
import pandas as pd

from geoconformal import (
    GeoConformalSpatialRegression,
    GeoSIMConformalSpatialRegression
)

from geoconformal.geocp import (
    weighted_quantile
)


print(
    "===== FINAL GCCP END-TO-END EQUIVALENCE TESTS ====="
)


EPS = 1e-8


# ============================================================
# 0. Shared calibration residuals / validation predictions
# ============================================================

calib_predictions = model.predict(
    X_calib_spatial
)


calibration_scores = np.abs(
    y_calib
    -
    calib_predictions
)


val_predictions = model.predict(
    X_val_spatial
)


# ============================================================
# TEST A
#
# Equal causal weights over the SAME 7-dimensional active
# feature space must reproduce official GeoSIMCP-EUC.
#
# We test:
#   (1) official selected setting: h=500, lambda=0
#   (2) a mixed setting: h=30000, lambda=0.5
# ============================================================

print(
    "\n===== TEST A ====="
)

print(
    "Equal causal weights -> official GeoSIMCP-EUC"
)


# d_equal was produced in Step 14B:
#
# equal weights on active 7 features
# -> exactly ordinary EUC raw distance.

d_equal_official_style = (
    d_equal
    +
    EPS
)


test_A_configs = [
    (500.0, 0.0),
    (30000.0, 0.5),
]


test_A_rows = []


for bw, lam in test_A_configs:

    # --------------------------------------------------------
    # Manual GCCP with equal causal weights
    # --------------------------------------------------------

    manual_joint = np.sqrt(

        lam
        *
        d_geo ** 2

        +

        (1.0 - lam)
        *
        d_equal_official_style ** 2
    )


    manual_weights = np.exp(

        -0.5
        *
        (
            manual_joint
            /
            (
                bw
                +
                EPS
            )
        ) ** 2
    )


    manual_q = weighted_quantile(

        scores=calibration_scores,

        weights=manual_weights,

        q=0.9,

        self_weights=None,

        interpolate=False
    )


    manual_lower = (
        val_predictions
        -
        manual_q
    )


    manual_upper = (
        val_predictions
        +
        manual_q
    )


    # --------------------------------------------------------
    # Official GeoSIMCP-EUC
    # --------------------------------------------------------

    official_sim = GeoSIMConformalSpatialRegression(

        predict_f=model.predict,

        miscoverage_level=0.1,

        bandwidth=bw,

        lambda_weight=lam,

        distance_metric="euclidean",

        standardize_weights=True,

        coord_calib=coords_calib,

        coord_test=coords_val,

        X_calib=X_calib_spatial,

        y_calib=y_calib,

        X_test=X_val_spatial,

        y_test=y_val,

        X_calib_weight=X_calib,

        X_test_weight=X_val,

        include_test_atom=False
    )


    official_result = (
        official_sim.analyze()
    )


    official_lower = np.asarray(
        official_result.lower_bound
    )


    official_upper = np.asarray(
        official_result.upper_bound
    )


    # --------------------------------------------------------
    # Compare final intervals
    # --------------------------------------------------------

    lower_diff = np.max(
        np.abs(
            manual_lower
            -
            official_lower
        )
    )


    upper_diff = np.max(
        np.abs(
            manual_upper
            -
            official_upper
        )
    )


    q_official = (
        official_upper
        -
        official_lower
    ) / 2.0


    q_diff = np.max(
        np.abs(
            manual_q
            -
            q_official
        )
    )


    manual_coverage = np.mean(

        (y_val >= manual_lower)

        &

        (y_val <= manual_upper)
    )


    test_A_rows.append(
        {
            "bandwidth":
                bw,

            "lambda":
                lam,

            "manual_coverage":
                manual_coverage,

            "official_coverage":
                official_result.coverage_probability,

            "max_q_difference":
                q_diff,

            "max_lower_difference":
                lower_diff,

            "max_upper_difference":
                upper_diff,
        }
    )


test_A_table = pd.DataFrame(
    test_A_rows
)


display(
    test_A_table
)


TEST_A_PASS = bool(

    (
        test_A_table[
            [
                "max_q_difference",
                "max_lower_difference",
                "max_upper_difference"
            ]
        ]
        .to_numpy()
        <
        1e-10
    )
    .all()
)


print(
    "TEST A passed:",
    TEST_A_PASS
)


assert TEST_A_PASS


# ============================================================
# TEST B
#
# Use REAL causal distance.
#
# When lambda=1:
#
# (1-lambda)*d_causal^2 = 0
#
# Therefore GCCP must reduce exactly to GeoCP.
# ============================================================

print(
    "\n===== TEST B ====="
)

print(
    "Actual causal GCCP with lambda=1 -> official GeoCP"
)


GEOGRAPHIC_BW = float(
    best_geocp[
        "bandwidth"
    ]
)


print(
    "GeoCP bandwidth:",
    GEOGRAPHIC_BW
)


# ------------------------------------------------------------
# 1. Real causal distance with official epsilon
# ------------------------------------------------------------

d_causal_official_style = (
    d_causal
    +
    EPS
)


lambda_geo_only = 1.0


manual_joint_geo = np.sqrt(

    lambda_geo_only
    *
    d_geo ** 2

    +

    (1.0 - lambda_geo_only)
    *
    d_causal_official_style ** 2
)


manual_geo_weights = np.exp(

    -0.5
    *
    (
        manual_joint_geo
        /
        (
            GEOGRAPHIC_BW
            +
            EPS
        )
    ) ** 2
)


manual_geo_q = weighted_quantile(

    scores=calibration_scores,

    weights=manual_geo_weights,

    q=0.9,

    self_weights=None,

    interpolate=False
)


manual_geo_lower = (
    val_predictions
    -
    manual_geo_q
)


manual_geo_upper = (
    val_predictions
    +
    manual_geo_q
)


# ------------------------------------------------------------
# 2. Official GeoCP
# ------------------------------------------------------------

official_geocp = GeoConformalSpatialRegression(

    predict_f=model.predict,

    miscoverage_level=0.1,

    bandwidth=GEOGRAPHIC_BW,

    coord_calib=coords_calib,

    coord_test=coords_val,

    X_calib=X_calib_spatial,

    y_calib=y_calib,

    X_test=X_val_spatial,

    y_test=y_val,

    include_test_atom=False
)


official_geocp_result = (
    official_geocp.analyze()
)


official_geo_lower = np.asarray(
    official_geocp_result.lower_bound
)


official_geo_upper = np.asarray(
    official_geocp_result.upper_bound
)


official_geo_q = (
    official_geo_upper
    -
    official_geo_lower
) / 2.0


# ------------------------------------------------------------
# 3. Compare end to end
# ------------------------------------------------------------

geo_q_diff = np.max(
    np.abs(
        manual_geo_q
        -
        official_geo_q
    )
)


geo_lower_diff = np.max(
    np.abs(
        manual_geo_lower
        -
        official_geo_lower
    )
)


geo_upper_diff = np.max(
    np.abs(
        manual_geo_upper
        -
        official_geo_upper
    )
)


manual_geo_coverage = np.mean(

    (y_val >= manual_geo_lower)

    &

    (y_val <= manual_geo_upper)
)


print(
    "\nManual GCCP(lambda=1) Coverage:",
    manual_geo_coverage
)


print(
    "Official GeoCP Coverage:",
    official_geocp_result.coverage_probability
)


print(
    "\nMaximum q* difference:",
    geo_q_diff
)


print(
    "Maximum lower-bound difference:",
    geo_lower_diff
)


print(
    "Maximum upper-bound difference:",
    geo_upper_diff
)


TEST_B_PASS = (

    geo_q_diff
    <
    1e-10

    and

    geo_lower_diff
    <
    1e-10

    and

    geo_upper_diff
    <
    1e-10
)


print(
    "\nTEST B passed:",
    TEST_B_PASS
)


assert TEST_B_PASS


# ============================================================
# FINAL
# ============================================================

print(
    "\n===== FINAL UNIT-TEST STATUS ====="
)

print(
    "TEST A — equal causal weights -> GeoSIMCP-EUC:",
    TEST_A_PASS
)

print(
    "TEST B — lambda=1 -> GeoCP:",
    TEST_B_PASS
)


print(
    "\n✓ GCCP implementation equivalence tests completed."
)

print(
    "✓ Feature-space handling matches official GeoSIMCP."
)

print(
    "✓ Numerical epsilon handling matches official GeoSIMCP."
)

print(
    "✓ Weighted quantile and final intervals are equivalent."
)

print(
    "✓ Calibration + Validation only."
)

print(
    "✓ Test remains untouched."
)

===== FINAL GCCP END-TO-END EQUIVALENCE TESTS =====

===== TEST A =====
Equal causal weights -> official GeoSIMCP-EUC


,bandwidth,lambda,manual_coverage,official_coverage,max_q_difference,max_lower_difference,max_upper_difference
0,500.0,0.0,0.920000,0.920000,0.0,0.0,0.0
1,30000.0,0.5,0.923333,0.923333,0.0,0.0,0.0


TEST A passed: True

===== TEST B =====
Actual causal GCCP with lambda=1 -> official GeoCP
GeoCP bandwidth: 28982.758620689656

Manual GCCP(lambda=1) Coverage: 0.9233333333333333
Official GeoCP Coverage: 0.9233333333333333

Maximum q* difference: 0.0
Maximum lower-bound difference: 0.0
Maximum upper-bound difference: 0.0

TEST B passed: True

===== FINAL UNIT-TEST STATUS =====
TEST A — equal causal weights -> GeoSIMCP-EUC: True
TEST B — lambda=1 -> GeoCP: True

✓ GCCP implementation equivalence tests completed.
✓ Feature-space handling matches official GeoSIMCP.
✓ Numerical epsilon handling matches official GeoSIMCP.
✓ Weighted quantile and final intervals are equivalent.
✓ Calibration + Validation only.
✓ Test remains untouched.


In [33]:
# ============================================================
# Step 16 — GCCP-original Validation grid search
#
# 核心原则：
#
# 完全保留 GeoSIMCP 0.3.0：
# - same Calibration / Validation
# - same residuals
# - same standardization
# - same active feature space
# - same joint-distance formula
# - same Gaussian kernel
# - same +1e-8 numerical treatment
# - same weighted quantile
# - same bandwidth grid
# - same lambda grid
# - same selection rule
#
# 唯一改变：
#
#     EUC feature distance
#          ->
#     causal-weighted feature distance
#
# Test is NOT used.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from geoconformal.geocp import (
    weighted_quantile
)


print(
    "===== GCCP-ORIGINAL VALIDATION GRID SEARCH ====="
)


# ------------------------------------------------------------
# 1. Frozen official GeoSIMCP grid
# ------------------------------------------------------------

gccp_bandwidths = np.linspace(
    500,
    30000,
    15
)


gccp_lambdas = np.arange(
    0,
    1.05,
    0.1
)


print(
    "\nBandwidth candidates:",
    len(gccp_bandwidths)
)

print(
    "Lambda candidates:",
    len(gccp_lambdas)
)

print(
    "Total configurations:",
    len(gccp_bandwidths)
    *
    len(gccp_lambdas)
)


# ------------------------------------------------------------
# 2. Frozen calibration scores / predictions
# ------------------------------------------------------------

calib_predictions = model.predict(
    X_calib_spatial
)


calibration_scores = np.abs(
    y_calib
    -
    calib_predictions
)


val_predictions = model.predict(
    X_val_spatial
)


# ------------------------------------------------------------
# 3. Official numerical treatment
#
# GeoSIMCP source:
#
# d_feat = raw feature distance + 1e-8
# denominator = bandwidth + 1e-8
# ------------------------------------------------------------

EPS = 1e-8


d_causal_official = (
    d_causal
    +
    EPS
)


# ------------------------------------------------------------
# 4. Interval Score
# ------------------------------------------------------------

def gccp_interval_score(
    lower,
    upper,
    y_true,
    alpha=0.1
):

    width = (
        upper
        -
        lower
    )


    penalty = (
        2.0
        /
        alpha
    ) * (

        np.maximum(
            lower
            -
            y_true,
            0.0
        )

        +

        np.maximum(
            y_true
            -
            upper,
            0.0
        )
    )


    return float(
        np.mean(
            width
            +
            penalty
        )
    )


# ------------------------------------------------------------
# 5. Validation grid search
# ------------------------------------------------------------

gccp_validation_rows = []


for bw in gccp_bandwidths:

    for lam in gccp_lambdas:

        # ----------------------------------------------------
        # Original GeoSIMCP joint-distance formula
        # ----------------------------------------------------

        d_joint = np.sqrt(

            lam
            *
            d_geo ** 2

            +

            (1.0 - lam)
            *
            d_causal_official ** 2
        )


        # ----------------------------------------------------
        # Original Gaussian similarity kernel
        # ----------------------------------------------------

        similarity_weights = np.exp(

            -0.5
            *
            (
                d_joint
                /
                (
                    bw
                    +
                    EPS
                )
            ) ** 2
        )


        # ----------------------------------------------------
        # Local weighted conformal quantile
        # ----------------------------------------------------

        q_local = weighted_quantile(

            scores=calibration_scores,

            weights=similarity_weights,

            q=0.9,

            self_weights=None,

            interpolate=False
        )


        lower = (
            val_predictions
            -
            q_local
        )


        upper = (
            val_predictions
            +
            q_local
        )


        # ----------------------------------------------------
        # Validation metrics
        # ----------------------------------------------------

        coverage = float(
            np.mean(

                (y_val >= lower)

                &

                (y_val <= upper)
            )
        )


        mean_interval = float(
            np.mean(
                upper
                -
                lower
            )
        )


        score = gccp_interval_score(

            lower=lower,

            upper=upper,

            y_true=y_val,

            alpha=0.1
        )


        # ----------------------------------------------------
        # Effective sample size diagnostic
        #
        # n_eff = (sum w)^2 / sum(w^2)
        # ----------------------------------------------------

        weight_sum = np.sum(
            similarity_weights,
            axis=1
        )


        weight_sq_sum = np.sum(
            similarity_weights ** 2,
            axis=1
        )


        n_eff = (

            weight_sum ** 2

            /

            np.where(
                weight_sq_sum > 0,
                weight_sq_sum,
                np.nan
            )
        )


        gccp_validation_rows.append(
            {
                "bandwidth":
                    float(bw),

                "lambda":
                    float(lam),

                "coverage":
                    coverage,

                "mean_interval":
                    mean_interval,

                "interval_score":
                    score,

                "median_q":
                    float(
                        np.median(
                            q_local
                        )
                    ),

                "mean_q":
                    float(
                        np.mean(
                            q_local
                        )
                    ),

                "median_n_eff":
                    float(
                        np.nanmedian(
                            n_eff
                        )
                    ),

                "mean_n_eff":
                    float(
                        np.nanmean(
                            n_eff
                        )
                    ),
            }
        )


df_gccp_val = pd.DataFrame(
    gccp_validation_rows
)


# ============================================================
# 6. Official GeoSIMCP parameter-selection rule
# ============================================================

valid_gccp = (
    df_gccp_val[
        df_gccp_val[
            "coverage"
        ]
        >=
        0.90
    ]
    .copy()
)


if len(valid_gccp) > 0:

    gccp_selection_mode = (
        "Coverage >= 0.90 -> minimum Interval Score"
    )


    best_gccp = valid_gccp.loc[

        valid_gccp[
            "interval_score"
        ]
        .idxmin()
    ]


else:

    gccp_selection_mode = (
        "No Coverage >= 0.90 candidate "
        "-> minimum Interval Score over all candidates"
    )


    best_gccp = df_gccp_val.loc[

        df_gccp_val[
            "interval_score"
        ]
        .idxmin()
    ]


# ============================================================
# 7. Report result
# ============================================================

print(
    "\n===== GCCP-ORIGINAL VALIDATION RESULT ====="
)


print(
    "Selection mode:",
    gccp_selection_mode
)


print(
    "Candidates with Coverage >= 0.90:",
    len(
        valid_gccp
    )
)


print(
    "\nBest parameters:"
)


print(
    "Bandwidth =",
    best_gccp[
        "bandwidth"
    ]
)


print(
    "Lambda    =",
    best_gccp[
        "lambda"
    ]
)


print(
    "\nValidation metrics:"
)


print(
    "Coverage       =",
    best_gccp[
        "coverage"
    ]
)


print(
    "Mean Interval  =",
    best_gccp[
        "mean_interval"
    ]
)


print(
    "Interval Score =",
    best_gccp[
        "interval_score"
    ]
)


print(
    "Median q*      =",
    best_gccp[
        "median_q"
    ]
)


print(
    "Median n_eff   =",
    best_gccp[
        "median_n_eff"
    ]
)


# ============================================================
# 8. Top candidates satisfying nominal coverage
# ============================================================

if len(valid_gccp) > 0:

    print(
        "\nTop 15 candidates satisfying Coverage >= 0.90:"
    )


    display(

        valid_gccp
        .sort_values(
            [
                "interval_score",
                "mean_interval"
            ]
        )
        .head(15)
        .reset_index(
            drop=True
        )
        .round(6)
    )


else:

    print(
        "\nNo configuration reached 0.90 Coverage."
    )


    display(

        df_gccp_val
        .sort_values(
            "interval_score"
        )
        .head(15)
        .reset_index(
            drop=True
        )
        .round(6)
    )


# ============================================================
# 9. Inspect best result for EACH lambda
#
# This is important for diagnosing the scale problem.
# ============================================================

best_by_lambda_rows = []


for lam in gccp_lambdas:

    subset = (
        df_gccp_val[
            np.isclose(
                df_gccp_val[
                    "lambda"
                ],
                lam
            )
        ]
        .copy()
    )


    valid_subset = (
        subset[
            subset[
                "coverage"
            ]
            >=
            0.90
        ]
    )


    if len(valid_subset) > 0:

        row = valid_subset.loc[

            valid_subset[
                "interval_score"
            ]
            .idxmin()
        ]

        branch_mode = (
            "coverage>=0.90"
        )

    else:

        row = subset.loc[

            subset[
                "interval_score"
            ]
            .idxmin()
        ]

        branch_mode = (
            "no-valid-coverage"
        )


    best_by_lambda_rows.append(
        {
            "lambda":
                float(lam),

            "bandwidth":
                float(
                    row[
                        "bandwidth"
                    ]
                ),

            "coverage":
                float(
                    row[
                        "coverage"
                    ]
                ),

            "mean_interval":
                float(
                    row[
                        "mean_interval"
                    ]
                ),

            "interval_score":
                float(
                    row[
                        "interval_score"
                    ]
                ),

            "median_n_eff":
                float(
                    row[
                        "median_n_eff"
                    ]
                ),

            "selection_branch":
                branch_mode,
        }
    )


best_by_lambda = pd.DataFrame(
    best_by_lambda_rows
)


print(
    "\n===== BEST CONFIGURATION WITHIN EACH LAMBDA ====="
)


display(
    best_by_lambda.round(
        6
    )
)


# ============================================================
# 10. Important special cases
# ============================================================

print(
    "\n===== SPECIAL CASES ====="
)


# lambda = 0 -> causal feature only
lambda_zero = (
    df_gccp_val[
        np.isclose(
            df_gccp_val[
                "lambda"
            ],
            0.0
        )
    ]
    .sort_values(
        "interval_score"
    )
    .head(5)
)


print(
    "\nBest lambda=0 configurations "
    "(causal-feature-only):"
)

display(
    lambda_zero.round(
        6
    )
)


# lambda = 1 -> GeoCP branch
lambda_one = (
    df_gccp_val[
        np.isclose(
            df_gccp_val[
                "lambda"
            ],
            1.0
        )
    ]
    .sort_values(
        "interval_score"
    )
    .head(5)
)


print(
    "\nBest lambda=1 configurations "
    "(pure geographic / GeoCP branch):"
)

display(
    lambda_one.round(
        6
    )
)


# ============================================================
# 11. Save full Validation results
# ============================================================

GCCP_ORIGINAL_VAL_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccp_author_aligned_original_validation_v1.csv"
)


GCCP_ORIGINAL_LAMBDA_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccp_author_aligned_original_best_by_lambda_v1.csv"
)


df_gccp_val.to_csv(
    GCCP_ORIGINAL_VAL_PATH,
    index=False
)


best_by_lambda.to_csv(
    GCCP_ORIGINAL_LAMBDA_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    GCCP_ORIGINAL_VAL_PATH
)

print(
    GCCP_ORIGINAL_LAMBDA_PATH
)


print(
    "\n✓ GCCP-original Validation search completed."
)

print(
    "✓ Same official GeoSIMCP parameter grid was used."
)

print(
    "✓ Same official selection rule was used."
)

print(
    "✓ Test was NOT used."
)

===== GCCP-ORIGINAL VALIDATION GRID SEARCH =====

Bandwidth candidates: 15
Lambda candidates: 11
Total configurations: 165

===== GCCP-ORIGINAL VALIDATION RESULT =====
Selection mode: Coverage >= 0.90 -> minimum Interval Score
Candidates with Coverage >= 0.90: 153

Best parameters:
Bandwidth = 6821.4285714285725
Lambda    = 0.0

Validation metrics:
Coverage       = 0.92
Mean Interval  = 0.2541355109619765
Interval Score = 0.3210766044099667
Median q*      = 0.12706775548098825
Median n_eff   = 299.99999999999534

Top 15 candidates satisfying Coverage >= 0.90:


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
0,6821.428571,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
1,8928.571429,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
2,11035.714286,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
3,13142.857143,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
4,15250.000000,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
5,17357.142857,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
6,19464.285714,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
7,21571.428571,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
8,23678.571429,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
9,25785.714286,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0



===== BEST CONFIGURATION WITHIN EACH LAMBDA =====


,lambda,bandwidth,coverage,mean_interval,interval_score,median_n_eff,selection_branch
0,0.0,6821.428571,0.920000,0.254136,0.321077,300.000000,coverage>=0.90
1,0.1,11035.714286,0.923333,0.257107,0.321989,291.403747,coverage>=0.90
2,0.2,15250.000000,0.923333,0.257133,0.322015,290.721583,coverage>=0.90
3,0.3,27892.857143,0.923333,0.257244,0.321989,297.738655,coverage>=0.90
4,0.4,21571.428571,0.923333,0.257133,0.322015,290.728184,coverage>=0.90
5,0.5,25785.714286,0.923333,0.257121,0.322003,292.621350,coverage>=0.90
6,0.6,27892.857143,0.923333,0.257121,0.322003,292.297752,coverage>=0.90
7,0.7,30000.000000,0.923333,0.257121,0.322003,292.181803,coverage>=0.90
8,0.8,30000.000000,0.923333,0.257131,0.322150,290.192121,coverage>=0.90
9,0.9,30000.000000,0.923333,0.257267,0.322200,288.087596,coverage>=0.90



===== SPECIAL CASES =====

Best lambda=0 configurations (causal-feature-only):


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
33,6821.428571,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
66,13142.857143,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
55,11035.714286,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
44,8928.571429,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0
77,15250.000000,0.0,0.92,0.254136,0.321077,0.127068,0.127068,300.0,300.0



Best lambda=1 configurations (pure geographic / GeoCP branch):


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
164,30000.000000,1.0,0.923333,0.257427,0.322360,0.129113,0.128713,285.985445,284.121857
142,25785.714286,1.0,0.923333,0.257615,0.322513,0.129113,0.128807,278.137861,274.948187
153,27892.857143,1.0,0.923333,0.257559,0.322517,0.129113,0.128780,282.538998,280.119972
131,23678.571429,1.0,0.923333,0.257544,0.322577,0.129113,0.128772,272.283151,268.259744
120,21571.428571,1.0,0.923333,0.257381,0.322894,0.129113,0.128690,264.572871,259.625947



===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp_author_aligned_original_validation_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp_author_aligned_original_best_by_lambda_v1.csv

✓ GCCP-original Validation search completed.
✓ Same official GeoSIMCP parameter grid was used.
✓ Same official selection rule was used.
✓ Test was NOT used.


In [34]:
# ============================================================
# Step 17 — Author-aligned GeoSIMCP-MND Validation tuning
#
# 目的：
# 在与 GeoSIMCP-EUC / GCCP-original 完全相同的实验协议下
# 对 MND 进行公平的 Validation grid search。
#
# Same:
# - Train / Calibration / Validation
# - XGBoost predictor
# - UTM coordinates
# - bandwidth grid
# - lambda grid
# - weighted conformal framework
# - selection rule
# - include_test_atom=False
#
# Only difference:
# distance_metric = "mnd"
#
# Test is NOT used.
# ============================================================

import numpy as np
import pandas as pd

from geoconformal import (
    GeoSIMConformalSpatialRegression
)

from geoconformal.geocp.weights import (
    joint_geo_feature_weights
)


print(
    "===== GEOSIMCP-MND VALIDATION GRID SEARCH ====="
)


# ------------------------------------------------------------
# 1. Same official GeoSIMCP parameter grid
# ------------------------------------------------------------

mnd_bandwidths = np.linspace(
    500,
    30000,
    15
)


mnd_lambdas = np.arange(
    0,
    1.05,
    0.1
)


print(
    "\nBandwidth candidates:",
    len(mnd_bandwidths)
)

print(
    "Lambda candidates:",
    len(mnd_lambdas)
)

print(
    "Total configurations:",
    len(mnd_bandwidths)
    *
    len(mnd_lambdas)
)


# ------------------------------------------------------------
# 2. Input format expected by official weight function
#
# First 2 columns = coordinates
# Remaining columns = feature-weight variables
# ------------------------------------------------------------

official_input_val = np.hstack(
    [
        coords_val,
        X_val
    ]
)


# ------------------------------------------------------------
# 3. Validation grid search
# ------------------------------------------------------------

mnd_validation_rows = []


for bw in mnd_bandwidths:

    for lam in mnd_lambdas:

        # ====================================================
        # Official GeoSIMCP-MND
        # ====================================================

        mnd_model = GeoSIMConformalSpatialRegression(

            predict_f=model.predict,

            miscoverage_level=0.1,

            bandwidth=bw,

            lambda_weight=lam,

            distance_metric="mnd",

            # For MND the source uses its own
            # range-normalized feature-distance calculation.
            # This argument is kept explicit for API consistency.
            standardize_weights=True,

            coord_calib=coords_calib,

            coord_test=coords_val,

            X_calib=X_calib_spatial,

            y_calib=y_calib,

            X_test=X_val_spatial,

            y_test=y_val,

            # Same 8 non-spatial variables supplied
            # to the official GeoSIMCP feature-weight component.
            X_calib_weight=X_calib,

            X_test_weight=X_val,

            include_test_atom=False
        )


        result = mnd_model.analyze()


        # ====================================================
        # Validation metrics
        # ====================================================

        lower = np.asarray(
            result.lower_bound
        )


        upper = np.asarray(
            result.upper_bound
        )


        coverage = float(
            result.coverage_probability
        )


        mean_interval = float(
            np.mean(
                upper
                -
                lower
            )
        )


        score = interval_score(

            upper,

            lower,

            y_val,

            alpha=0.1
        )


        q_local = (
            upper
            -
            lower
        ) / 2.0


        # ====================================================
        # Effective sample size diagnostic
        #
        # Use the SAME official weight implementation.
        # ====================================================

        mnd_weight_fn = (
            joint_geo_feature_weights(

                coord_calib=coords_calib,

                feat_calib=X_calib,

                bandwidth=bw,

                lambda_weight=lam,

                distance_metric="mnd",

                standardize=True
            )
        )


        mnd_weights = (
            mnd_weight_fn(
                official_input_val,
                None
            )
        )


        weight_sum = np.sum(
            mnd_weights,
            axis=1
        )


        weight_sq_sum = np.sum(
            mnd_weights ** 2,
            axis=1
        )


        n_eff = (

            weight_sum ** 2

            /

            np.where(
                weight_sq_sum > 0,
                weight_sq_sum,
                np.nan
            )
        )


        mnd_validation_rows.append(
            {
                "bandwidth":
                    float(bw),

                "lambda":
                    float(lam),

                "coverage":
                    coverage,

                "mean_interval":
                    mean_interval,

                "interval_score":
                    score,

                "median_q":
                    float(
                        np.median(
                            q_local
                        )
                    ),

                "mean_q":
                    float(
                        np.mean(
                            q_local
                        )
                    ),

                "median_n_eff":
                    float(
                        np.nanmedian(
                            n_eff
                        )
                    ),

                "mean_n_eff":
                    float(
                        np.nanmean(
                            n_eff
                        )
                    ),
            }
        )


df_mnd_val = pd.DataFrame(
    mnd_validation_rows
)


# ============================================================
# 4. Same official GeoSIMCP selection rule
# ============================================================

valid_mnd = (
    df_mnd_val[
        df_mnd_val[
            "coverage"
        ]
        >=
        0.90
    ]
    .copy()
)


if len(valid_mnd) > 0:

    mnd_selection_mode = (
        "Coverage >= 0.90 -> minimum Interval Score"
    )


    best_mnd = valid_mnd.loc[

        valid_mnd[
            "interval_score"
        ]
        .idxmin()
    ]


else:

    mnd_selection_mode = (
        "No Coverage >= 0.90 candidate "
        "-> minimum Interval Score over all candidates"
    )


    best_mnd = df_mnd_val.loc[

        df_mnd_val[
            "interval_score"
        ]
        .idxmin()
    ]


# ============================================================
# 5. Main result
# ============================================================

print(
    "\n===== GEOSIMCP-MND VALIDATION RESULT ====="
)


print(
    "Selection mode:",
    mnd_selection_mode
)


print(
    "Candidates with Coverage >= 0.90:",
    len(
        valid_mnd
    )
)


print(
    "\nBest parameters:"
)


print(
    "Bandwidth =",
    best_mnd[
        "bandwidth"
    ]
)


print(
    "Lambda    =",
    best_mnd[
        "lambda"
    ]
)


print(
    "\nValidation metrics:"
)


print(
    "Coverage       =",
    best_mnd[
        "coverage"
    ]
)


print(
    "Mean Interval  =",
    best_mnd[
        "mean_interval"
    ]
)


print(
    "Interval Score =",
    best_mnd[
        "interval_score"
    ]
)


print(
    "Median q*      =",
    best_mnd[
        "median_q"
    ]
)


print(
    "Median n_eff   =",
    best_mnd[
        "median_n_eff"
    ]
)


# ============================================================
# 6. Top 15 valid candidates
# ============================================================

if len(valid_mnd) > 0:

    print(
        "\nTop 15 candidates satisfying Coverage >= 0.90:"
    )


    display(

        valid_mnd
        .sort_values(
            [
                "interval_score",
                "mean_interval"
            ]
        )
        .head(15)
        .reset_index(
            drop=True
        )
        .round(6)
    )


else:

    print(
        "\nNo configuration reached 0.90 Coverage."
    )


    display(

        df_mnd_val
        .sort_values(
            "interval_score"
        )
        .head(15)
        .reset_index(
            drop=True
        )
        .round(6)
    )


# ============================================================
# 7. Best configuration within each lambda
# ============================================================

best_mnd_by_lambda_rows = []


for lam in mnd_lambdas:

    subset = (
        df_mnd_val[
            np.isclose(
                df_mnd_val[
                    "lambda"
                ],
                lam
            )
        ]
        .copy()
    )


    valid_subset = (
        subset[
            subset[
                "coverage"
            ]
            >=
            0.90
        ]
    )


    if len(valid_subset) > 0:

        row = valid_subset.loc[

            valid_subset[
                "interval_score"
            ]
            .idxmin()
        ]

        branch = (
            "coverage>=0.90"
        )


    else:

        row = subset.loc[

            subset[
                "interval_score"
            ]
            .idxmin()
        ]

        branch = (
            "no-valid-coverage"
        )


    best_mnd_by_lambda_rows.append(
        {
            "lambda":
                float(lam),

            "bandwidth":
                float(
                    row[
                        "bandwidth"
                    ]
                ),

            "coverage":
                float(
                    row[
                        "coverage"
                    ]
                ),

            "mean_interval":
                float(
                    row[
                        "mean_interval"
                    ]
                ),

            "interval_score":
                float(
                    row[
                        "interval_score"
                    ]
                ),

            "median_n_eff":
                float(
                    row[
                        "median_n_eff"
                    ]
                ),

            "selection_branch":
                branch,
        }
    )


best_mnd_by_lambda = pd.DataFrame(
    best_mnd_by_lambda_rows
)


print(
    "\n===== BEST MND CONFIGURATION WITHIN EACH LAMBDA ====="
)


display(
    best_mnd_by_lambda.round(
        6
    )
)


# ============================================================
# 8. Special cases
# ============================================================

print(
    "\n===== MND SPECIAL CASES ====="
)


mnd_lambda_zero = (
    df_mnd_val[
        np.isclose(
            df_mnd_val[
                "lambda"
            ],
            0.0
        )
    ]
    .sort_values(
        "interval_score"
    )
    .head(5)
)


print(
    "\nBest lambda=0 configurations "
    "(MND-feature-only):"
)

display(
    mnd_lambda_zero.round(
        6
    )
)


mnd_lambda_one = (
    df_mnd_val[
        np.isclose(
            df_mnd_val[
                "lambda"
            ],
            1.0
        )
    ]
    .sort_values(
        "interval_score"
    )
    .head(5)
)


print(
    "\nBest lambda=1 configurations "
    "(pure geographic / GeoCP branch):"
)

display(
    mnd_lambda_one.round(
        6
    )
)


# ============================================================
# 9. Save
# ============================================================

MND_VAL_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "geosimcp_mnd_author_aligned_validation_v1.csv"
)


MND_LAMBDA_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "geosimcp_mnd_author_aligned_best_by_lambda_v1.csv"
)


df_mnd_val.to_csv(
    MND_VAL_PATH,
    index=False
)


best_mnd_by_lambda.to_csv(
    MND_LAMBDA_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    MND_VAL_PATH
)

print(
    MND_LAMBDA_PATH
)


print(
    "\n✓ GeoSIMCP-MND Validation tuning completed."
)

print(
    "✓ Same grid as EUC and GCCP-original."
)

print(
    "✓ Same selection rule."
)

print(
    "✓ Test was NOT used."
)

===== GEOSIMCP-MND VALIDATION GRID SEARCH =====

Bandwidth candidates: 15
Lambda candidates: 11
Total configurations: 165

===== GEOSIMCP-MND VALIDATION RESULT =====
Selection mode: Coverage >= 0.90 -> minimum Interval Score
Candidates with Coverage >= 0.90: 153

Best parameters:
Bandwidth = 30000.0
Lambda    = 1.0

Validation metrics:
Coverage       = 0.9233333333333333
Mean Interval  = 0.2574269467512775
Interval Score = 0.32235986627651625
Median q*      = 0.12911264938329392
Median n_eff   = 285.98544541225453

Top 15 candidates satisfying Coverage >= 0.90:


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
0,30000.000000,1.0,0.923333,0.257427,0.322360,0.129113,0.128713,285.985445,284.121857
1,25785.714286,1.0,0.923333,0.257615,0.322513,0.129113,0.128807,278.137861,274.948187
2,27892.857143,1.0,0.923333,0.257559,0.322517,0.129113,0.128780,282.538998,280.119972
3,23678.571429,1.0,0.923333,0.257544,0.322577,0.129113,0.128772,272.283151,268.259744
4,21571.428571,1.0,0.923333,0.257381,0.322894,0.129113,0.128690,264.572871,259.625947
5,19464.285714,1.0,0.923333,0.257099,0.322974,0.129113,0.128550,255.331214,248.527931
6,17357.142857,1.0,0.923333,0.256714,0.323260,0.129113,0.128357,242.712238,234.340806
7,15250.000000,1.0,0.923333,0.255616,0.323807,0.129113,0.127808,226.072384,216.300203
8,13142.857143,1.0,0.926667,0.255047,0.325315,0.129113,0.127524,205.607243,193.487643
9,11035.714286,1.0,0.926667,0.252327,0.328290,0.127068,0.126164,174.096662,165.039502



===== BEST MND CONFIGURATION WITHIN EACH LAMBDA =====


,lambda,bandwidth,coverage,mean_interval,interval_score,median_n_eff,selection_branch
0,0.0,500.000000,0.920000,inf,inf,300.000000,coverage>=0.90
1,0.1,2607.142857,0.933333,inf,inf,124.202823,coverage>=0.90
2,0.2,2607.142857,0.930000,inf,inf,80.806613,coverage>=0.90
3,0.3,2607.142857,0.916667,inf,inf,61.862429,coverage>=0.90
4,0.4,2607.142857,0.913333,inf,inf,48.124333,coverage>=0.90
5,0.5,2607.142857,0.906667,inf,inf,39.063277,coverage>=0.90
6,0.6,2607.142857,0.903333,inf,inf,32.433627,coverage>=0.90
7,0.7,2607.142857,0.906667,inf,inf,27.985690,coverage>=0.90
8,0.8,2607.142857,0.903333,inf,inf,24.597916,coverage>=0.90
9,0.9,4714.285714,0.916667,inf,inf,65.957976,coverage>=0.90



===== MND SPECIAL CASES =====

Best lambda=0 configurations (MND-feature-only):


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
0,500.000000,0.0,0.92,inf,inf,0.127068,inf,300.0,300.0
11,2607.142857,0.0,0.92,inf,inf,0.127068,inf,300.0,300.0
22,4714.285714,0.0,0.92,inf,inf,0.127068,inf,300.0,300.0
33,6821.428571,0.0,0.92,inf,inf,0.127068,inf,300.0,300.0
44,8928.571429,0.0,0.92,inf,inf,0.127068,inf,300.0,300.0



Best lambda=1 configurations (pure geographic / GeoCP branch):


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
164,30000.000000,1.0,0.923333,0.257427,0.322360,0.129113,0.128713,285.985445,284.121857
142,25785.714286,1.0,0.923333,0.257615,0.322513,0.129113,0.128807,278.137861,274.948187
153,27892.857143,1.0,0.923333,0.257559,0.322517,0.129113,0.128780,282.538998,280.119972
131,23678.571429,1.0,0.923333,0.257544,0.322577,0.129113,0.128772,272.283151,268.259744
120,21571.428571,1.0,0.923333,0.257381,0.322894,0.129113,0.128690,264.572871,259.625947



===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geosimcp_mnd_author_aligned_validation_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geosimcp_mnd_author_aligned_best_by_lambda_v1.csv

✓ GeoSIMCP-MND Validation tuning completed.
✓ Same grid as EUC and GCCP-original.
✓ Same selection rule.
✓ Test was NOT used.


In [35]:
# ============================================================
# Step 17A — Diagnose GeoSIMCP-MND infinite intervals
#
# 目的：
# 证明 MND 的 +inf 是否来自：
#
# Calibration waterfront range = 0
# +
# Validation waterfront = 1
#
# 只做诊断，不修改 MND。
# ============================================================

import numpy as np
import pandas as pd

from geoconformal.geocp import weighted_quantile


print(
    "===== MND INFINITE-INTERVAL DIAGNOSTIC ====="
)


feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


# ------------------------------------------------------------
# 1. Calibration ranges exactly as MND uses them
# ------------------------------------------------------------

feature_min = np.min(
    X_calib,
    axis=0
)

feature_max = np.max(
    X_calib,
    axis=0
)

feature_range_raw = (
    feature_max
    -
    feature_min
)


# Source-style numerical safeguard
feature_range_mnd = (
    feature_range_raw.copy()
)

feature_range_mnd[
    feature_range_mnd == 0
] = 1e-8


range_table = pd.DataFrame(
    {
        "feature":
            feature_order,

        "calib_min":
            feature_min,

        "calib_max":
            feature_max,

        "raw_range":
            feature_range_raw,

        "MND_range_used":
            feature_range_mnd,
    }
)


print(
    "\n[1] Calibration feature ranges"
)

display(
    range_table
)


# ------------------------------------------------------------
# 2. Manual MND distances
#
# d_MND = max normalized feature difference
# ------------------------------------------------------------

normalized_diff = (

    np.abs(
        X_val[:, None, :]
        -
        X_calib[None, :, :]
    )

    /

    feature_range_mnd[
        None,
        None,
        :
    ]
)


d_mnd_manual = np.max(
    normalized_diff,
    axis=2
)


# Which feature caused the maximum difference?
max_feature_index = np.argmax(
    normalized_diff,
    axis=2
)


print(
    "\n[2] MND distance distribution"
)

print(
    "Minimum:",
    np.min(
        d_mnd_manual
    )
)

print(
    "Median:",
    np.median(
        d_mnd_manual
    )
)

print(
    "Mean:",
    np.mean(
        d_mnd_manual
    )
)

print(
    "Maximum:",
    np.max(
        d_mnd_manual
    )
)


# ------------------------------------------------------------
# 3. Per-validation-sample maximum MND distance
# ------------------------------------------------------------

per_val_max_mnd = np.max(
    d_mnd_manual,
    axis=1
)


problem_indices = np.where(
    per_val_max_mnd > 1e6
)[0]


print(
    "\n[3] Validation samples with MND distance > 1e6"
)

print(
    "Count:",
    len(
        problem_indices
    )
)

print(
    "Indices:",
    problem_indices.tolist()
)


problem_rows = []


waterfront_idx = feature_order.index(
    "waterfront"
)


for idx in problem_indices:

    # Feature responsible most often for max MND
    responsible = (
        max_feature_index[
            idx
        ]
    )

    values, counts = np.unique(
        responsible,
        return_counts=True
    )

    dominant_feature_idx = values[
        np.argmax(
            counts
        )
    ]


    problem_rows.append(
        {
            "val_index":
                int(
                    idx
                ),

            "waterfront_value":
                X_val[
                    idx,
                    waterfront_idx
                ],

            "max_MND_distance":
                per_val_max_mnd[
                    idx
                ],

            "dominant_max_feature":
                feature_order[
                    int(
                        dominant_feature_idx
                    )
                ],

            "true_y":
                y_val[
                    idx
                ],
        }
    )


problem_table = pd.DataFrame(
    problem_rows
)


display(
    problem_table
)


# ------------------------------------------------------------
# 4. Reproduce one problematic MND kernel
#
# Use lambda=0, bandwidth=500.
# ------------------------------------------------------------

bw = 500.0
lam = 0.0
EPS = 1e-8


d_mnd_official = (
    d_mnd_manual
    +
    EPS
)


d_joint = np.sqrt(

    lam
    *
    d_geo ** 2

    +

    (1.0 - lam)
    *
    d_mnd_official ** 2
)


mnd_weights_manual = np.exp(

    -0.5
    *
    (
        d_joint
        /
        (
            bw
            +
            EPS
        )
    ) ** 2
)


weight_sums = np.sum(
    mnd_weights_manual,
    axis=1
)


n_zero_mass = np.sum(
    weight_sums == 0
)


print(
    "\n[4] Kernel mass at bandwidth=500, lambda=0"
)

print(
    "Validation samples with total weight = 0:",
    n_zero_mass
)


zero_mass_indices = np.where(
    weight_sums == 0
)[0]


print(
    "Zero-mass indices:",
    zero_mass_indices.tolist()
)


# ------------------------------------------------------------
# 5. Weighted quantile
# ------------------------------------------------------------

q_mnd_manual = weighted_quantile(

    scores=calibration_scores,

    weights=mnd_weights_manual,

    q=0.9,

    self_weights=None,

    interpolate=False
)


inf_q_indices = np.where(
    np.isinf(
        q_mnd_manual
    )
)[0]


print(
    "\n[5] Infinite local q*"
)

print(
    "Number of +inf q*:",
    len(
        inf_q_indices
    )
)

print(
    "Indices:",
    inf_q_indices.tolist()
)


# ------------------------------------------------------------
# 6. Cross-check
# ------------------------------------------------------------

same_problem = np.array_equal(
    np.sort(
        zero_mass_indices
    ),
    np.sort(
        inf_q_indices
    )
)


print(
    "\nZero weight mass indices == +inf q* indices:",
    same_problem
)


if len(
    inf_q_indices
) > 0:

    print(
        "\nTheir waterfront values:"
    )

    print(
        X_val[
            inf_q_indices,
            waterfront_idx
        ]
    )


print(
    "\n✓ MND degeneracy diagnostic completed."
)

print(
    "✓ No MND implementation was modified."
)

print(
    "✓ Test was not used."
)

===== MND INFINITE-INTERVAL DIAGNOSTIC =====

[1] Calibration feature ranges


,feature,calib_min,calib_max,raw_range,MND_range_used
0,bathrooms,0.75,6.0,5.25,5.250000e+00
1,sqft_living,730.00,5490.0,4760.00,4.760000e+03
2,sqft_lot,834.00,19897.0,19063.00,1.906300e+04
3,grade,4.00,12.0,8.00,8.000000e+00
4,condition,2.00,5.0,3.00,3.000000e+00
5,waterfront,0.00,0.0,0.00,1.000000e-08
6,view,0.00,4.0,4.00,4.000000e+00
7,age,0.00,115.0,115.00,1.150000e+02



[2] MND distance distribution
Minimum: 0.004931018202801238
Median: 0.4434782608695652
Mean: 333333.8011764706
Maximum: 100000000.0

[3] Validation samples with MND distance > 1e6
Count: 1
Indices: [168]


,val_index,waterfront_value,max_MND_distance,dominant_max_feature,true_y
0,168,1.0,100000000.0,waterfront,6.079181



[4] Kernel mass at bandwidth=500, lambda=0
Validation samples with total weight = 0: 1
Zero-mass indices: [168]

[5] Infinite local q*
Number of +inf q*: 1
Indices: [168]

Zero weight mass indices == +inf q* indices: True

Their waterfront values:
[1.]

✓ MND degeneracy diagnostic completed.
✓ No MND implementation was modified.
✓ Test was not used.


In [36]:
# ============================================================
# Step 18 — GeoSIMCP-MND-filtered Validation tuning
#
# Sensitivity baseline:
#
# Same MND formula as GeoSIMCP,
# but Calibration zero-range features are removed first,
# consistent with the effective feature-space handling of EUC.
#
# IMPORTANT:
# This is NOT called official MND.
# It is MND-filtered / zero-range-filtered MND.
#
# Calibration + Validation only.
# Test remains untouched.
# ============================================================

import numpy as np
import pandas as pd

from geoconformal.geocp import (
    weighted_quantile
)


print(
    "===== GEOSIMCP-MND-FILTERED VALIDATION GRID SEARCH ====="
)


EPS = 1e-8


# ============================================================
# 1. Active feature space
# ============================================================

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


calib_feature_min = np.min(
    X_calib,
    axis=0
)


calib_feature_max = np.max(
    X_calib,
    axis=0
)


calib_feature_range = (
    calib_feature_max
    -
    calib_feature_min
)


active_mnd_mask = (
    calib_feature_range > 1e-8
)


active_mnd_indices = np.where(
    active_mnd_mask
)[0]


active_mnd_features = [
    feature_order[i]
    for i in active_mnd_indices
]


dropped_mnd_features = [
    feature_order[i]
    for i in range(
        len(feature_order)
    )
    if not active_mnd_mask[i]
]


print(
    "\n[1] MND-filtered feature space"
)

print(
    "Active features:",
    active_mnd_features
)

print(
    "Dropped features:",
    dropped_mnd_features
)


assert dropped_mnd_features == [
    "waterfront"
]


# ============================================================
# 2. Compute filtered MND distance
# ============================================================

X_calib_mnd = (
    X_calib[
        :,
        active_mnd_indices
    ]
)


X_val_mnd = (
    X_val[
        :,
        active_mnd_indices
    ]
)


mnd_min = np.min(
    X_calib_mnd,
    axis=0
)


mnd_max = np.max(
    X_calib_mnd,
    axis=0
)


mnd_range = (
    mnd_max
    -
    mnd_min
)


assert np.all(
    mnd_range > 1e-8
)


normalized_difference = (

    np.abs(
        X_val_mnd[:, None, :]
        -
        X_calib_mnd[None, :, :]
    )

    /

    mnd_range[
        None,
        None,
        :
    ]
)


d_mnd_filtered_raw = np.max(
    normalized_difference,
    axis=2
)


# Preserve official numerical treatment
d_mnd_filtered = (
    d_mnd_filtered_raw
    +
    EPS
)


print(
    "\n[2] Filtered MND distance"
)

print(
    "Min   :",
    d_mnd_filtered_raw.min()
)

print(
    "Median:",
    np.median(
        d_mnd_filtered_raw
    )
)

print(
    "Mean  :",
    d_mnd_filtered_raw.mean()
)

print(
    "Max   :",
    d_mnd_filtered_raw.max()
)


assert np.isfinite(
    d_mnd_filtered_raw
).all()


# ============================================================
# 3. Shared Calibration scores
# ============================================================

calib_predictions = model.predict(
    X_calib_spatial
)


calibration_scores = np.abs(
    y_calib
    -
    calib_predictions
)


val_predictions = model.predict(
    X_val_spatial
)


# ============================================================
# 4. Same official parameter grid
# ============================================================

mndf_bandwidths = np.linspace(
    500,
    30000,
    15
)


mndf_lambdas = np.arange(
    0,
    1.05,
    0.1
)


rows = []


for bw in mndf_bandwidths:

    for lam in mndf_lambdas:

        # ----------------------------------------------------
        # Same original GeoSIMCP fusion
        # ----------------------------------------------------

        d_joint = np.sqrt(

            lam
            *
            d_geo ** 2

            +

            (1.0 - lam)
            *
            d_mnd_filtered ** 2
        )


        weights = np.exp(

            -0.5
            *
            (
                d_joint
                /
                (
                    bw
                    +
                    EPS
                )
            ) ** 2
        )


        q_local = weighted_quantile(

            scores=calibration_scores,

            weights=weights,

            q=0.9,

            self_weights=None,

            interpolate=False
        )


        lower = (
            val_predictions
            -
            q_local
        )


        upper = (
            val_predictions
            +
            q_local
        )


        coverage = float(
            np.mean(

                (y_val >= lower)

                &

                (y_val <= upper)
            )
        )


        mean_interval = float(
            np.mean(
                upper
                -
                lower
            )
        )


        score = gccp_interval_score(

            lower=lower,

            upper=upper,

            y_true=y_val,

            alpha=0.1
        )


        weight_sum = np.sum(
            weights,
            axis=1
        )


        weight_sq_sum = np.sum(
            weights ** 2,
            axis=1
        )


        n_eff = (

            weight_sum ** 2

            /

            np.where(
                weight_sq_sum > 0,
                weight_sq_sum,
                np.nan
            )
        )


        rows.append(
            {
                "bandwidth":
                    float(bw),

                "lambda":
                    float(lam),

                "coverage":
                    coverage,

                "mean_interval":
                    mean_interval,

                "interval_score":
                    score,

                "median_q":
                    float(
                        np.median(
                            q_local
                        )
                    ),

                "mean_q":
                    float(
                        np.mean(
                            q_local
                        )
                    ),

                "n_inf_q":
                    int(
                        np.isinf(
                            q_local
                        ).sum()
                    ),

                "median_n_eff":
                    float(
                        np.nanmedian(
                            n_eff
                        )
                    ),

                "mean_n_eff":
                    float(
                        np.nanmean(
                            n_eff
                        )
                    ),
            }
        )


df_mnd_filtered_val = pd.DataFrame(
    rows
)


# ============================================================
# 5. Sanity check: no infinite q*
# ============================================================

print(
    "\n[3] Infinite-q diagnostic"
)

print(
    "Maximum number of +inf q*:",
    df_mnd_filtered_val[
        "n_inf_q"
    ].max()
)


assert (
    df_mnd_filtered_val[
        "n_inf_q"
    ].max()
    ==
    0
)


# ============================================================
# 6. Same official selection rule
# ============================================================

valid_mnd_filtered = (
    df_mnd_filtered_val[
        df_mnd_filtered_val[
            "coverage"
        ]
        >=
        0.90
    ]
    .copy()
)


if len(
    valid_mnd_filtered
) > 0:

    selection_mode = (
        "Coverage >= 0.90 -> minimum Interval Score"
    )


    best_mnd_filtered = (
        valid_mnd_filtered.loc[

            valid_mnd_filtered[
                "interval_score"
            ]
            .idxmin()
        ]
    )


else:

    selection_mode = (
        "No Coverage >= 0.90 candidate "
        "-> minimum Interval Score over all candidates"
    )


    best_mnd_filtered = (
        df_mnd_filtered_val.loc[

            df_mnd_filtered_val[
                "interval_score"
            ]
            .idxmin()
        ]
    )


# ============================================================
# 7. Main result
# ============================================================

print(
    "\n===== MND-FILTERED VALIDATION RESULT ====="
)


print(
    "Selection mode:",
    selection_mode
)


print(
    "Candidates Coverage >= 0.90:",
    len(
        valid_mnd_filtered
    )
)


print(
    "\nBest bandwidth:",
    best_mnd_filtered[
        "bandwidth"
    ]
)


print(
    "Best lambda:",
    best_mnd_filtered[
        "lambda"
    ]
)


print(
    "\nCoverage:",
    best_mnd_filtered[
        "coverage"
    ]
)


print(
    "Mean Interval:",
    best_mnd_filtered[
        "mean_interval"
    ]
)


print(
    "Interval Score:",
    best_mnd_filtered[
        "interval_score"
    ]
)


print(
    "Median n_eff:",
    best_mnd_filtered[
        "median_n_eff"
    ]
)


# ============================================================
# 8. Top 15
# ============================================================

print(
    "\nTop 15 valid candidates:"
)


display(

    valid_mnd_filtered
    .sort_values(
        [
            "interval_score",
            "mean_interval"
        ]
    )
    .head(15)
    .reset_index(
        drop=True
    )
    .round(6)
)


# ============================================================
# 9. Best per lambda
# ============================================================

best_by_lambda_rows = []


for lam in mndf_lambdas:

    subset = (
        df_mnd_filtered_val[
            np.isclose(
                df_mnd_filtered_val[
                    "lambda"
                ],
                lam
            )
        ]
        .copy()
    )


    valid_subset = (
        subset[
            subset[
                "coverage"
            ]
            >=
            0.90
        ]
    )


    if len(
        valid_subset
    ) > 0:

        row = valid_subset.loc[

            valid_subset[
                "interval_score"
            ]
            .idxmin()
        ]

        branch = (
            "coverage>=0.90"
        )

    else:

        row = subset.loc[

            subset[
                "interval_score"
            ]
            .idxmin()
        ]

        branch = (
            "no-valid-coverage"
        )


    best_by_lambda_rows.append(
        {
            "lambda":
                float(lam),

            "bandwidth":
                float(
                    row[
                        "bandwidth"
                    ]
                ),

            "coverage":
                float(
                    row[
                        "coverage"
                    ]
                ),

            "mean_interval":
                float(
                    row[
                        "mean_interval"
                    ]
                ),

            "interval_score":
                float(
                    row[
                        "interval_score"
                    ]
                ),

            "median_n_eff":
                float(
                    row[
                        "median_n_eff"
                    ]
                ),

            "selection_branch":
                branch,
        }
    )


mnd_filtered_best_by_lambda = pd.DataFrame(
    best_by_lambda_rows
)


print(
    "\n===== BEST MND-FILTERED BY LAMBDA ====="
)


display(
    mnd_filtered_best_by_lambda.round(
        6
    )
)


# ============================================================
# 10. Save sensitivity baseline
# ============================================================

MND_FILTERED_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "geosimcp_mnd_filtered_validation_v1.csv"
)


MND_FILTERED_LAMBDA_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "geosimcp_mnd_filtered_best_by_lambda_v1.csv"
)


df_mnd_filtered_val.to_csv(
    MND_FILTERED_PATH,
    index=False
)


mnd_filtered_best_by_lambda.to_csv(
    MND_FILTERED_LAMBDA_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    MND_FILTERED_PATH
)

print(
    MND_FILTERED_LAMBDA_PATH
)


print(
    "\n✓ MND-filtered sensitivity baseline completed."
)

print(
    "✓ Official MND source code was NOT modified."
)

print(
    "✓ Zero-range feature filtering was explicit."
)

print(
    "✓ Test was NOT used."
)

===== GEOSIMCP-MND-FILTERED VALIDATION GRID SEARCH =====

[1] MND-filtered feature space
Active features: ['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'view', 'age']
Dropped features: ['waterfront']

[2] Filtered MND distance
Min   : 0.004931018202801238
Median: 0.4434782608695652
Mean  : 0.47103305746367835
Max   : 1.3333333333333333

[3] Infinite-q diagnostic
Maximum number of +inf q*: 0

===== MND-FILTERED VALIDATION RESULT =====
Selection mode: Coverage >= 0.90 -> minimum Interval Score
Candidates Coverage >= 0.90: 153

Best bandwidth: 2607.1428571428573
Best lambda: 0.0

Coverage: 0.92
Mean Interval: 0.2541355109619765
Interval Score: 0.3210766044099667
Median n_eff: 299.99999999999994

Top 15 valid candidates:


,bandwidth,lambda,coverage,mean_interval,interval_score,median_q,mean_q,n_inf_q,median_n_eff,mean_n_eff
0,2607.142857,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
1,4714.285714,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
2,6821.428571,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
3,8928.571429,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
4,11035.714286,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
5,13142.857143,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
6,15250.000000,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
7,17357.142857,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
8,19464.285714,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0
9,21571.428571,0.0,0.92,0.254136,0.321077,0.127068,0.127068,0,300.0,300.0



===== BEST MND-FILTERED BY LAMBDA =====


,lambda,bandwidth,coverage,mean_interval,interval_score,median_n_eff,selection_branch
0,0.0,2607.142857,0.920000,0.254136,0.321077,300.000000,coverage>=0.90
1,0.1,11035.714286,0.923333,0.257107,0.321989,291.403746,coverage>=0.90
2,0.2,15250.000000,0.923333,0.257133,0.322015,290.721582,coverage>=0.90
3,0.3,27892.857143,0.923333,0.257244,0.321989,297.738655,coverage>=0.90
4,0.4,21571.428571,0.923333,0.257133,0.322015,290.728184,coverage>=0.90
5,0.5,25785.714286,0.923333,0.257121,0.322003,292.621350,coverage>=0.90
6,0.6,27892.857143,0.923333,0.257121,0.322003,292.297752,coverage>=0.90
7,0.7,30000.000000,0.923333,0.257121,0.322003,292.181803,coverage>=0.90
8,0.8,30000.000000,0.923333,0.257131,0.322150,290.192121,coverage>=0.90
9,0.9,30000.000000,0.923333,0.257267,0.322200,288.087596,coverage>=0.90



===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geosimcp_mnd_filtered_validation_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geosimcp_mnd_filtered_best_by_lambda_v1.csv

✓ MND-filtered sensitivity baseline completed.
✓ Official MND source code was NOT modified.
✓ Zero-range feature filtering was explicit.
✓ Test was NOT used.


In [37]:
# ============================================================
# Step 19 — Freeze author-aligned Validation baseline table
#
# Methods:
# CP
# GeoCP
# GeoSIMCP-EUC
# GeoSIMCP-MND-original
# GeoSIMCP-MND-filtered
# GCCP-original
#
# Validation only.
# Test remains untouched.
# ============================================================

import numpy as np
import pandas as pd

from geoconformal.geocp import weighted_quantile
from geoconformal.geocp.weights import joint_geo_feature_weights


print(
    "===== AUTHOR-ALIGNED VALIDATION BASELINE SUMMARY ====="
)


# ------------------------------------------------------------
# 1. Shared Calibration residuals / Validation predictions
# ------------------------------------------------------------

calib_predictions = model.predict(
    X_calib_spatial
)

calibration_scores = np.abs(
    y_calib - calib_predictions
)

val_predictions = model.predict(
    X_val_spatial
)


# ============================================================
# 2. CP
# ============================================================

q_cp = weighted_quantile(

    scores=calibration_scores,

    weights=np.ones(
        (
            1,
            len(calibration_scores)
        )
    ),

    q=0.9,

    self_weights=None,

    interpolate=False
)[0]


cp_lower = val_predictions - q_cp
cp_upper = val_predictions + q_cp


cp_coverage = float(
    np.mean(
        (y_val >= cp_lower)
        &
        (y_val <= cp_upper)
    )
)


cp_width = float(
    np.mean(
        cp_upper - cp_lower
    )
)


cp_score = float(
    interval_score(
        cp_upper,
        cp_lower,
        y_val,
        alpha=0.1
    )
)


# ============================================================
# 3. Helper: effective sample size for official joint weights
# ============================================================

official_input_val = np.hstack(
    [
        coords_val,
        X_val
    ]
)


def official_median_neff(
    bandwidth,
    lambda_weight,
    distance_metric="euclidean"
):

    weight_fn = joint_geo_feature_weights(

        coord_calib=coords_calib,

        feat_calib=X_calib,

        bandwidth=float(
            bandwidth
        ),

        lambda_weight=float(
            lambda_weight
        ),

        distance_metric=distance_metric,

        standardize=True
    )


    weights = weight_fn(
        official_input_val,
        None
    )


    sum_w = np.sum(
        weights,
        axis=1
    )

    sum_w2 = np.sum(
        weights ** 2,
        axis=1
    )


    neff = (
        sum_w ** 2
        /
        np.where(
            sum_w2 > 0,
            sum_w2,
            np.nan
        )
    )


    return float(
        np.nanmedian(
            neff
        )
    )


# ============================================================
# 4. GeoCP
#
# Use official GeoCP's own 30-bandwidth Validation tuning.
# ============================================================

geo_bw = float(
    best_geocp[
        "bandwidth"
    ]
)


geo_neff = official_median_neff(
    bandwidth=geo_bw,
    lambda_weight=1.0,
    distance_metric="euclidean"
)


# ============================================================
# 5. GeoSIMCP-EUC
# ============================================================

euc_bw = float(
    best_simcp_euc[
        "bandwidth"
    ]
)

euc_lambda = float(
    best_simcp_euc[
        "lambda"
    ]
)


euc_neff = official_median_neff(
    bandwidth=euc_bw,
    lambda_weight=euc_lambda,
    distance_metric="euclidean"
)


# ============================================================
# 6. Build summary
# ============================================================

summary_rows = [

    {
        "method":
            "CP",

        "variant":
            "global",

        "bandwidth":
            np.nan,

        "lambda":
            np.nan,

        "coverage":
            cp_coverage,

        "mean_interval":
            cp_width,

        "interval_score":
            cp_score,

        "median_n_eff":
            300.0,

        "status":
            "valid"
    },


    {
        "method":
            "GeoCP",

        "variant":
            "official",

        "bandwidth":
            geo_bw,

        "lambda":
            1.0,

        "coverage":
            float(
                best_geocp[
                    "coverage"
                ]
            ),

        "mean_interval":
            float(
                best_geocp[
                    "mean_interval"
                ]
            ),

        "interval_score":
            float(
                best_geocp[
                    "interval_score"
                ]
            ),

        "median_n_eff":
            geo_neff,

        "status":
            "valid"
    },


    {
        "method":
            "GeoSIMCP-EUC",

        "variant":
            "official",

        "bandwidth":
            euc_bw,

        "lambda":
            euc_lambda,

        "coverage":
            float(
                best_simcp_euc[
                    "coverage"
                ]
            ),

        "mean_interval":
            float(
                best_simcp_euc[
                    "mean_interval"
                ]
            ),

        "interval_score":
            float(
                best_simcp_euc[
                    "interval_score"
                ]
            ),

        "median_n_eff":
            euc_neff,

        "status":
            "valid"
    },


    {
        "method":
            "GeoSIMCP-MND",

        "variant":
            "official-original",

        "bandwidth":
            float(
                best_mnd[
                    "bandwidth"
                ]
            ),

        "lambda":
            float(
                best_mnd[
                    "lambda"
                ]
            ),

        "coverage":
            float(
                best_mnd[
                    "coverage"
                ]
            ),

        "mean_interval":
            float(
                best_mnd[
                    "mean_interval"
                ]
            ),

        "interval_score":
            float(
                best_mnd[
                    "interval_score"
                ]
            ),

        "median_n_eff":
            float(
                best_mnd[
                    "median_n_eff"
                ]
            ),

        "status":
            (
                "degenerate feature branch: "
                "lambda<1 produced +inf interval(s); "
                "selected lambda=1 = GeoCP branch"
            )
    },


    {
        "method":
            "GeoSIMCP-MND",

        "variant":
            "zero-range-filtered sensitivity",

        "bandwidth":
            float(
                best_mnd_filtered[
                    "bandwidth"
                ]
            ),

        "lambda":
            float(
                best_mnd_filtered[
                    "lambda"
                ]
            ),

        "coverage":
            float(
                best_mnd_filtered[
                    "coverage"
                ]
            ),

        "mean_interval":
            float(
                best_mnd_filtered[
                    "mean_interval"
                ]
            ),

        "interval_score":
            float(
                best_mnd_filtered[
                    "interval_score"
                ]
            ),

        "median_n_eff":
            float(
                best_mnd_filtered[
                    "median_n_eff"
                ]
            ),

        "status":
            "sensitivity baseline"
    },


    {
        "method":
            "GCCP",

        "variant":
            "original GeoSIMCP fusion",

        "bandwidth":
            float(
                best_gccp[
                    "bandwidth"
                ]
            ),

        "lambda":
            float(
                best_gccp[
                    "lambda"
                ]
            ),

        "coverage":
            float(
                best_gccp[
                    "coverage"
                ]
            ),

        "mean_interval":
            float(
                best_gccp[
                    "mean_interval"
                ]
            ),

        "interval_score":
            float(
                best_gccp[
                    "interval_score"
                ]
            ),

        "median_n_eff":
            float(
                best_gccp[
                    "median_n_eff"
                ]
            ),

        "status":
            "valid"
    },
]


validation_summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# 7. Relative differences versus CP
# ============================================================

validation_summary[
    "coverage_change_vs_CP_pp"
] = (

    validation_summary[
        "coverage"
    ]

    -
    cp_coverage
) * 100.0


validation_summary[
    "width_change_vs_CP_pct"
] = (

    (
        validation_summary[
            "mean_interval"
        ]

        /
        cp_width
    )

    -
    1.0
) * 100.0


validation_summary[
    "score_change_vs_CP_pct"
] = (

    (
        validation_summary[
            "interval_score"
        ]

        /
        cp_score
    )

    -
    1.0
) * 100.0


print(
    "\n[1] Validation summary"
)


display(
    validation_summary.round(
        6
    )
)


# ============================================================
# 8. Focused valid-method comparison
# ============================================================

focused_methods = validation_summary[
    validation_summary[
        "status"
    ].isin(
        [
            "valid",
            "sensitivity baseline"
        ]
    )
].copy()


print(
    "\n[2] Finite / usable Validation methods"
)


display(

    focused_methods[
        [
            "method",
            "variant",
            "bandwidth",
            "lambda",
            "coverage",
            "mean_interval",
            "interval_score",
            "median_n_eff",
        ]
    ]
    .sort_values(
        "interval_score"
    )
    .reset_index(
        drop=True
    )
    .round(6)
)


# ============================================================
# 9. Check whether GCCP/MND-filtered collapse to CP
# ============================================================

gccp_row = validation_summary[
    validation_summary[
        "method"
    ].eq(
        "GCCP"
    )
].iloc[0]


mndf_row = validation_summary[
    validation_summary[
        "variant"
    ].eq(
        "zero-range-filtered sensitivity"
    )
].iloc[0]


print(
    "\n[3] Collapse-to-global checks"
)


print(
    "GCCP |width - CP|:",
    abs(
        gccp_row[
            "mean_interval"
        ]
        -
        cp_width
    )
)


print(
    "GCCP |score - CP|:",
    abs(
        gccp_row[
            "interval_score"
        ]
        -
        cp_score
    )
)


print(
    "MND-filtered |width - CP|:",
    abs(
        mndf_row[
            "mean_interval"
        ]
        -
        cp_width
    )
)


print(
    "MND-filtered |score - CP|:",
    abs(
        mndf_row[
            "interval_score"
        ]
        -
        cp_score
    )
)


# ============================================================
# 10. Save frozen Validation summary
# ============================================================

VALIDATION_SUMMARY_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "seattle_author_aligned_validation_summary_v1.csv"
)


validation_summary.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    VALIDATION_SUMMARY_PATH
)


print(
    "\n✓ Validation baseline table frozen."
)

print(
    "✓ No Test results were used."
)

===== AUTHOR-ALIGNED VALIDATION BASELINE SUMMARY =====

[1] Validation summary


,method,variant,bandwidth,lambda,coverage,mean_interval,interval_score,median_n_eff,status,coverage_change_vs_CP_pp,width_change_vs_CP_pct,score_change_vs_CP_pct
0,CP,global,NaN,NaN,0.920000,0.254136,0.321077,300.000000,valid,0.000000,0.000000,0.000000
1,GeoCP,official,28982.758621,1.0,0.923333,0.257507,0.322355,284.440568,valid,0.333333,1.326732,0.398087
2,GeoSIMCP-EUC,official,500.000000,0.0,0.920000,0.254190,0.320995,300.000000,valid,0.000000,0.021457,-0.025475
3,GeoSIMCP-MND,official-original,30000.000000,1.0,0.923333,0.257427,0.322360,285.985445,degenerate feature branch: lambda<1 produced +...,0.333333,1.295150,0.399675
4,GeoSIMCP-MND,zero-range-filtered sensitivity,2607.142857,0.0,0.920000,0.254136,0.321077,300.000000,sensitivity baseline,0.000000,0.000000,0.000000
5,GCCP,original GeoSIMCP fusion,6821.428571,0.0,0.920000,0.254136,0.321077,300.000000,valid,0.000000,0.000000,0.000000



[2] Finite / usable Validation methods


,method,variant,bandwidth,lambda,coverage,mean_interval,interval_score,median_n_eff
0,GeoSIMCP-EUC,official,500.000000,0.0,0.920000,0.254190,0.320995,300.000000
1,CP,global,NaN,NaN,0.920000,0.254136,0.321077,300.000000
2,GeoSIMCP-MND,zero-range-filtered sensitivity,2607.142857,0.0,0.920000,0.254136,0.321077,300.000000
3,GCCP,original GeoSIMCP fusion,6821.428571,0.0,0.920000,0.254136,0.321077,300.000000
4,GeoCP,official,28982.758621,1.0,0.923333,0.257507,0.322355,284.440568



[3] Collapse-to-global checks
GCCP |width - CP|: 0.0
GCCP |score - CP|: 0.0
MND-filtered |width - CP|: 0.0
MND-filtered |score - CP|: 0.0

===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_author_aligned_validation_summary_v1.csv

✓ Validation baseline table frozen.
✓ No Test results were used.


In [38]:
# ============================================================
# Step 20A — Scale-aware bandwidth design for additive models
#
# 目的：
# 1. 只利用 Calibration 数据确定距离尺度
# 2. 分别为 geographic / EUC / causal distance
#    建立尺度匹配的 bandwidth candidates
# 3. 检查这些 bandwidth 是否产生有意义的
#    local -> global effective sample size 变化
#
# 不使用 Validation y
# 不使用 Test
# 不进行模型选择
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances


print(
    "===== SCALE-AWARE BANDWIDTH DESIGN ====="
)


EPS = 1e-8


# ============================================================
# 1. Active feature space
#
# 与前面 GeoSIMCP / GCCP 保持一致：
# Calibration 零方差 waterfront 被排除。
# ============================================================

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]


calib_std_all = np.std(
    X_calib,
    axis=0
)


active_mask = (
    calib_std_all > 1e-8
)


active_indices = np.where(
    active_mask
)[0]


active_features = [
    feature_order[i]
    for i in active_indices
]


print(
    "\n[1] Active feature space"
)

print(
    "Active:",
    active_features
)

print(
    "Dimension:",
    len(active_features)
)


assert len(active_features) == 7


# ============================================================
# 2. Calibration standardization for EUC / causal distances
# ============================================================

X_calib_active = (
    X_calib[
        :,
        active_indices
    ]
)


cal_mean = np.mean(
    X_calib_active,
    axis=0
)


cal_std = np.std(
    X_calib_active,
    axis=0
)


assert np.all(
    cal_std > 1e-8
)


Z_calib_scale = (
    X_calib_active
    -
    cal_mean
) / cal_std


X_val_active = (
    X_val[
        :,
        active_indices
    ]
)


Z_val_scale = (
    X_val_active
    -
    cal_mean
) / cal_std


# ============================================================
# 3. Calibration-Calibraton geographic distance
# ============================================================

d_geo_calcal = pairwise_distances(
    coords_calib,
    coords_calib,
    metric="euclidean"
)


# ============================================================
# 4. Calibration-Calibraton EUC distance
# ============================================================

d_euc_calcal = pairwise_distances(
    Z_calib_scale,
    Z_calib_scale,
    metric="euclidean"
)


# ============================================================
# 5. Calibration-Calibraton causal distance
#
# Use frozen Train-GCCM causal weights.
# Restrict + renormalize to same active 7 dimensions.
# ============================================================

CAUSAL_WEIGHT_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
    / "seattle_gccm_train_logprice_causal_weights_v1.csv"
)


cw = pd.read_csv(
    CAUSAL_WEIGHT_PATH
)


cw = (
    cw
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)


all_causal_weights = (
    cw[
        "causal_weight"
    ]
    .to_numpy(
        dtype=float
    )
)


active_causal_weights = (
    all_causal_weights[
        active_indices
    ]
)


active_causal_weights = (

    active_causal_weights

    /

    active_causal_weights.sum()
)


p_eff = len(
    active_causal_weights
)


causal_coefficients = (
    p_eff
    *
    active_causal_weights
)


calcal_diff = (

    Z_calib_scale[:, None, :]

    -

    Z_calib_scale[None, :, :]
)


d_causal_calcal = np.sqrt(

    np.sum(

        causal_coefficients[
            None,
            None,
            :
        ]

        *

        calcal_diff ** 2,

        axis=2
    )
)


# ============================================================
# 6. Remove diagonal zeros when estimating typical scale
# ============================================================

upper_idx = np.triu_indices(
    len(X_calib),
    k=1
)


geo_nonzero = (
    d_geo_calcal[
        upper_idx
    ]
)


euc_nonzero = (
    d_euc_calcal[
        upper_idx
    ]
)


causal_nonzero = (
    d_causal_calcal[
        upper_idx
    ]
)


geo_scale = float(
    np.median(
        geo_nonzero
    )
)


euc_scale = float(
    np.median(
        euc_nonzero
    )
)


causal_scale = float(
    np.median(
        causal_nonzero
    )
)


print(
    "\n[2] Calibration-only median distance scales"
)

print(
    "Geographic median:",
    geo_scale
)

print(
    "EUC median       :",
    euc_scale
)

print(
    "Causal median    :",
    causal_scale
)


# ============================================================
# 7. Dimensionless bandwidth multipliers
# ============================================================

BANDWIDTH_MULTIPLIERS = np.array(
    [
        0.25,
        0.50,
        1.00,
        2.00,
        4.00,
    ],
    dtype=float
)


geo_bandwidth_grid_v2 = (
    geo_scale
    *
    BANDWIDTH_MULTIPLIERS
)


euc_bandwidth_grid_v2 = (
    euc_scale
    *
    BANDWIDTH_MULTIPLIERS
)


causal_bandwidth_grid_v2 = (
    causal_scale
    *
    BANDWIDTH_MULTIPLIERS
)


bandwidth_grid_table = pd.DataFrame(
    {
        "multiplier":
            BANDWIDTH_MULTIPLIERS,

        "h_geo":
            geo_bandwidth_grid_v2,

        "h_EUC":
            euc_bandwidth_grid_v2,

        "h_causal":
            causal_bandwidth_grid_v2,
    }
)


print(
    "\n[3] Scale-aware bandwidth grids"
)

display(
    bandwidth_grid_table.round(
        6
    )
)


# ============================================================
# 8. Validation distances
#
# Only X / coordinates are used here.
# No Validation y is used.
# ============================================================

d_geo_val_v2 = pairwise_distances(
    coords_val,
    coords_calib,
    metric="euclidean"
)


d_euc_val_v2 = pairwise_distances(
    Z_val_scale,
    Z_calib_scale,
    metric="euclidean"
)


valcal_diff = (

    Z_val_scale[:, None, :]

    -

    Z_calib_scale[None, :, :]
)


d_causal_val_v2 = np.sqrt(

    np.sum(

        causal_coefficients[
            None,
            None,
            :
        ]

        *

        valcal_diff ** 2,

        axis=2
    )
)


# ============================================================
# 9. Effective sample size helper
# ============================================================

def kernel_neff(
    distance_matrix,
    bandwidth
):

    weights = np.exp(

        -0.5
        *
        (
            distance_matrix
            /
            (
                bandwidth
                +
                EPS
            )
        ) ** 2
    )


    sum_w = np.sum(
        weights,
        axis=1
    )


    sum_w2 = np.sum(
        weights ** 2,
        axis=1
    )


    neff = (

        sum_w ** 2

        /

        np.where(
            sum_w2 > 0,
            sum_w2,
            np.nan
        )
    )


    return neff


# ============================================================
# 10. Diagnose locality produced by each bandwidth grid
# ============================================================

locality_rows = []


for distance_name, distance_matrix, bandwidth_grid in [

    (
        "Geographic",
        d_geo_val_v2,
        geo_bandwidth_grid_v2
    ),

    (
        "EUC",
        d_euc_val_v2,
        euc_bandwidth_grid_v2
    ),

    (
        "Causal",
        d_causal_val_v2,
        causal_bandwidth_grid_v2
    ),
]:

    for multiplier, bandwidth in zip(

        BANDWIDTH_MULTIPLIERS,

        bandwidth_grid
    ):

        neff = kernel_neff(
            distance_matrix,
            bandwidth
        )


        locality_rows.append(
            {
                "distance":
                    distance_name,

                "multiplier":
                    float(
                        multiplier
                    ),

                "bandwidth":
                    float(
                        bandwidth
                    ),

                "median_n_eff":
                    float(
                        np.nanmedian(
                            neff
                        )
                    ),

                "p25_n_eff":
                    float(
                        np.nanquantile(
                            neff,
                            0.25
                        )
                    ),

                "p75_n_eff":
                    float(
                        np.nanquantile(
                            neff,
                            0.75
                        )
                    ),

                "min_n_eff":
                    float(
                        np.nanmin(
                            neff
                        )
                    ),

                "max_n_eff":
                    float(
                        np.nanmax(
                            neff
                        )
                    ),
            }
        )


locality_table = pd.DataFrame(
    locality_rows
)


print(
    "\n[4] Locality diagnostic"
)

display(
    locality_table.round(
        6
    )
)


# ============================================================
# 11. Compact pivot
# ============================================================

print(
    "\n[5] Median effective sample size"
)


median_neff_pivot = (
    locality_table
    .pivot(
        index="multiplier",
        columns="distance",
        values="median_n_eff"
    )
)


display(
    median_neff_pivot.round(
        3
    )
)


print(
    "\n✓ Scale-aware bandwidth design completed."
)

print(
    "✓ Distance scales were determined from Calibration only."
)

print(
    "✓ No Validation y was used."
)

print(
    "✓ Test was not used."
)

===== SCALE-AWARE BANDWIDTH DESIGN =====

[1] Active feature space
Active: ['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'view', 'age']
Dimension: 7

[2] Calibration-only median distance scales
Geographic median: 17761.92265024405
EUC median       : 3.3655413398211977
Causal median    : 3.027983720113191

[3] Scale-aware bandwidth grids


,multiplier,h_geo,h_EUC,h_causal
0,0.25,4440.480663,0.841385,0.756996
1,0.50,8880.961325,1.682771,1.513992
2,1.00,17761.922650,3.365541,3.027984
3,2.00,35523.845300,6.731083,6.055967
4,4.00,71047.690601,13.462165,12.111935



[4] Locality diagnostic


,distance,multiplier,bandwidth,median_n_eff,p25_n_eff,p75_n_eff,min_n_eff,max_n_eff
0,Geographic,0.25,4440.480663,56.186674,31.151694,70.075391,9.557736,86.232948
1,Geographic,0.50,8880.961325,136.275351,111.259540,156.275137,34.169851,203.181847
2,Geographic,1.00,17761.922650,245.351772,224.282826,263.461672,131.415056,278.373450
3,Geographic,2.00,35523.845300,291.921499,287.764532,296.471720,269.625801,298.131918
4,Geographic,4.00,71047.690601,299.354347,299.023116,299.749739,297.773095,299.874025
5,EUC,0.25,0.841385,40.417442,21.784959,51.287318,1.061076,75.181161
6,EUC,0.50,1.682771,165.692955,132.458279,195.571262,9.519301,228.502919
7,EUC,1.00,3.365541,274.681300,267.285596,281.800871,169.482016,289.595590
8,EUC,2.00,6.731083,297.567323,296.715035,298.209569,287.804570,298.897433
9,EUC,4.00,13.462165,299.818949,299.747406,299.869608,299.220053,299.920711



[5] Median effective sample size


distance,Causal,EUC,Geographic
multiplier,,,
0.25,42.933,40.417,56.187
0.50,165.749,165.693,136.275
1.00,265.543,274.681,245.352
2.00,295.251,297.567,291.921
4.00,299.589,299.819,299.354



✓ Scale-aware bandwidth design completed.
✓ Distance scales were determined from Calibration only.
✓ No Validation y was used.
✓ Test was not used.


In [39]:
# ============================================================
# Step 20B — Parallel additive-fusion Validation experiment
#
# Methods:
#
# 1. Additive-EUC
# 2. Additive-GCCP
#
# SAME:
# - Calibration residuals
# - geographic kernel
# - bandwidth multiplier philosophy
# - 5 geo bandwidth candidates
# - 5 feature bandwidth candidates
# - lambda grid
# - weighted quantile
# - Validation selection rule
#
# ONLY difference:
#
# Additive-EUC:
#     feature distance = ordinary EUC
#
# Additive-GCCP:
#     feature distance = causal-weighted EUC
#
# Test is NOT used.
# ============================================================

import numpy as np
import pandas as pd

from geoconformal.geocp import (
    weighted_quantile
)


print(
    "===== ADDITIVE-EUC vs ADDITIVE-GCCP VALIDATION ====="
)


EPS = 1e-8


# ============================================================
# 1. Frozen grids from Step 20A
# ============================================================

lambda_grid_v2 = np.arange(
    0,
    1.05,
    0.1
)


print(
    "\nGeo bandwidth candidates:"
)

print(
    geo_bandwidth_grid_v2
)


print(
    "\nEUC bandwidth candidates:"
)

print(
    euc_bandwidth_grid_v2
)


print(
    "\nCausal bandwidth candidates:"
)

print(
    causal_bandwidth_grid_v2
)


print(
    "\nLambda candidates:"
)

print(
    lambda_grid_v2
)


print(
    "\nConfigurations per method:",
    len(geo_bandwidth_grid_v2)
    *
    5
    *
    len(lambda_grid_v2)
)


# ============================================================
# 2. Shared Calibration residuals
# ============================================================

calib_predictions = model.predict(
    X_calib_spatial
)


calibration_scores = np.abs(
    y_calib
    -
    calib_predictions
)


val_predictions = model.predict(
    X_val_spatial
)


# ============================================================
# 3. Helper functions
# ============================================================

def gaussian_kernel(
    distance,
    bandwidth
):

    return np.exp(

        -0.5
        *
        (
            distance
            /
            (
                bandwidth
                +
                EPS
            )
        ) ** 2
    )


def effective_sample_size(
    weights
):

    sum_w = np.sum(
        weights,
        axis=1
    )

    sum_w2 = np.sum(
        weights ** 2,
        axis=1
    )

    return (

        sum_w ** 2

        /

        np.where(
            sum_w2 > 0,
            sum_w2,
            np.nan
        )
    )


def evaluate_additive_configuration(
    K_geo,
    K_feature,
    lam
):

    # --------------------------------------------------------
    # Additive fusion
    # --------------------------------------------------------

    weights = (

        lam
        *
        K_geo

        +

        (1.0 - lam)
        *
        K_feature
    )


    # --------------------------------------------------------
    # Weighted conformal quantile
    # --------------------------------------------------------

    q_local = weighted_quantile(

        scores=calibration_scores,

        weights=weights,

        q=0.9,

        self_weights=None,

        interpolate=False
    )


    lower = (
        val_predictions
        -
        q_local
    )


    upper = (
        val_predictions
        +
        q_local
    )


    coverage = float(
        np.mean(

            (y_val >= lower)

            &

            (y_val <= upper)
        )
    )


    mean_interval = float(
        np.mean(
            upper
            -
            lower
        )
    )


    score = gccp_interval_score(

        lower=lower,

        upper=upper,

        y_true=y_val,

        alpha=0.1
    )


    n_eff = effective_sample_size(
        weights
    )


    return {

        "coverage":
            coverage,

        "mean_interval":
            mean_interval,

        "interval_score":
            score,

        "median_q":
            float(
                np.median(
                    q_local
                )
            ),

        "mean_q":
            float(
                np.mean(
                    q_local
                )
            ),

        "median_n_eff":
            float(
                np.nanmedian(
                    n_eff
                )
            ),

        "mean_n_eff":
            float(
                np.nanmean(
                    n_eff
                )
            ),
    }


# ============================================================
# 4. Pre-compute geographic kernels
# ============================================================

geo_kernels = {}


for geo_multiplier, h_geo in zip(

    BANDWIDTH_MULTIPLIERS,

    geo_bandwidth_grid_v2
):

    geo_kernels[
        float(
            geo_multiplier
        )
    ] = gaussian_kernel(

        d_geo_val_v2,

        h_geo
    )


# ============================================================
# 5. Pre-compute EUC kernels
# ============================================================

euc_kernels = {}


for feature_multiplier, h_euc in zip(

    BANDWIDTH_MULTIPLIERS,

    euc_bandwidth_grid_v2
):

    euc_kernels[
        float(
            feature_multiplier
        )
    ] = gaussian_kernel(

        d_euc_val_v2,

        h_euc
    )


# ============================================================
# 6. Pre-compute causal kernels
# ============================================================

causal_kernels = {}


for feature_multiplier, h_causal in zip(

    BANDWIDTH_MULTIPLIERS,

    causal_bandwidth_grid_v2
):

    causal_kernels[
        float(
            feature_multiplier
        )
    ] = gaussian_kernel(

        d_causal_val_v2,

        h_causal
    )


# ============================================================
# 7. Parallel experiment function
# ============================================================

def run_additive_grid(
    method_name,
    feature_kernel_dict,
    feature_bandwidth_grid
):

    rows = []


    for geo_idx, geo_multiplier in enumerate(
        BANDWIDTH_MULTIPLIERS
    ):

        h_geo = float(
            geo_bandwidth_grid_v2[
                geo_idx
            ]
        )


        K_geo = (
            geo_kernels[
                float(
                    geo_multiplier
                )
            ]
        )


        for feat_idx, feature_multiplier in enumerate(
            BANDWIDTH_MULTIPLIERS
        ):

            h_feature = float(
                feature_bandwidth_grid[
                    feat_idx
                ]
            )


            K_feature = (
                feature_kernel_dict[
                    float(
                        feature_multiplier
                    )
                ]
            )


            for lam in lambda_grid_v2:

                metrics = (
                    evaluate_additive_configuration(
                        K_geo,
                        K_feature,
                        float(
                            lam
                        )
                    )
                )


                rows.append(
                    {
                        "method":
                            method_name,

                        "geo_multiplier":
                            float(
                                geo_multiplier
                            ),

                        "feature_multiplier":
                            float(
                                feature_multiplier
                            ),

                        "h_geo":
                            h_geo,

                        "h_feature":
                            h_feature,

                        "lambda":
                            float(
                                lam
                            ),

                        **metrics,
                    }
                )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 8. Run Additive-EUC
# ============================================================

print(
    "\n[1] Running Additive-EUC..."
)


df_additive_euc = run_additive_grid(

    method_name="Additive-EUC",

    feature_kernel_dict=euc_kernels,

    feature_bandwidth_grid=euc_bandwidth_grid_v2
)


print(
    "Configurations:",
    len(
        df_additive_euc
    )
)


# ============================================================
# 9. Run Additive-GCCP
# ============================================================

print(
    "\n[2] Running Additive-GCCP..."
)


df_additive_gccp = run_additive_grid(

    method_name="Additive-GCCP",

    feature_kernel_dict=causal_kernels,

    feature_bandwidth_grid=causal_bandwidth_grid_v2
)


print(
    "Configurations:",
    len(
        df_additive_gccp
    )
)


assert len(
    df_additive_euc
) == 275


assert len(
    df_additive_gccp
) == 275


# ============================================================
# 10. SAME selection rule for both methods
# ============================================================

def select_best_validation(
    df
):

    valid = df[
        df[
            "coverage"
        ]
        >=
        0.90
    ].copy()


    if len(
        valid
    ) > 0:

        best = valid.loc[

            valid[
                "interval_score"
            ]
            .idxmin()
        ]


        mode = (
            "Coverage >= 0.90 -> minimum Interval Score"
        )

    else:

        best = df.loc[

            df[
                "interval_score"
            ]
            .idxmin()
        ]


        mode = (
            "No Coverage >= 0.90 "
            "-> minimum Interval Score"
        )


    return (
        best,
        valid,
        mode
    )


best_add_euc, valid_add_euc, mode_add_euc = (
    select_best_validation(
        df_additive_euc
    )
)


best_add_gccp, valid_add_gccp, mode_add_gccp = (
    select_best_validation(
        df_additive_gccp
    )
)


# ============================================================
# 11. Main comparison
# ============================================================

main_comparison = pd.DataFrame(
    [
        {
            "method":
                "Additive-EUC",

            "geo_multiplier":
                best_add_euc[
                    "geo_multiplier"
                ],

            "feature_multiplier":
                best_add_euc[
                    "feature_multiplier"
                ],

            "h_geo":
                best_add_euc[
                    "h_geo"
                ],

            "h_feature":
                best_add_euc[
                    "h_feature"
                ],

            "lambda":
                best_add_euc[
                    "lambda"
                ],

            "coverage":
                best_add_euc[
                    "coverage"
                ],

            "mean_interval":
                best_add_euc[
                    "mean_interval"
                ],

            "interval_score":
                best_add_euc[
                    "interval_score"
                ],

            "median_n_eff":
                best_add_euc[
                    "median_n_eff"
                ],

            "valid_candidates":
                len(
                    valid_add_euc
                ),
        },

        {
            "method":
                "Additive-GCCP",

            "geo_multiplier":
                best_add_gccp[
                    "geo_multiplier"
                ],

            "feature_multiplier":
                best_add_gccp[
                    "feature_multiplier"
                ],

            "h_geo":
                best_add_gccp[
                    "h_geo"
                ],

            "h_feature":
                best_add_gccp[
                    "h_feature"
                ],

            "lambda":
                best_add_gccp[
                    "lambda"
                ],

            "coverage":
                best_add_gccp[
                    "coverage"
                ],

            "mean_interval":
                best_add_gccp[
                    "mean_interval"
                ],

            "interval_score":
                best_add_gccp[
                    "interval_score"
                ],

            "median_n_eff":
                best_add_gccp[
                    "median_n_eff"
                ],

            "valid_candidates":
                len(
                    valid_add_gccp
                ),
        },
    ]
)


print(
    "\n===== ADDITIVE MAIN COMPARISON ====="
)


display(
    main_comparison.round(
        6
    )
)


# ============================================================
# 12. Top 15 valid candidates for each
# ============================================================

print(
    "\n===== TOP ADDITIVE-EUC ====="
)


display(

    valid_add_euc
    .sort_values(
        [
            "interval_score",
            "mean_interval"
        ]
    )
    .head(15)
    .reset_index(
        drop=True
    )
    .round(6)
)


print(
    "\n===== TOP ADDITIVE-GCCP ====="
)


display(

    valid_add_gccp
    .sort_values(
        [
            "interval_score",
            "mean_interval"
        ]
    )
    .head(15)
    .reset_index(
        drop=True
    )
    .round(6)
)


# ============================================================
# 13. Best result within each lambda
# ============================================================

def best_by_lambda_table(
    df
):

    rows = []


    for lam in lambda_grid_v2:

        subset = df[
            np.isclose(
                df[
                    "lambda"
                ],
                lam
            )
        ].copy()


        valid = subset[
            subset[
                "coverage"
            ]
            >=
            0.90
        ]


        if len(
            valid
        ) > 0:

            best = valid.loc[

                valid[
                    "interval_score"
                ]
                .idxmin()
            ]

            branch = (
                "coverage>=0.90"
            )

        else:

            best = subset.loc[

                subset[
                    "interval_score"
                ]
                .idxmin()
            ]

            branch = (
                "no-valid-coverage"
            )


        rows.append(
            {
                "lambda":
                    float(
                        lam
                    ),

                "geo_multiplier":
                    float(
                        best[
                            "geo_multiplier"
                        ]
                    ),

                "feature_multiplier":
                    float(
                        best[
                            "feature_multiplier"
                        ]
                    ),

                "coverage":
                    float(
                        best[
                            "coverage"
                        ]
                    ),

                "mean_interval":
                    float(
                        best[
                            "mean_interval"
                        ]
                    ),

                "interval_score":
                    float(
                        best[
                            "interval_score"
                        ]
                    ),

                "median_n_eff":
                    float(
                        best[
                            "median_n_eff"
                        ]
                    ),

                "selection_branch":
                    branch,
            }
        )


    return pd.DataFrame(
        rows
    )


add_euc_by_lambda = best_by_lambda_table(
    df_additive_euc
)


add_gccp_by_lambda = best_by_lambda_table(
    df_additive_gccp
)


print(
    "\n===== ADDITIVE-EUC BEST BY LAMBDA ====="
)

display(
    add_euc_by_lambda.round(
        6
    )
)


print(
    "\n===== ADDITIVE-GCCP BEST BY LAMBDA ====="
)

display(
    add_gccp_by_lambda.round(
        6
    )
)


# ============================================================
# 14. Direct GCCP vs EUC improvement
# ============================================================

coverage_difference_pp = (

    best_add_gccp[
        "coverage"
    ]

    -

    best_add_euc[
        "coverage"
    ]

) * 100.0


width_difference_pct = (

    (
        best_add_gccp[
            "mean_interval"
        ]

        /
        best_add_euc[
            "mean_interval"
        ]
    )

    -
    1.0

) * 100.0


score_difference_pct = (

    (
        best_add_gccp[
            "interval_score"
        ]

        /
        best_add_euc[
            "interval_score"
        ]
    )

    -
    1.0

) * 100.0


print(
    "\n===== ADDITIVE-GCCP vs ADDITIVE-EUC ====="
)


print(
    "Coverage difference (pp):",
    coverage_difference_pp
)


print(
    "Width difference (%):",
    width_difference_pct
)


print(
    "Interval Score difference (%):",
    score_difference_pct
)


# ============================================================
# 15. Save
# ============================================================

ADDITIVE_EUC_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "additive_euc_validation_v1.csv"
)


ADDITIVE_GCCP_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "additive_gccp_validation_v1.csv"
)


ADDITIVE_COMPARISON_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "additive_euc_vs_gccp_validation_summary_v1.csv"
)


df_additive_euc.to_csv(
    ADDITIVE_EUC_PATH,
    index=False
)


df_additive_gccp.to_csv(
    ADDITIVE_GCCP_PATH,
    index=False
)


main_comparison.to_csv(
    ADDITIVE_COMPARISON_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)

print(
    ADDITIVE_EUC_PATH
)

print(
    ADDITIVE_GCCP_PATH
)

print(
    ADDITIVE_COMPARISON_PATH
)


print(
    "\n✓ Additive-EUC completed."
)

print(
    "✓ Additive-GCCP completed."
)

print(
    "✓ Both used identical experimental structure."
)

print(
    "✓ Bandwidth scales came from Calibration only."
)

print(
    "✓ Validation y was used only for parameter selection."
)

print(
    "✓ Test was NOT used."
)

===== ADDITIVE-EUC vs ADDITIVE-GCCP VALIDATION =====

Geo bandwidth candidates:
[ 4440.48066256  8880.96132512 17761.92265024 35523.84530049
 71047.69060098]

EUC bandwidth candidates:
[ 0.84138533  1.68277067  3.36554134  6.73108268 13.46216536]

Causal bandwidth candidates:
[ 0.75699593  1.51399186  3.02798372  6.05596744 12.11193488]

Lambda candidates:
[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]

Configurations per method: 275

[1] Running Additive-EUC...
Configurations: 275

[2] Running Additive-GCCP...
Configurations: 275

===== ADDITIVE MAIN COMPARISON =====


,method,geo_multiplier,feature_multiplier,h_geo,h_feature,lambda,coverage,mean_interval,interval_score,median_n_eff,valid_candidates
0,Additive-EUC,0.50,1.0,8880.961325,3.365541,0.1,0.920000,0.246603,0.317491,276.358156,270
1,Additive-GCCP,0.25,1.0,4440.480663,3.027984,0.2,0.906667,0.242745,0.316927,265.159004,270



===== TOP ADDITIVE-EUC =====


,method,geo_multiplier,feature_multiplier,h_geo,h_feature,lambda,coverage,mean_interval,interval_score,median_q,mean_q,median_n_eff,mean_n_eff
0,Additive-EUC,0.50,1.0,8880.961325,3.365541,0.1,0.920000,0.246603,0.317491,0.121919,0.123301,276.358156,273.330563
1,Additive-EUC,4.00,1.0,71047.690601,3.365541,0.1,0.920000,0.247699,0.317538,0.125410,0.123849,281.261694,280.057624
2,Additive-EUC,1.00,1.0,17761.922650,3.365541,0.1,0.920000,0.247309,0.317589,0.123665,0.123654,279.023069,276.694808
3,Additive-EUC,0.25,1.0,4440.480663,3.365541,0.0,0.916667,0.245874,0.317728,0.121919,0.122937,274.681300,271.715549
4,Additive-EUC,0.50,1.0,8880.961325,3.365541,0.0,0.916667,0.245874,0.317728,0.121919,0.122937,274.681300,271.715549
5,Additive-EUC,1.00,1.0,17761.922650,3.365541,0.0,0.916667,0.245874,0.317728,0.121919,0.122937,274.681300,271.715549
6,Additive-EUC,2.00,1.0,35523.845300,3.365541,0.0,0.916667,0.245874,0.317728,0.121919,0.122937,274.681300,271.715549
7,Additive-EUC,4.00,1.0,71047.690601,3.365541,0.0,0.916667,0.245874,0.317728,0.121919,0.122937,274.681300,271.715549
8,Additive-EUC,2.00,1.0,35523.845300,3.365541,0.1,0.920000,0.247793,0.317865,0.125410,0.123896,280.700441,279.209584
9,Additive-EUC,0.25,1.0,4440.480663,3.365541,0.2,0.916667,0.246495,0.317918,0.121919,0.123247,274.156120,270.858080



===== TOP ADDITIVE-GCCP =====


...


===== ADDITIVE-EUC BEST BY LAMBDA =====


,lambda,geo_multiplier,feature_multiplier,coverage,mean_interval,interval_score,median_n_eff,selection_branch
0,0.0,0.25,1.00,0.916667,0.245874,0.317728,274.681300,coverage>=0.90
1,0.1,0.50,1.00,0.920000,0.246603,0.317491,276.358156,coverage>=0.90
2,0.2,0.25,1.00,0.916667,0.246495,0.317918,274.156120,coverage>=0.90
3,0.3,4.00,0.50,0.916667,0.249991,0.318470,271.921251,coverage>=0.90
4,0.4,0.25,1.00,0.916667,0.246800,0.318792,267.528290,coverage>=0.90
5,0.5,0.25,1.00,0.916667,0.247203,0.319112,258.332888,coverage>=0.90
6,0.6,0.25,1.00,0.916667,0.247338,0.319146,242.236726,coverage>=0.90
7,0.7,4.00,1.00,0.923333,0.255103,0.321076,298.406163,coverage>=0.90
8,0.8,4.00,1.00,0.923333,0.255335,0.321308,299.074078,coverage>=0.90
9,0.9,2.00,0.50,0.923333,0.256899,0.321695,292.140027,coverage>=0.90



===== ADDITIVE-GCCP BEST BY LAMBDA =====


,lambda,geo_multiplier,feature_multiplier,coverage,mean_interval,interval_score,median_n_eff,selection_branch
0,0.0,0.25,1.00,0.906667,0.242576,0.317218,265.543140,coverage>=0.90
1,0.1,4.00,1.00,0.916667,0.244147,0.317013,274.192124,coverage>=0.90
2,0.2,0.25,1.00,0.906667,0.242745,0.316927,265.159004,coverage>=0.90
3,0.3,0.25,1.00,0.906667,0.243020,0.317659,262.966203,coverage>=0.90
4,0.4,0.25,1.00,0.906667,0.243187,0.317330,258.885403,coverage>=0.90
5,0.5,0.25,1.00,0.906667,0.243579,0.318024,249.918467,coverage>=0.90
6,0.6,0.25,1.00,0.906667,0.244167,0.318382,233.714365,coverage>=0.90
7,0.7,0.25,1.00,0.906667,0.245748,0.320820,208.722491,coverage>=0.90
8,0.8,4.00,1.00,0.923333,0.255022,0.321131,298.873172,coverage>=0.90
9,0.9,4.00,1.00,0.923333,0.256017,0.321580,299.318501,coverage>=0.90



===== ADDITIVE-GCCP vs ADDITIVE-EUC =====
Coverage difference (pp): -1.333333333333342
Width difference (%): -1.5643440148576881
Interval Score difference (%): -0.17773520301140877

===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\additive_euc_validation_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\additive_gccp_validation_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\additive_euc_vs_gccp_validation_summary_v1.csv

✓ Additive-EUC completed.
✓ Additive-GCCP completed.
✓ Both used identical experimental structure.
✓ Bandwidth scales came from Calibration only.
✓ Validation y was used only for parameter selection.
✓ Test was NOT used.


In [40]:
# ============================================================
# Step 21 — FROZEN FINAL TEST EVALUATION
#
# IMPORTANT:
#
# - NO Test tuning
# - NO parameter search
# - NO model modification
# - NO causal-weight modification
#
# Frozen methods:
#
# 1. CP
# 2. GeoCP
# 3. GeoSIMCP-EUC
# 4. Additive-EUC
# 5. Additive-GCCP
#
# Calibration set remains the same.
# Validation-selected parameters remain frozen.
#
# After this cell, Seattle Test results must NOT be used
# to modify the method or choose new parameters.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances

from geoconformal import (
    GeoConformalSpatialRegression,
    GeoSIMConformalSpatialRegression
)

from geoconformal.geocp import (
    weighted_quantile
)


print(
    "===== FROZEN FINAL TEST EVALUATION ====="
)


EPS = 1e-8
ALPHA = 0.1
TARGET_COVERAGE = 1.0 - ALPHA


# ============================================================
# 1. Freeze all Validation-selected parameters
# ============================================================

# ------------------------------------------------------------
# GeoCP
# ------------------------------------------------------------

FROZEN_GEOCP_BW = float(
    best_geocp[
        "bandwidth"
    ]
)


# ------------------------------------------------------------
# GeoSIMCP-EUC
# ------------------------------------------------------------

FROZEN_EUC_BW = float(
    best_simcp_euc[
        "bandwidth"
    ]
)

FROZEN_EUC_LAMBDA = float(
    best_simcp_euc[
        "lambda"
    ]
)


# ------------------------------------------------------------
# Additive-EUC
#
# Validation-selected:
#
# geo_multiplier     = 0.50
# feature_multiplier = 1.00
# lambda             = 0.10
# ------------------------------------------------------------

FROZEN_ADD_EUC_GEO_MULT = 0.50
FROZEN_ADD_EUC_FEAT_MULT = 1.00
FROZEN_ADD_EUC_LAMBDA = 0.10


# ------------------------------------------------------------
# Additive-GCCP
#
# Validation-selected:
#
# geo_multiplier     = 0.25
# feature_multiplier = 1.00
# lambda             = 0.20
# ------------------------------------------------------------

FROZEN_ADD_GCCP_GEO_MULT = 0.25
FROZEN_ADD_GCCP_FEAT_MULT = 1.00
FROZEN_ADD_GCCP_LAMBDA = 0.20


# ============================================================
# 2. Recover the EXACT scale-aware bandwidths
#    from the Calibration-only grids constructed in Step 20A
# ============================================================

def bandwidth_from_multiplier(
    multiplier,
    multiplier_grid,
    bandwidth_grid
):

    matches = np.where(
        np.isclose(
            multiplier_grid,
            multiplier
        )
    )[0]


    assert len(matches) == 1


    return float(
        bandwidth_grid[
            matches[0]
        ]
    )


FROZEN_ADD_EUC_H_GEO = bandwidth_from_multiplier(

    FROZEN_ADD_EUC_GEO_MULT,

    BANDWIDTH_MULTIPLIERS,

    geo_bandwidth_grid_v2
)


FROZEN_ADD_EUC_H_FEATURE = bandwidth_from_multiplier(

    FROZEN_ADD_EUC_FEAT_MULT,

    BANDWIDTH_MULTIPLIERS,

    euc_bandwidth_grid_v2
)


FROZEN_ADD_GCCP_H_GEO = bandwidth_from_multiplier(

    FROZEN_ADD_GCCP_GEO_MULT,

    BANDWIDTH_MULTIPLIERS,

    geo_bandwidth_grid_v2
)


FROZEN_ADD_GCCP_H_FEATURE = bandwidth_from_multiplier(

    FROZEN_ADD_GCCP_FEAT_MULT,

    BANDWIDTH_MULTIPLIERS,

    causal_bandwidth_grid_v2
)


print(
    "\n===== FROZEN PARAMETERS ====="
)

print(
    "GeoCP bandwidth:",
    FROZEN_GEOCP_BW
)

print(
    "GeoSIMCP-EUC bandwidth:",
    FROZEN_EUC_BW
)

print(
    "GeoSIMCP-EUC lambda:",
    FROZEN_EUC_LAMBDA
)


print(
    "\nAdditive-EUC:"
)

print(
    "  geo multiplier     =",
    FROZEN_ADD_EUC_GEO_MULT
)

print(
    "  feature multiplier =",
    FROZEN_ADD_EUC_FEAT_MULT
)

print(
    "  h_geo              =",
    FROZEN_ADD_EUC_H_GEO
)

print(
    "  h_EUC              =",
    FROZEN_ADD_EUC_H_FEATURE
)

print(
    "  lambda             =",
    FROZEN_ADD_EUC_LAMBDA
)


print(
    "\nAdditive-GCCP:"
)

print(
    "  geo multiplier     =",
    FROZEN_ADD_GCCP_GEO_MULT
)

print(
    "  feature multiplier =",
    FROZEN_ADD_GCCP_FEAT_MULT
)

print(
    "  h_geo              =",
    FROZEN_ADD_GCCP_H_GEO
)

print(
    "  h_causal           =",
    FROZEN_ADD_GCCP_H_FEATURE
)

print(
    "  lambda             =",
    FROZEN_ADD_GCCP_LAMBDA
)


# ============================================================
# 3. Shared Calibration residuals
#
# IMPORTANT:
# Same predictor and same Calibration set as Validation stage.
# We do NOT retrain/refit the predictor after seeing Validation.
# ============================================================

calib_predictions_final = model.predict(
    X_calib_spatial
)


calibration_scores_final = np.abs(

    y_calib

    -

    calib_predictions_final
)


test_predictions_final = model.predict(
    X_test_spatial
)


print(
    "\n===== DATA CHECK ====="
)

print(
    "Calibration samples:",
    len(
        y_calib
    )
)

print(
    "Test samples:",
    len(
        y_test
    )
)


assert len(
    y_calib
) == 300


assert len(
    y_test
) == 300


# ============================================================
# 4. Metric helpers
# ============================================================

def final_interval_score(
    lower,
    upper,
    y_true,
    alpha=0.1
):

    lower = np.asarray(
        lower,
        dtype=float
    )

    upper = np.asarray(
        upper,
        dtype=float
    )

    y_true = np.asarray(
        y_true,
        dtype=float
    )


    width = (
        upper
        -
        lower
    )


    lower_penalty = (

        2.0
        /
        alpha

        *

        np.maximum(
            lower
            -
            y_true,
            0.0
        )
    )


    upper_penalty = (

        2.0
        /
        alpha

        *

        np.maximum(
            y_true
            -
            upper,
            0.0
        )
    )


    return float(
        np.mean(
            width
            +
            lower_penalty
            +
            upper_penalty
        )
    )


def effective_sample_size_matrix(
    weights
):

    weights = np.asarray(
        weights,
        dtype=float
    )


    sum_w = np.sum(
        weights,
        axis=1
    )


    sum_w2 = np.sum(
        weights ** 2,
        axis=1
    )


    return (

        sum_w ** 2

        /

        np.where(
            sum_w2 > 0,
            sum_w2,
            np.nan
        )
    )


def evaluate_intervals(
    method,
    lower,
    upper,
    y_true,
    q_values=None,
    n_eff=None
):

    lower = np.asarray(
        lower,
        dtype=float
    )

    upper = np.asarray(
        upper,
        dtype=float
    )

    y_true = np.asarray(
        y_true,
        dtype=float
    )


    covered = (

        (y_true >= lower)

        &

        (y_true <= upper)
    )


    widths = (
        upper
        -
        lower
    )


    result = {

        "method":
            method,

        "n_test":
            len(
                y_true
            ),

        "n_covered":
            int(
                covered.sum()
            ),

        "coverage":
            float(
                covered.mean()
            ),

        "mean_interval":
            float(
                np.mean(
                    widths
                )
            ),

        "median_interval":
            float(
                np.median(
                    widths
                )
            ),

        "interval_score":
            final_interval_score(
                lower,
                upper,
                y_true,
                alpha=ALPHA
            ),

        "n_infinite":
            int(
                (
                    ~np.isfinite(
                        lower
                    )

                    |

                    ~np.isfinite(
                        upper
                    )
                ).sum()
            ),
    }


    if q_values is None:

        result[
            "median_q"
        ] = np.nan

        result[
            "mean_q"
        ] = np.nan

    else:

        q_values = np.asarray(
            q_values,
            dtype=float
        )

        result[
            "median_q"
        ] = float(
            np.median(
                q_values
            )
        )

        result[
            "mean_q"
        ] = float(
            np.mean(
                q_values
            )
        )


    if n_eff is None:

        result[
            "median_n_eff"
        ] = np.nan

        result[
            "mean_n_eff"
        ] = np.nan

    else:

        n_eff = np.asarray(
            n_eff,
            dtype=float
        )

        result[
            "median_n_eff"
        ] = float(
            np.nanmedian(
                n_eff
            )
        )

        result[
            "mean_n_eff"
        ] = float(
            np.nanmean(
                n_eff
            )
        )


    return result


# ============================================================
# 5. CP
# ============================================================

print(
    "\n[1/5] CP"
)


uniform_weights_test = np.ones(
    (
        len(
            y_test
        ),
        len(
            y_calib
        )
    ),
    dtype=float
)


q_cp_test = weighted_quantile(

    scores=calibration_scores_final,

    weights=uniform_weights_test,

    q=TARGET_COVERAGE,

    self_weights=None,

    interpolate=False
)


cp_test_lower = (

    test_predictions_final

    -

    q_cp_test
)


cp_test_upper = (

    test_predictions_final

    +

    q_cp_test
)


cp_test_neff = np.full(
    len(
        y_test
    ),
    len(
        y_calib
    ),
    dtype=float
)


# ============================================================
# 6. GeoCP official
# ============================================================

print(
    "[2/5] GeoCP"
)


final_geocp = GeoConformalSpatialRegression(

    predict_f=model.predict,

    miscoverage_level=ALPHA,

    bandwidth=FROZEN_GEOCP_BW,

    coord_calib=coords_calib,

    coord_test=coords_test,

    X_calib=X_calib_spatial,

    y_calib=y_calib,

    X_test=X_test_spatial,

    y_test=y_test,

    include_test_atom=False
)


final_geocp_result = (
    final_geocp.analyze()
)


geocp_test_lower = np.asarray(
    final_geocp_result.lower_bound,
    dtype=float
)


geocp_test_upper = np.asarray(
    final_geocp_result.upper_bound,
    dtype=float
)


geocp_test_q = (

    geocp_test_upper

    -

    geocp_test_lower

) / 2.0


# Reconstruct GeoCP weights for n_eff diagnostic
d_geo_test = pairwise_distances(

    coords_test,

    coords_calib,

    metric="euclidean"
)


geocp_test_weights = np.exp(

    -0.5
    *
    (
        d_geo_test

        /

        (
            FROZEN_GEOCP_BW
            +
            EPS
        )
    ) ** 2
)


geocp_test_neff = effective_sample_size_matrix(
    geocp_test_weights
)


# ============================================================
# 7. GeoSIMCP-EUC official
# ============================================================

print(
    "[3/5] GeoSIMCP-EUC"
)


final_euc = GeoSIMConformalSpatialRegression(

    predict_f=model.predict,

    miscoverage_level=ALPHA,

    bandwidth=FROZEN_EUC_BW,

    lambda_weight=FROZEN_EUC_LAMBDA,

    distance_metric="euclidean",

    standardize_weights=True,

    coord_calib=coords_calib,

    coord_test=coords_test,

    X_calib=X_calib_spatial,

    y_calib=y_calib,

    X_test=X_test_spatial,

    y_test=y_test,

    X_calib_weight=X_calib,

    X_test_weight=X_test,

    include_test_atom=False
)


final_euc_result = (
    final_euc.analyze()
)


euc_test_lower = np.asarray(
    final_euc_result.lower_bound,
    dtype=float
)


euc_test_upper = np.asarray(
    final_euc_result.upper_bound,
    dtype=float
)


euc_test_q = (

    euc_test_upper

    -

    euc_test_lower

) / 2.0


# ============================================================
# 8. Prepare Test feature distances for additive methods
#
# IMPORTANT:
# Same active 7-dimensional feature space
# determined from Calibration.
# ============================================================

calib_std_all_final = np.std(
    X_calib,
    axis=0
)


active_mask_final = (
    calib_std_all_final > 1e-8
)


active_indices_final = np.where(
    active_mask_final
)[0]


active_features_final = [
    feature_order[i]
    for i in active_indices_final
]


print(
    "\nActive additive feature space:"
)

print(
    active_features_final
)


assert active_features_final == [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "view",
    "age",
]


X_calib_active_final = (
    X_calib[
        :,
        active_indices_final
    ]
)


X_test_active_final = (
    X_test[
        :,
        active_indices_final
    ]
)


cal_mean_final = np.mean(
    X_calib_active_final,
    axis=0
)


cal_std_final = np.std(
    X_calib_active_final,
    axis=0
)


assert np.all(
    cal_std_final > 1e-8
)


Z_calib_final = (

    X_calib_active_final

    -

    cal_mean_final

) / cal_std_final


Z_test_final = (

    X_test_active_final

    -

    cal_mean_final

) / cal_std_final


# ------------------------------------------------------------
# Ordinary EUC Test-Cal distance
# ------------------------------------------------------------

d_euc_test = pairwise_distances(

    Z_test_final,

    Z_calib_final,

    metric="euclidean"
)


# ------------------------------------------------------------
# Frozen causal weights
# ------------------------------------------------------------

CAUSAL_WEIGHT_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "gccm_author_aligned"
    / "seattle_gccm_train_logprice_causal_weights_v1.csv"
)


causal_weight_final_df = pd.read_csv(
    CAUSAL_WEIGHT_PATH
)


causal_weight_final_df = (

    causal_weight_final_df

    .set_index(
        "feature"
    )

    .loc[
        feature_order
    ]

    .reset_index()
)


all_causal_weights_final = (

    causal_weight_final_df[
        "causal_weight"
    ]

    .to_numpy(
        dtype=float
    )
)


active_causal_weights_final = (

    all_causal_weights_final[
        active_indices_final
    ]
)


active_causal_weights_final = (

    active_causal_weights_final

    /

    active_causal_weights_final.sum()
)


p_eff_final = len(
    active_causal_weights_final
)


causal_coefficients_final = (

    p_eff_final

    *

    active_causal_weights_final
)


test_cal_diff = (

    Z_test_final[
        :,
        None,
        :
    ]

    -

    Z_calib_final[
        None,
        :,
        :
    ]
)


d_causal_test = np.sqrt(

    np.sum(

        causal_coefficients_final[
            None,
            None,
            :
        ]

        *

        test_cal_diff ** 2,

        axis=2
    )
)


# ============================================================
# 9. Additive-EUC
# ============================================================

print(
    "[4/5] Additive-EUC"
)


K_geo_add_euc_test = np.exp(

    -0.5
    *
    (
        d_geo_test

        /

        (
            FROZEN_ADD_EUC_H_GEO
            +
            EPS
        )
    ) ** 2
)


K_euc_test = np.exp(

    -0.5
    *
    (
        d_euc_test

        /

        (
            FROZEN_ADD_EUC_H_FEATURE
            +
            EPS
        )
    ) ** 2
)


add_euc_test_weights = (

    FROZEN_ADD_EUC_LAMBDA
    *
    K_geo_add_euc_test

    +

    (
        1.0
        -
        FROZEN_ADD_EUC_LAMBDA
    )
    *
    K_euc_test
)


add_euc_test_q = weighted_quantile(

    scores=calibration_scores_final,

    weights=add_euc_test_weights,

    q=TARGET_COVERAGE,

    self_weights=None,

    interpolate=False
)


add_euc_test_lower = (

    test_predictions_final

    -

    add_euc_test_q
)


add_euc_test_upper = (

    test_predictions_final

    +

    add_euc_test_q
)


add_euc_test_neff = effective_sample_size_matrix(
    add_euc_test_weights
)


# ============================================================
# 10. Additive-GCCP
# ============================================================

print(
    "[5/5] Additive-GCCP"
)


K_geo_add_gccp_test = np.exp(

    -0.5
    *
    (
        d_geo_test

        /

        (
            FROZEN_ADD_GCCP_H_GEO
            +
            EPS
        )
    ) ** 2
)


K_causal_test = np.exp(

    -0.5
    *
    (
        d_causal_test

        /

        (
            FROZEN_ADD_GCCP_H_FEATURE
            +
            EPS
        )
    ) ** 2
)


add_gccp_test_weights = (

    FROZEN_ADD_GCCP_LAMBDA
    *
    K_geo_add_gccp_test

    +

    (
        1.0
        -
        FROZEN_ADD_GCCP_LAMBDA
    )
    *
    K_causal_test
)


add_gccp_test_q = weighted_quantile(

    scores=calibration_scores_final,

    weights=add_gccp_test_weights,

    q=TARGET_COVERAGE,

    self_weights=None,

    interpolate=False
)


add_gccp_test_lower = (

    test_predictions_final

    -

    add_gccp_test_q
)


add_gccp_test_upper = (

    test_predictions_final

    +

    add_gccp_test_q
)


add_gccp_test_neff = effective_sample_size_matrix(
    add_gccp_test_weights
)


# ============================================================
# 11. Safety checks
# ============================================================

for method_name, lower, upper in [

    (
        "CP",
        cp_test_lower,
        cp_test_upper
    ),

    (
        "GeoCP",
        geocp_test_lower,
        geocp_test_upper
    ),

    (
        "GeoSIMCP-EUC",
        euc_test_lower,
        euc_test_upper
    ),

    (
        "Additive-EUC",
        add_euc_test_lower,
        add_euc_test_upper
    ),

    (
        "Additive-GCCP",
        add_gccp_test_lower,
        add_gccp_test_upper
    ),
]:

    assert np.isfinite(
        lower
    ).all(), (
        f"{method_name}: non-finite lower bounds found"
    )


    assert np.isfinite(
        upper
    ).all(), (
        f"{method_name}: non-finite upper bounds found"
    )


# ============================================================
# 12. Final Test summary
# ============================================================

test_summary_rows = []


test_summary_rows.append(

    evaluate_intervals(

        method="CP",

        lower=cp_test_lower,

        upper=cp_test_upper,

        y_true=y_test,

        q_values=q_cp_test,

        n_eff=cp_test_neff
    )
)


test_summary_rows.append(

    evaluate_intervals(

        method="GeoCP",

        lower=geocp_test_lower,

        upper=geocp_test_upper,

        y_true=y_test,

        q_values=geocp_test_q,

        n_eff=geocp_test_neff
    )
)


test_summary_rows.append(

    evaluate_intervals(

        method="GeoSIMCP-EUC",

        lower=euc_test_lower,

        upper=euc_test_upper,

        y_true=y_test,

        q_values=euc_test_q,

        n_eff=None
    )
)


test_summary_rows.append(

    evaluate_intervals(

        method="Additive-EUC",

        lower=add_euc_test_lower,

        upper=add_euc_test_upper,

        y_true=y_test,

        q_values=add_euc_test_q,

        n_eff=add_euc_test_neff
    )
)


test_summary_rows.append(

    evaluate_intervals(

        method="Additive-GCCP",

        lower=add_gccp_test_lower,

        upper=add_gccp_test_upper,

        y_true=y_test,

        q_values=add_gccp_test_q,

        n_eff=add_gccp_test_neff
    )
)


test_summary = pd.DataFrame(
    test_summary_rows
)


print(
    "\n===== FINAL TEST SUMMARY ====="
)


display(
    test_summary.round(
        6
    )
)


# ============================================================
# 13. Direct Additive-GCCP vs Additive-EUC comparison
# ============================================================

row_add_euc_test = (
    test_summary[
        test_summary[
            "method"
        ]
        ==
        "Additive-EUC"
    ]
    .iloc[0]
)


row_add_gccp_test = (
    test_summary[
        test_summary[
            "method"
        ]
        ==
        "Additive-GCCP"
    ]
    .iloc[0]
)


coverage_diff_pp_test = (

    row_add_gccp_test[
        "coverage"
    ]

    -

    row_add_euc_test[
        "coverage"
    ]

) * 100.0


width_diff_pct_test = (

    (
        row_add_gccp_test[
            "mean_interval"
        ]

        /
        row_add_euc_test[
            "mean_interval"
        ]
    )

    -
    1.0

) * 100.0


score_diff_pct_test = (

    (
        row_add_gccp_test[
            "interval_score"
        ]

        /
        row_add_euc_test[
            "interval_score"
        ]
    )

    -
    1.0

) * 100.0


print(
    "\n===== ADDITIVE-GCCP vs ADDITIVE-EUC — TEST ====="
)


print(
    "Coverage difference (percentage points):",
    coverage_diff_pp_test
)


print(
    "Mean Width difference (%):",
    width_diff_pct_test
)


print(
    "Interval Score difference (%):",
    score_diff_pct_test
)


# ============================================================
# 14. Exact coverage counts
#
# With n=300:
# 90% nominal coverage corresponds to 270 / 300.
# ============================================================

print(
    "\n===== COVERAGE COUNTS ====="
)


for _, row in test_summary.iterrows():

    print(
        f"{row['method']:18s}: "
        f"{int(row['n_covered'])}/"
        f"{int(row['n_test'])} "
        f"= {row['coverage']:.6f}"
    )


# ============================================================
# 15. Per-sample Additive comparison
#
# How often does GCCP give narrower/wider intervals?
# ============================================================

width_add_euc_sample = (

    add_euc_test_upper

    -

    add_euc_test_lower
)


width_add_gccp_sample = (

    add_gccp_test_upper

    -

    add_gccp_test_lower
)


width_difference_sample = (

    width_add_gccp_sample

    -

    width_add_euc_sample
)


n_gccp_narrower = int(
    np.sum(
        width_difference_sample < -1e-12
    )
)


n_same_width = int(
    np.sum(
        np.abs(
            width_difference_sample
        ) <= 1e-12
    )
)


n_gccp_wider = int(
    np.sum(
        width_difference_sample > 1e-12
    )
)


print(
    "\n===== PER-SAMPLE WIDTH COMPARISON ====="
)


print(
    "GCCP narrower:",
    n_gccp_narrower
)


print(
    "Same width:",
    n_same_width
)


print(
    "GCCP wider:",
    n_gccp_wider
)


# ============================================================
# 16. Validation vs Test comparison
#
# This checks whether the Validation advantage generalized.
# It does NOT perform any new selection.
# ============================================================

validation_vs_test = pd.DataFrame(
    [
        {
            "method":
                "Additive-EUC",

            "validation_coverage":
                float(
                    best_add_euc[
                        "coverage"
                    ]
                ),

            "test_coverage":
                float(
                    row_add_euc_test[
                        "coverage"
                    ]
                ),

            "validation_width":
                float(
                    best_add_euc[
                        "mean_interval"
                    ]
                ),

            "test_width":
                float(
                    row_add_euc_test[
                        "mean_interval"
                    ]
                ),

            "validation_score":
                float(
                    best_add_euc[
                        "interval_score"
                    ]
                ),

            "test_score":
                float(
                    row_add_euc_test[
                        "interval_score"
                    ]
                ),
        },

        {
            "method":
                "Additive-GCCP",

            "validation_coverage":
                float(
                    best_add_gccp[
                        "coverage"
                    ]
                ),

            "test_coverage":
                float(
                    row_add_gccp_test[
                        "coverage"
                    ]
                ),

            "validation_width":
                float(
                    best_add_gccp[
                        "mean_interval"
                    ]
                ),

            "test_width":
                float(
                    row_add_gccp_test[
                        "mean_interval"
                    ]
                ),

            "validation_score":
                float(
                    best_add_gccp[
                        "interval_score"
                    ]
                ),

            "test_score":
                float(
                    row_add_gccp_test[
                        "interval_score"
                    ]
                ),
        },
    ]
)


print(
    "\n===== VALIDATION vs TEST ====="
)


display(
    validation_vs_test.round(
        6
    )
)


# ============================================================
# 17. Save per-sample final Test results
# ============================================================

per_sample_test = pd.DataFrame(
    {
        "test_index":
            np.arange(
                len(
                    y_test
                )
            ),

        "y_true":
            y_test,

        "prediction":
            test_predictions_final,

        # CP
        "CP_q":
            q_cp_test,

        "CP_lower":
            cp_test_lower,

        "CP_upper":
            cp_test_upper,

        # GeoCP
        "GeoCP_q":
            geocp_test_q,

        "GeoCP_lower":
            geocp_test_lower,

        "GeoCP_upper":
            geocp_test_upper,

        # GeoSIMCP-EUC
        "GeoSIMCP_EUC_q":
            euc_test_q,

        "GeoSIMCP_EUC_lower":
            euc_test_lower,

        "GeoSIMCP_EUC_upper":
            euc_test_upper,

        # Additive-EUC
        "Additive_EUC_q":
            add_euc_test_q,

        "Additive_EUC_lower":
            add_euc_test_lower,

        "Additive_EUC_upper":
            add_euc_test_upper,

        "Additive_EUC_n_eff":
            add_euc_test_neff,

        # Additive-GCCP
        "Additive_GCCP_q":
            add_gccp_test_q,

        "Additive_GCCP_lower":
            add_gccp_test_lower,

        "Additive_GCCP_upper":
            add_gccp_test_upper,

        "Additive_GCCP_n_eff":
            add_gccp_test_neff,
    }
)


# ============================================================
# 18. Save frozen outputs
# ============================================================

FINAL_TEST_SUMMARY_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "seattle_author_aligned_final_test_summary_v1.csv"
)


FINAL_TEST_SAMPLE_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "seattle_author_aligned_final_test_per_sample_v1.csv"
)


FINAL_VALIDATION_TEST_PATH = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
    / "seattle_additive_validation_vs_test_v1.csv"
)


test_summary.to_csv(
    FINAL_TEST_SUMMARY_PATH,
    index=False
)


per_sample_test.to_csv(
    FINAL_TEST_SAMPLE_PATH,
    index=False
)


validation_vs_test.to_csv(
    FINAL_VALIDATION_TEST_PATH,
    index=False
)


print(
    "\n===== SAVED ====="
)


print(
    FINAL_TEST_SUMMARY_PATH
)


print(
    FINAL_TEST_SAMPLE_PATH
)


print(
    FINAL_VALIDATION_TEST_PATH
)


print(
    "\n✓ FINAL frozen Test evaluation completed."
)

print(
    "✓ No Test parameter search was performed."
)

print(
    "✓ No model was modified using Test results."
)

print(
    "✓ Seattle Test results must now be treated as final."
)

===== FROZEN FINAL TEST EVALUATION =====

===== FROZEN PARAMETERS =====
GeoCP bandwidth: 28982.758620689656
GeoSIMCP-EUC bandwidth: 500.0
GeoSIMCP-EUC lambda: 0.0

Additive-EUC:
  geo multiplier     = 0.5
  feature multiplier = 1.0
  h_geo              = 8880.961325122025
  h_EUC              = 3.3655413398211977
  lambda             = 0.1

Additive-GCCP:
  geo multiplier     = 0.25
  feature multiplier = 1.0
  h_geo              = 4440.480662561013
  h_causal           = 3.027983720113191
  lambda             = 0.2

===== DATA CHECK =====
Calibration samples: 300
Test samples: 300

[1/5] CP
[2/5] GeoCP
[3/5] GeoSIMCP-EUC

Active additive feature space:
['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'view', 'age']
[4/5] Additive-EUC
[5/5] Additive-GCCP

===== FINAL TEST SUMMARY =====


,method,n_test,n_covered,coverage,mean_interval,median_interval,interval_score,n_infinite,median_q,mean_q,median_n_eff,mean_n_eff
0,CP,300,275,0.916667,0.254136,0.254136,0.353382,0,0.127068,0.127068,300.000000,300.000000
1,GeoCP,300,275,0.916667,0.257533,0.258225,0.353660,0,0.129113,0.128766,285.062697,282.320111
2,GeoSIMCP-EUC,300,275,0.916667,0.254299,0.254136,0.353273,0,0.127068,0.127150,NaN,NaN
3,Additive-EUC,300,274,0.913333,0.248073,0.250820,0.345760,0,0.125410,0.124037,276.047948,272.454015
4,Additive-GCCP,300,271,0.903333,0.244212,0.243838,0.345913,0,0.121919,0.122106,265.150082,255.625781



===== ADDITIVE-GCCP vs ADDITIVE-EUC — TEST =====
Coverage difference (percentage points): -1.0000000000000009
Mean Width difference (%): -1.5562161962082488
Interval Score difference (%): 0.04417447412803366

===== COVERAGE COUNTS =====
CP                : 275/300 = 0.916667
GeoCP             : 275/300 = 0.916667
GeoSIMCP-EUC      : 275/300 = 0.916667
Additive-EUC      : 274/300 = 0.913333
Additive-GCCP     : 271/300 = 0.903333

===== PER-SAMPLE WIDTH COMPARISON =====
GCCP narrower: 176
Same width: 81
GCCP wider: 43

===== VALIDATION vs TEST =====


,method,validation_coverage,test_coverage,validation_width,test_width,validation_score,test_score
0,Additive-EUC,0.920000,0.913333,0.246603,0.248073,0.317491,0.345760
1,Additive-GCCP,0.906667,0.903333,0.242745,0.244212,0.316927,0.345913



===== SAVED =====
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_author_aligned_final_test_summary_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_author_aligned_final_test_per_sample_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_additive_validation_vs_test_v1.csv

✓ FINAL frozen Test evaluation completed.
✓ No Test parameter search was performed.
✓ No model was modified using Test results.
✓ Seattle Test results must now be treated as final.
